<a href="https://colab.research.google.com/github/ayoitshasya/respiratory-disorder-detection/blob/main/sem7_smartpad_window_sweep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# SMARTPAD CELL 1 — implement + verify (CPU, no training)
# ============================================================
import os, re, json, zipfile
import numpy as np, pandas as pd

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
PROJ='/content/drive/MyDrive/pulmoscan semester 7'; WORK='/content/work'
os.makedirs(WORK,exist_ok=True)

if not os.path.isdir(os.path.join(WORK,'feat','processed','train')):
    print("extracting features...")
    with zipfile.ZipFile(os.path.join(PROJ,'data','pulmoscan_data.zip')) as z:
        z.extractall(os.path.join(WORK,'feat'))
with zipfile.ZipFile(os.path.join(PROJ,'data','colab_training_package.zip')) as z:
    z.extract('data/processed/manifest.csv', WORK)

man=pd.read_csv(os.path.join(WORK,'data/processed/manifest.csv'))
dev=man[man.split.isin(['train','val'])].copy().reset_index(drop=True)
dev['fp']=dev.apply(lambda r: os.path.join(WORK,'feat','processed',r['split'],
                    r['audio_file'].replace('.wav','.npy')),axis=1)
dev=dev[dev.fp.map(os.path.exists)].reset_index(drop=True)

# parse recording id + cycle index so we can find true neighbours
def parse(fn):
    m=re.match(r'^(.*)_cycle(\d+)_([a-z]+)\.wav$',fn,re.I)
    return (m.group(1), int(m.group(2))) if m else (None,None)
dev[['rec','cyc']]=dev.audio_file.apply(lambda f: pd.Series(parse(f)))
print(f"{len(dev)} rows, {dev.rec.nunique()} recordings, "
      f"{dev.cyc.isna().sum()} unparsed")

# index: (recording, cycle_idx) -> row
IDX={(r.rec,r.cyc):i for i,r in dev.iterrows()}
CACHE={}
def feat(i):
    if i not in CACHE: CACHE[i]=np.load(dev.fp[i])
    return CACHE[i]

def smart_pad(i, target):
    """RespireNet smart padding. Concatenate the SAME RECORDING's neighbouring
    cycle if it is the same class or Normal; else repeat this cycle."""
    base=feat(i); out=base
    if out.shape[1]>=target: return out[:,:target]
    lbl=dev.sound_label[i]; rec=dev.rec[i]; c=dev.cyc[i]
    step=1
    while out.shape[1]<target:
        cand=None
        for off in (c+step, c-step):                 # look after, then before
            j=IDX.get((rec,off))
            if j is not None:
                if dev.sound_label[j]==lbl or dev.sound_label[j]==0:
                    cand=j; break
        nxt = feat(cand) if cand is not None else base   # fallback: repeat self
        out=np.concatenate([out,nxt],axis=1)
        step+=1
        if step>12: break
    while out.shape[1]<target:                        # last resort
        out=np.concatenate([out,base],axis=1)
    return out[:,:target]

def zero_pad(i,target):
    a=feat(i); c=a.shape[1]
    return np.pad(a,[(0,0),(0,target-c)]) if c<target else a[:,:target]

print("\n"+"="*70); print("VERIFICATION"); print("="*70)
for T in [192,256,320]:
    w=dev.fp.map(lambda p: np.load(p,mmap_mode='r').shape[1])
    short=(w<T)
    print(f"\n  target {T} frames ({T*256/16000:.2f}s): "
          f"{short.mean()*100:.1f}% of cycles need padding")

print("\n  sample traces (target 256):")
tried=0
for i in range(len(dev)):
    if feat(i).shape[1]>=256: continue
    rec,c,lbl=dev.rec[i],dev.cyc[i],dev.sound_label[i]
    nb=[(o,dev.sound_label[IDX[(rec,o)]]) for o in (c-1,c+1) if (rec,o) in IDX]
    sp=smart_pad(i,256); zp=zero_pad(i,256)
    print(f"    {dev.audio_file[i][:46]:46s} w={feat(i).shape[1]:3d} lbl={lbl} "
          f"neighbours={nb}")
    print(f"        smartpad std={sp.std():.3f} zeros={np.mean(sp==0)*100:5.1f}%  |  "
          f"zeropad std={zp.std():.3f} zeros={np.mean(zp==0)*100:5.1f}%")
    tried+=1
    if tried>=6: break

print("\n  aggregate over 300 short samples, target 256:")
sz,zz=[],[]
n=0
for i in range(len(dev)):
    if feat(i).shape[1]>=256: continue
    sz.append(np.mean(smart_pad(i,256)==0)); zz.append(np.mean(zero_pad(i,256)==0))
    n+=1
    if n>=300: break
print(f"    mean zero-fraction  smartpad {np.mean(sz)*100:.1f}%   "
      f"zeropad {np.mean(zz)*100:.1f}%")
print("\n  If smartpad zero-fraction is much lower, the padding is working.")

extracting features...


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/pulmoscan semester 7/data/pulmoscan_data.zip'

In [ ]:
# ============================================================
# SMARTPAD CELL 2 — PRE-REGISTRATION (run before training)
# ============================================================
import json, datetime, os
PROJ='/content/drive/MyDrive/pulmoscan semester 7'
proto={
 "written_at": datetime.datetime.now().isoformat(),
 "experiment": "smart padding vs zero padding across analysis windows",
 "motivation": ("Arm D (256f, zero-pad) scored 46.54% official vs Arm B (192f) 53.86%, "
                "recorded as a negative. RespireNet (Gairola et al. EMBC 2021) reports 4s > 3s "
                "and smart padding beating zero-padding by 5%. Hypothesis: Arm D failed due to "
                "padding, not window length. 88.6% of cycles require padding at 256 frames."),
 "arms": {"E_192f_smartpad":192, "F_256f_smartpad":256, "G_320f_smartpad":320},
 "baselines_from_ledger": {"B_192f_zeropad":"CV pending", "D_256f_zeropad":"CV pending"},
 "protocol": "5-fold patient-grouped CV, 3 seeds per fold, 15 runs per arm",
 "padding": "same-recording neighbour if same class or Normal, else repeat self",
 "unchanged": "architecture, focal loss gamma 4.0, balanced class weights, cosine LR, patience 20",
 "primary_test": "F_256f_smartpad vs D_256f_zeropad, difference in mean official ICBHI > pooled SD",
 "secondary_test": "does fold-level SD fall below the 11.09 measured for 192f zero-pad",
 "if_F_beats_D": "padding explains the Arm D negative; report and test G",
 "if_F_fails": "window length is the binding constraint; RespireNet's padding does not transfer",
 "caveat": ("at 256f, 88.6% of inputs contain concatenated audio from neighbouring cycles of the "
            "same recording; no cross-patient leakage, but effective independent sample count is "
            "lower than the row count"),
 "commitment": "all arms reported regardless of outcome; no post-hoc arm selection"
}
json.dump(proto,open(os.path.join(PROJ,'results/smartpad_protocol.json'),'w'),indent=2)
print(json.dumps(proto,indent=2))

{
  "written_at": "2026-09-08T04:29:23.307835",
  "experiment": "smart padding vs zero padding across analysis windows",
  "motivation": "Arm D (256f, zero-pad) scored 46.54% official vs Arm B (192f) 53.86%, recorded as a negative. RespireNet (Gairola et al. EMBC 2021) reports 4s > 3s and smart padding beating zero-padding by 5%. Hypothesis: Arm D failed due to padding, not window length. 88.6% of cycles require padding at 256 frames.",
  "arms": {
    "E_192f_smartpad": 192,
    "F_256f_smartpad": 256,
    "G_320f_smartpad": 320
  },
  "baselines_from_ledger": {
    "B_192f_zeropad": "CV pending",
    "D_256f_zeropad": "CV pending"
  },
  "protocol": "5-fold patient-grouped CV, 3 seeds per fold, 15 runs per arm",
  "padding": "same-recording neighbour if same class or Normal, else repeat self",
  "unchanged": "architecture, focal loss gamma 4.0, balanced class weights, cosine LR, patience 20",
  "primary_test": "F_256f_smartpad vs D_256f_zeropad, difference in mean official ICBHI > po

In [ ]:
# ============================================================
# SMARTPAD CELL 3 — CV TRAINING (resumable; primary test first)
# ============================================================
import os, re, json, time, random, gc, zipfile
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.metrics import confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
PROJ='/content/drive/MyDrive/pulmoscan semester 7'; WORK='/content/work'
RES=os.path.join(PROJ,'results'); LED=os.path.join(RES,'smartpad_cv.json')
N_MELS=128; EPOCHS=80; BATCH=32; LR=1e-3; LR_MIN=1e-6; DIAG_W=0.1; NS=4; ND=7
SOUND=['Normal','Crackle','Wheeze','Both']

os.makedirs(WORK,exist_ok=True)
if not os.path.isdir(os.path.join(WORK,'feat','processed','train')):
    with zipfile.ZipFile(os.path.join(PROJ,'data','pulmoscan_data.zip')) as z:
        z.extractall(os.path.join(WORK,'feat'))
with zipfile.ZipFile(os.path.join(PROJ,'data','colab_training_package.zip')) as z:
    z.extract('data/processed/manifest.csv', WORK)

man=pd.read_csv(os.path.join(WORK,'data/processed/manifest.csv'))
dev=man[man.split.isin(['train','val'])].copy().reset_index(drop=True)
dev['fp']=dev.apply(lambda r: os.path.join(WORK,'feat','processed',r['split'],
                    r['audio_file'].replace('.wav','.npy')),axis=1)
dev=dev[dev.fp.map(os.path.exists)].reset_index(drop=True)
assert 'test' not in set(dev.split)
def parse(fn):
    m=re.match(r'^(.*)_cycle(\d+)_([a-z]+)\.wav$',fn,re.I)
    return (m.group(1),int(m.group(2))) if m else (None,None)
dev[['rec','cyc']]=dev.audio_file.apply(lambda f: pd.Series(parse(f)))
IDX={(r.rec,r.cyc):i for i,r in dev.iterrows()}
RAW={i:np.load(dev.fp[i]) for i in range(len(dev))}
print(f"{len(dev)} rows, {dev.patient_id.nunique()} patients loaded")

def smart_pad(i,T):
    out=RAW[i]
    if out.shape[1]>=T: return out[:,:T]
    lbl=dev.sound_label[i]; rec=dev.rec[i]; c=dev.cyc[i]; step=1
    while out.shape[1]<T and step<=12:
        cand=None
        for off in (c+step,c-step):
            j=IDX.get((rec,off))
            if j is not None and (dev.sound_label[j]==lbl or dev.sound_label[j]==0):
                cand=j; break
        out=np.concatenate([out, RAW[cand] if cand is not None else RAW[i]],axis=1)
        step+=1
    while out.shape[1]<T: out=np.concatenate([out,RAW[i]],axis=1)
    return out[:,:T]
def zero_pad(i,T):
    a=RAW[i]; c=a.shape[1]
    return np.pad(a,[(0,0),(0,T-c)]) if c<T else a[:,:T]

def build_X(T,mode):
    pad = smart_pad if mode=='smart' else zero_pad
    X=np.empty((len(dev),N_MELS,T,1),np.float32)
    for i in range(len(dev)): X[i,...,0]=pad(i,T)
    return X
ys=dev.sound_label.values.astype(np.int32)
yd=dev.diagnosis_label.values.astype(np.int32)
pid=dev.patient_id.values

rng=np.random.RandomState(42)
pdf=dev.groupby('patient_id')['diagnosis_label'].agg(lambda x:x.mode()[0]).reset_index()
folds=[[] for _ in range(5)]
for _,g in pdf.groupby('diagnosis_label'):
    ps=g.patient_id.values.copy(); rng.shuffle(ps)
    for j,p in enumerate(ps): folds[j%5].append(p)

def macro(yt,yp,n=4):
    cm=confusion_matrix(yt,yp,labels=list(range(n))); se=[];sp=[]
    for i in range(n):
        tp=cm[i,i];fn=cm[i,:].sum()-tp;fp=cm[:,i].sum()-tp;tn=cm.sum()-tp-fn-fp
        se.append(tp/(tp+fn) if tp+fn else 0.); sp.append(tn/(tn+fp) if tn+fp else 0.)
    return (np.mean(se)+np.mean(sp))/2
def official(yt,yp):
    cm=confusion_matrix(yt,yp,labels=[0,1,2,3])
    sp=cm[0,0]/cm[0].sum() if cm[0].sum() else 0.
    se=sum(cm[i,i] for i in [1,2,3])/sum(cm[i].sum() for i in [1,2,3])
    return (se+sp)/2,se,sp
def focal(g=2.0,cw=None):
    def fn(yt,yp):
        yp=tf.clip_by_value(yp,1e-7,1.0); ce=-yt*tf.math.log(yp)
        p=tf.reduce_sum(yt*yp,axis=-1,keepdims=True); f=tf.pow(1.0-p,g)*ce
        if cw is not None: f=f*tf.reduce_sum(yt*tf.constant(cw,tf.float32),axis=-1,keepdims=True)
        return tf.reduce_mean(f)
    fn.__name__='focal'; return fn
def build(T):
    inp=tf.keras.Input(shape=(N_MELS,T,1)); x=inp
    for f in [32,64,128]:
        x=tf.keras.layers.Conv2D(f,3,padding='same',activation='relu')(x)
        x=tf.keras.layers.BatchNormalization()(x); x=tf.keras.layers.MaxPooling2D(2)(x)
    x=tf.keras.layers.Conv2D(256,3,padding='same',activation='relu')(x)
    x=tf.keras.layers.BatchNormalization()(x)
    x=tf.keras.layers.GlobalAveragePooling2D()(x)
    sh=tf.keras.layers.Dropout(0.5)(tf.keras.layers.Dense(256,activation='relu')(x))
    s=tf.keras.layers.Dropout(0.3)(tf.keras.layers.Dense(128,activation='relu')(sh))
    so=tf.keras.layers.Dense(NS,activation='softmax',name='sound')(s)
    d=tf.keras.layers.Dropout(0.3)(tf.keras.layers.Dense(128,activation='relu')(sh))
    do=tf.keras.layers.Dense(ND,activation='softmax',name='diagnosis')(d)
    return tf.keras.Model(inp,[so,do])
class Cb(tf.keras.callbacks.Callback):
    def __init__(s,X,y): super().__init__(); s.X=X;s.y=y;s.best=0.;s.w=None
    def on_epoch_end(s,e,logs=None):
        p=np.argmax(s.model.predict(s.X,verbose=0)[0],axis=1)
        sc=macro(s.y,p); logs['val_icbhi']=sc
        if sc>s.best: s.best=sc; s.w=s.model.get_weights()

XC={}
def run(arm,T,mode,k,seed):
    tag=f'{arm}_f{k}_s{seed}'
    L=json.load(open(LED)) if os.path.exists(LED) else {}
    if tag in L: print(f"  SKIP {tag} ({L[tag]['official']*100:.2f}%)"); return
    key=(T,mode)
    if key not in XC:
        XC.clear(); print(f"  building X for {T}f {mode}pad..."); t=time.time()
        XC[key]=build_X(T,mode); print(f"    {time.time()-t:.0f}s")
    X=XC[key]
    te=np.isin(pid,folds[k]); tr=~te
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    tf.keras.backend.clear_session()
    cw=compute_class_weight('balanced',classes=np.unique(ys[tr]),y=ys[tr])
    m=build(T); m.compile(optimizer=tf.keras.optimizers.Adam(LR),
        loss={'sound':focal(4.0,[float(c) for c in cw]),'diagnosis':focal(2.0)},
        loss_weights={'sound':1.0,'diagnosis':DIAG_W},metrics={'sound':'accuracy'})
    cb=Cb(X[te],ys[te]); t0=time.time()
    m.fit(X[tr],{'sound':tf.keras.utils.to_categorical(ys[tr],NS),
                 'diagnosis':tf.keras.utils.to_categorical(yd[tr],ND)},
          validation_data=(X[te],{'sound':tf.keras.utils.to_categorical(ys[te],NS),
                                  'diagnosis':tf.keras.utils.to_categorical(yd[te],ND)}),
          batch_size=BATCH,epochs=EPOCHS,verbose=0,
          callbacks=[cb,tf.keras.callbacks.EarlyStopping(monitor='val_icbhi',mode='max',
                        patience=20,restore_best_weights=False),
                     tf.keras.callbacks.LearningRateScheduler(
                        lambda e: float(LR_MIN+0.5*(LR-LR_MIN)*(1+np.cos(np.pi*e/EPOCHS))))])
    m.set_weights(cb.w)
    yp=np.argmax(m.predict(X[te],verbose=0)[0],axis=1)
    mac=macro(ys[te],yp); off,se,sp=official(ys[te],yp)
    print(f"  {tag}: macro {mac*100:.2f}%  official {off*100:.2f}% "
          f"(Se {se*100:.1f} Sp {sp*100:.1f})  {(time.time()-t0)/60:.1f}min")
    L=json.load(open(LED)) if os.path.exists(LED) else {}
    L[tag]={'arm':arm,'frames':T,'pad':mode,'fold':k,'seed':seed,
            'macro':float(mac),'official':float(off),'se':float(se),'sp':float(sp),
            'per_class_recall':{n:float((yp[ys[te]==i]==i).mean()) for i,n in enumerate(SOUND)},
            'confusion':confusion_matrix(ys[te],yp,labels=[0,1,2,3]).tolist()}
    json.dump(L,open(LED,'w'),indent=2)
    del m; gc.collect()

def summarize():
    if not os.path.exists(LED): return
    L=json.load(open(LED))
    print("\n"+"="*72); print("SUMMARY"); print("="*72)
    for arm in ['F_256f_smartpad','D_256f_zeropad','E_192f_smartpad',
                'B_192f_zeropad','G_320f_smartpad']:
        r=[v for v in L.values() if v['arm']==arm]
        if not r: continue
        o=[v['official'] for v in r]; mc=[v['macro'] for v in r]
        print(f"  {arm:20s} official {np.mean(o)*100:6.2f}% ± {np.std(o)*100:5.2f}   "
              f"macro {np.mean(mc)*100:6.2f}% ± {np.std(mc)*100:5.2f}   (n={len(r)})")
    def cmp(a,b):
        A=[v['official'] for v in L.values() if v['arm']==a]
        B=[v['official'] for v in L.values() if v['arm']==b]
        if not A or not B: return
        d=(np.mean(A)-np.mean(B))*100; s=np.sqrt((np.var(A)+np.var(B))/2)*100
        print(f"  {a} vs {b}: {d:+.2f}pt, pooled SD {s:.2f} -> "
              f"{'PASS' if d>s else 'FAIL'}")
    print()
    cmp('F_256f_smartpad','D_256f_zeropad')
    cmp('E_192f_smartpad','B_192f_zeropad')
    cmp('G_320f_smartpad','F_256f_smartpad')

# ---------- PRIMARY TEST (30 runs, ~2h) ----------
print("\n### PRIMARY: 256f smartpad vs 256f zeropad ###")
for k in range(5):
    for s in [0,1,2]: run('F_256f_smartpad',256,'smart',k,s)
for k in range(5):
    for s in [0,1,2]: run('D_256f_zeropad',256,'zero',k,s)
summarize()

# ---------- SECONDARY (uncomment when quota allows) ----------
# for k in range(5):
#     for s in [0,1,2]: run('E_192f_smartpad',192,'smart',k,s)
# for k in range(5):
#     for s in [0,1,2]: run('B_192f_zeropad',192,'zero',k,s)
# for k in range(5):
#     for s in [0,1,2]: run('G_320f_smartpad',320,'smart',k,s)
# summarize()

3690 rows, 82 patients loaded

### PRIMARY: 256f smartpad vs 256f zeropad ###
  building X for 256f smartpad...
    1s


In [ ]:
import torch, sys
print("python", sys.version.split()[0])
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

python 3.13.15
cuda available: True
gpu: Tesla T4


In [ ]:
import os
root = "/content/drive/MyDrive"
for f in sorted(os.listdir(root)):
    print(repr(f))

'11 cbse science'
'14th oct - aqua line and marines '
'16010422269-OOSE EXP7 (Hasya Abburi).docx'
'16010422269_IBB EXPERIMENTS_(Hasya Abburi).gdoc'
'16010422269_OOSE IA2_Task Management System.mp4'
'16010422269_OOSE_SRS TAGS.gdoc'
'16010422269_STD LATEX DOCUMENT.pdf'
'16010422269_Study of applications of microorganisms in food biotechnology.gdoc'
'20251014_124144.mp4'
'20251014_124211PMByGPSMapCamera.mp4'
'20251014_124316.mp4'
'20260209_53909PMByGPSMapCamera.jpg'
'3D CNN.gdoc'
'AD PYQS SCHEMA (?).gdoc'
'AD notes.gdoc'
'Adobe Scan Nov 16, 2022.pdf'
'AlphaPulse.mp4'
'CC ISE NOTES.gdoc'
'Classroom'
'Colab Notebooks'
'Copy of 50 Free HTML & CSS Coding Templates — HubSpot.gdoc'
'DL EXP1 AND EXP2 VIVA NOTES.gdoc'
'DL IA2 - Deep Learning for Network Intrusion Detection Systems: A Comparative Study.gdoc'
'DL ISE NOTES.gdoc'
'DOC-20250122-WA0009. (1).pdf'
'DS 2022 THEORY.gdoc'
'Drowsiness Detection System for Drivers.gdoc'
'EY TECHATHON 6.0 - CLUB SANDWICH (PHARMA)'
'Exploring the Vast Cosmos: 

In [ ]:
import os
root = "/content/drive/MyDrive"
for d in ["pulmoscan semester 7", "pulmoscan drive", "checkpoints", "processed", "results", "Colab Notebooks"]:
    p = os.path.join(root, d)
    print("\n=====", d, "=====")
    if not os.path.isdir(p):
        print("not found")
        continue
    items = sorted(os.listdir(p))
    print(len(items), "items")
    for f in items[:40]:
        fp = os.path.join(p, f)
        size = os.path.getsize(fp) / 1e6 if os.path.isfile(fp) else 0
        print(f"  {f:65s} {size:8.1f} MB")
    if len(items) > 40:
        print("  ... and", len(items) - 40, "more")


===== pulmoscan semester 7 =====
6 items
  V9_test02_buckboost.png                                                0.3 MB
  V9b_test02_ltspice_raw.png                                             0.2 MB
  checkpoints                                                            0.0 MB
  data                                                                   0.0 MB
  notebooks                                                              0.0 MB
  results                                                                0.0 MB

===== pulmoscan drive =====
3 items
  data                                                                   0.0 MB
  src                                                                    0.0 MB
  train_highres.py                                                       0.0 MB

===== checkpoints =====
2 items
  cnn_baseline_best.keras                                                1.4 MB
  multitask_best.keras                                                   6.4 MB

===== p

In [ ]:
import tensorflow as tf
print("tf", tf.__version__)
print("gpus:", tf.config.list_physical_devices('GPU'))

tf 2.20.0
gpus: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
import os
root = "/content/drive/MyDrive/pulmoscan semester 7"
for dirpath, dirnames, filenames in os.walk(root):
    depth = dirpath[len(root):].count(os.sep)
    if depth > 2:
        continue
    print("\n" + dirpath.replace(root, "[root]"))
    for f in sorted(filenames)[:30]:
        size = os.path.getsize(os.path.join(dirpath, f)) / 1e6
        print(f"   {f:65s} {size:8.1f} MB")
    if len(filenames) > 30:
        print("   ... and", len(filenames) - 30, "more files")


[root]
   V9_test02_buckboost.png                                                0.3 MB
   V9b_test02_ltspice_raw.png                                             0.2 MB

[root]/data
   colab_training_package.zip                                          1634.4 MB
   pulmoscan_data.zip                                                   344.9 MB
   pulmoscan_raw_audio.zip                                              464.5 MB
   test_features.zip                                                    128.1 MB

[root]/notebooks
   multi_res_and_handcrafted.ipynb                                        0.6 MB
   pulmoscan_multitask_hf_lung_gpu.ipynb                                  0.2 MB
   se_attention_eval.ipynb                                                0.1 MB

[root]/results
   diagnosis_head_test.json                                               0.0 MB
   ensemble_component_training_curves.png                                 0.3 MB
   ensemble_construction_chart.png                    

In [ ]:
import json, os
R = "/content/drive/MyDrive/pulmoscan semester 7/results"
for name in ["sem7_window_runs.json", "sem7_cv_runs.json", "smartpad_protocol.json",
             "test_protocol.json", "test_result_FINAL.json", "ensemble_results.json",
             "diagnosis_head_test.json"]:
    p = os.path.join(R, name)
    print("\n=====", name, "=====")
    with open(p) as f:
        txt = json.dumps(json.load(f), indent=1)
    print(txt[:3500])
    if len(txt) > 3500:
        print("... [truncated,", len(txt), "chars total]")


===== sem7_window_runs.json =====
{
 "A_63f_gap_s0": {
  "frames": 63,
  "pooling": "gap",
  "seed": 0,
  "val_icbhi_macro": 0.612427041786832,
  "val_icbhi_official": 0.5411950788490334,
  "se": 0.41638225255972694,
  "sp": 0.66600790513834,
  "per_class_recall": {
   "Normal": 0.66600790513834,
   "Crackle": 0.5196078431372549,
   "Wheeze": 0.1,
   "Both": 0.3448275862068966
  },
  "normal_to_crackle": [
   131,
   506
  ],
  "confusion": [
   [
    337,
    131,
    14,
    24
   ],
   [
    71,
    106,
    3,
    24
   ],
   [
    28,
    22,
    6,
    4
   ],
   [
    8,
    10,
    1,
    10
   ]
  ],
  "epochs_run": 43,
  "minutes": 1.7,
  "params": 522763
 },
 "A_63f_gap_s1": {
  "frames": 63,
  "pooling": "gap",
  "seed": 1,
  "val_icbhi_macro": 0.5839533734390856,
  "val_icbhi_official": 0.48666513780032106,
  "se": 0.44368600682593856,
  "sp": 0.5296442687747036,
  "per_class_recall": {
   "Normal": 0.5296442687747036,
   "Crackle": 0.553921568627451,
   "Wheeze": 0.15,
 

In [ ]:
import json, os
NB = "/content/drive/MyDrive/Colab Notebooks"
files = ["sem7_smartpad_window_sweep.ipynb", "sem7_step0_2_window_truncation_diagnostic.ipynb",
         "pulmoscan.ipynb", "new-respiratory-multitask.ipynb"]
keys = ["focal", "gamma", "class_weight", "monitor", "ModelCheckpoint", "EarlyStopping",
        "GroupKFold", "def icbhi", "def official", "restore_best"]
for fn in files:
    p = os.path.join(NB, fn)
    print("\n=====", fn, "=====")
    nb = json.load(open(p))
    shown = 0
    for i, c in enumerate(nb["cells"]):
        if c["cell_type"] != "code":
            continue
        for line in "".join(c["source"]).splitlines():
            if any(k in line for k in keys) and shown < 50:
                print(f"[cell {i}] {line.strip()[:150]}")
                shown += 1


===== sem7_smartpad_window_sweep.ipynb =====
[cell 1] "unchanged": "architecture, focal loss gamma 4.0, balanced class weights, cosine LR, patience 20",
[cell 2] from sklearn.utils.class_weight import compute_class_weight
[cell 2] def official(yt,yp):
[cell 2] def focal(g=2.0,cw=None):
[cell 2] fn.__name__='focal'; return fn
[cell 2] cw=compute_class_weight('balanced',classes=np.unique(ys[tr]),y=ys[tr])
[cell 2] loss={'sound':focal(4.0,[float(c) for c in cw]),'diagnosis':focal(2.0)},
[cell 2] callbacks=[cb,tf.keras.callbacks.EarlyStopping(monitor='val_icbhi',mode='max',
[cell 2] patience=20,restore_best_weights=False),

===== sem7_step0_2_window_truncation_diagnostic.ipynb =====
[cell 5] def official_icbhi(rec, sup=SUP):
[cell 5] from augmentation import compute_class_weights
[cell 5] print("  computed:", compute_class_weights(tr.sound_label.values,4))
[cell 5] print(f"  {mod.__name__}.{fn}: 'class_weight=' passed to fit? "
[cell 5] f"{'YES' if 'class_weight=' in src else 'NO  <-- wei

In [ ]:
import json
nb = json.load(open("/content/drive/MyDrive/Colab Notebooks/sem7_smartpad_window_sweep.ipynb"))
print("total cells:", len(nb["cells"]))
for i, c in enumerate(nb["cells"]):
    first = "".join(c["source"]).strip().splitlines()[:1]
    print(i, c["cell_type"], first)
print("\n===== CELL 2 FULL =====\n")
print("".join(nb["cells"][2]["source"]))

total cells: 11
0 code ['# ============================================================']
1 code ['# ============================================================']
2 code ['# ============================================================']
3 code ['import torch, sys']
4 code ['import os']
5 code ['import os']
6 code ['import tensorflow as tf']
7 code ['import os']
8 code ['import json, os']
9 code ['import json, os']
10 code []

===== CELL 2 FULL =====

# ============================================================
# SMARTPAD CELL 3 — CV TRAINING (resumable; primary test first)
# ============================================================
import os, re, json, time, random, gc, zipfile
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.metrics import confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
PROJ='/content/drive/MyDrive/pulmos

In [ ]:
import json, os, datetime
RES = "/content/drive/MyDrive/pulmoscan semester 7/results"
assert not os.path.exists(os.path.join(RES, "smartpad_cv.json")), "runs already exist, stop and tell me"
P = os.path.join(RES, "smartpad_protocol.json")
d = json.load(open(P))
d["amendment_1"] = {
  "written_at": datetime.datetime.now().isoformat(),
  "made_before_any_smartpad_cv_run": True,
  "changes": [
    "held out fold no longer used for epoch selection: fixed 40 epochs, final weights, no early stopping",
    "smart padding is label free on held out rows (label aware padding kept for training rows only)",
    "primary metric unchanged: official ICBHI"],
  "reasons": [
    "original cell selected best epoch by macro on the held out fold, which is test set selection",
    "original smart_pad read the true label of held out samples, which leaks the answer",
    "40 epochs chosen because sweep best epochs (epochs_run minus patience) were 20 to 39"]}
json.dump(d, open(P, "w"), indent=2)
print("amendment written")

amendment written


In [ ]:
# ============================================================
# SMARTPAD CELL 3 (AMENDED) - CV TRAINING, no test fold selection
# ============================================================
import os, re, json, time, random, gc, zipfile
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.metrics import confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
PROJ='/content/drive/MyDrive/pulmoscan semester 7'; WORK='/content/work'
RES=os.path.join(PROJ,'results'); LED=os.path.join(RES,'smartpad_cv.json')
N_MELS=128; EPOCHS=40; BATCH=32; LR=1e-3; LR_MIN=1e-6; DIAG_W=0.1; NS=4; ND=7
SOUND=['Normal','Crackle','Wheeze','Both']

os.makedirs(WORK,exist_ok=True)
if not os.path.isdir(os.path.join(WORK,'feat','processed','train')):
    with zipfile.ZipFile(os.path.join(PROJ,'data','pulmoscan_data.zip')) as z:
        z.extractall(os.path.join(WORK,'feat'))
with zipfile.ZipFile(os.path.join(PROJ,'data','colab_training_package.zip')) as z:
    z.extract('data/processed/manifest.csv', WORK)

man=pd.read_csv(os.path.join(WORK,'data/processed/manifest.csv'))
dev=man[man.split.isin(['train','val'])].copy().reset_index(drop=True)
dev['fp']=dev.apply(lambda r: os.path.join(WORK,'feat','processed',r['split'],
                    r['audio_file'].replace('.wav','.npy')),axis=1)
dev=dev[dev.fp.map(os.path.exists)].reset_index(drop=True)
assert 'test' not in set(dev.split)
def parse(fn):
    m=re.match(r'^(.*)_cycle(\d+)_([a-z]+)\.wav$',fn,re.I)
    return (m.group(1),int(m.group(2))) if m else (None,None)
dev[['rec','cyc']]=dev.audio_file.apply(lambda f: pd.Series(parse(f)))
IDX={(r.rec,r.cyc):i for i,r in dev.iterrows()}
RAW={i:np.load(dev.fp[i]) for i in range(len(dev))}
print(f"{len(dev)} rows, {dev.patient_id.nunique()} patients loaded")

# feature check: is zero a sensible padding value?
_s=np.concatenate([RAW[i].ravel() for i in range(0,len(dev),40)])
print("feature stats  min %.2f  max %.2f  mean %.2f  std %.2f"%(_s.min(),_s.max(),_s.mean(),_s.std()))

def smart_pad(i,T,use_label=True):
    out=RAW[i]
    if out.shape[1]>=T: return out[:,:T]
    lbl=dev.sound_label[i]; rec=dev.rec[i]; c=dev.cyc[i]; step=1
    while out.shape[1]<T and step<=12:
        cand=None
        for off in (c+step,c-step):
            j=IDX.get((rec,off))
            if j is not None and (not use_label or dev.sound_label[j]==lbl or dev.sound_label[j]==0):
                cand=j; break
        out=np.concatenate([out, RAW[cand] if cand is not None else RAW[i]],axis=1)
        step+=1
    while out.shape[1]<T: out=np.concatenate([out,RAW[i]],axis=1)
    return out[:,:T]
def zero_pad(i,T):
    a=RAW[i]; c=a.shape[1]
    return np.pad(a,[(0,0),(0,T-c)]) if c<T else a[:,:T]

def build_X(T,mode,use_label=True):
    X=np.empty((len(dev),N_MELS,T,1),np.float32)
    for i in range(len(dev)):
        X[i,...,0]=smart_pad(i,T,use_label) if mode=='smart' else zero_pad(i,T)
    return X
ys=dev.sound_label.values.astype(np.int32)
yd=dev.diagnosis_label.values.astype(np.int32)
pid=dev.patient_id.values

rng=np.random.RandomState(42)
pdf=dev.groupby('patient_id')['diagnosis_label'].agg(lambda x:x.mode()[0]).reset_index()
folds=[[] for _ in range(5)]
for _,g in pdf.groupby('diagnosis_label'):
    ps=g.patient_id.values.copy(); rng.shuffle(ps)
    for j,p in enumerate(ps): folds[j%5].append(p)

def macro(yt,yp,n=4):
    cm=confusion_matrix(yt,yp,labels=list(range(n))); se=[];sp=[]
    for i in range(n):
        tp=cm[i,i];fn=cm[i,:].sum()-tp;fp=cm[:,i].sum()-tp;tn=cm.sum()-tp-fn-fp
        se.append(tp/(tp+fn) if tp+fn else 0.); sp.append(tn/(tn+fp) if tn+fp else 0.)
    return (np.mean(se)+np.mean(sp))/2
def official(yt,yp):
    cm=confusion_matrix(yt,yp,labels=[0,1,2,3])
    sp=cm[0,0]/cm[0].sum() if cm[0].sum() else 0.
    se=sum(cm[i,i] for i in [1,2,3])/sum(cm[i].sum() for i in [1,2,3])
    return (se+sp)/2,se,sp
def focal(g=2.0,cw=None):
    def fn(yt,yp):
        yp=tf.clip_by_value(yp,1e-7,1.0); ce=-yt*tf.math.log(yp)
        p=tf.reduce_sum(yt*yp,axis=-1,keepdims=True); f=tf.pow(1.0-p,g)*ce
        if cw is not None: f=f*tf.reduce_sum(yt*tf.constant(cw,tf.float32),axis=-1,keepdims=True)
        return tf.reduce_mean(f)
    fn.__name__='focal'; return fn
def build(T):
    inp=tf.keras.Input(shape=(N_MELS,T,1)); x=inp
    for f in [32,64,128]:
        x=tf.keras.layers.Conv2D(f,3,padding='same',activation='relu')(x)
        x=tf.keras.layers.BatchNormalization()(x); x=tf.keras.layers.MaxPooling2D(2)(x)
    x=tf.keras.layers.Conv2D(256,3,padding='same',activation='relu')(x)
    x=tf.keras.layers.BatchNormalization()(x)
    x=tf.keras.layers.GlobalAveragePooling2D()(x)
    sh=tf.keras.layers.Dropout(0.5)(tf.keras.layers.Dense(256,activation='relu')(x))
    s=tf.keras.layers.Dropout(0.3)(tf.keras.layers.Dense(128,activation='relu')(sh))
    so=tf.keras.layers.Dense(NS,activation='softmax',name='sound')(s)
    d=tf.keras.layers.Dropout(0.3)(tf.keras.layers.Dense(128,activation='relu')(sh))
    do=tf.keras.layers.Dense(ND,activation='softmax',name='diagnosis')(d)
    return tf.keras.Model(inp,[so,do])

XC={}
def run(arm,T,mode,k,seed):
    tag=f'{arm}_f{k}_s{seed}'
    L=json.load(open(LED)) if os.path.exists(LED) else {}
    if tag in L: print(f"  SKIP {tag} ({L[tag]['official']*100:.2f}%)"); return
    key=(T,mode)
    if key not in XC:
        XC.clear(); print(f"  building X for {T}f {mode}pad..."); t=time.time()
        XT=build_X(T,mode,True)                              # training rows: label aware padding
        XE=build_X(T,mode,False) if mode=='smart' else XT    # held out rows: label free padding
        XC[key]=(XT,XE); print(f"    {time.time()-t:.0f}s")
    XT,XE=XC[key]
    te=np.isin(pid,folds[k]); tr=~te
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    tf.keras.backend.clear_session()
    cw=compute_class_weight('balanced',classes=np.unique(ys[tr]),y=ys[tr])
    m=build(T); m.compile(optimizer=tf.keras.optimizers.Adam(LR),
        loss={'sound':focal(4.0,[float(c) for c in cw]),'diagnosis':focal(2.0)},
        loss_weights={'sound':1.0,'diagnosis':DIAG_W},metrics={'sound':'accuracy'})
    t0=time.time()
    m.fit(XT[tr],{'sound':tf.keras.utils.to_categorical(ys[tr],NS),
                  'diagnosis':tf.keras.utils.to_categorical(yd[tr],ND)},
          batch_size=BATCH,epochs=EPOCHS,verbose=0,
          callbacks=[tf.keras.callbacks.LearningRateScheduler(
              lambda e: float(LR_MIN+0.5*(LR-LR_MIN)*(1+np.cos(np.pi*e/EPOCHS))))])
    yp=np.argmax(m.predict(XE[te],verbose=0)[0],axis=1)      # final weights, no epoch picking
    mac=macro(ys[te],yp); off,se,sp=official(ys[te],yp)
    print(f"  {tag}: macro {mac*100:.2f}%  official {off*100:.2f}% "
          f"(Se {se*100:.1f} Sp {sp*100:.1f})  {(time.time()-t0)/60:.1f}min")
    L=json.load(open(LED)) if os.path.exists(LED) else {}
    L[tag]={'arm':arm,'frames':T,'pad':mode,'fold':k,'seed':seed,
            'macro':float(mac),'official':float(off),'se':float(se),'sp':float(sp),
            'per_class_recall':{n:float((yp[ys[te]==i]==i).mean()) for i,n in enumerate(SOUND)},
            'confusion':confusion_matrix(ys[te],yp,labels=[0,1,2,3]).tolist()}
    json.dump(L,open(LED,'w'),indent=2)
    del m; gc.collect()

def summarize():
    if not os.path.exists(LED): return
    L=json.load(open(LED))
    print("\n"+"="*72); print("SUMMARY"); print("="*72)
    for arm in ['F_256f_smartpad','D_256f_zeropad','E_192f_smartpad',
                'B_192f_zeropad','G_320f_smartpad']:
        r=[v for v in L.values() if v['arm']==arm]
        if not r: continue
        o=[v['official'] for v in r]; mc=[v['macro'] for v in r]
        print(f"  {arm:20s} official {np.mean(o)*100:6.2f}% +/- {np.std(o)*100:5.2f}   "
              f"macro {np.mean(mc)*100:6.2f}% +/- {np.std(mc)*100:5.2f}   (n={len(r)})")

# ---------- SMOKE TEST ONLY: fold 0, seed 0 ----------
run('D_256f_zeropad',256,'zero',0,0)
run('F_256f_smartpad',256,'smart',0,0)
summarize()

3690 rows, 82 patients loaded
feature stats  min -5.23  max 5.59  mean 0.00  std 1.00
  building X for 256f zeropad...
    0s
  D_256f_zeropad_f0_s0: macro 62.47%  official 49.14% (Se 53.3 Sp 44.9)  2.7min
  building X for 256f smartpad...
    1s
  F_256f_smartpad_f0_s0: macro 59.45%  official 45.45% (Se 47.3 Sp 43.6)  2.7min

SUMMARY
  F_256f_smartpad      official  45.45% +/-  0.00   macro  59.45% +/-  0.00   (n=1)
  D_256f_zeropad       official  49.14% +/-  0.00   macro  62.47% +/-  0.00   (n=1)


In [ ]:
# ============================================================
# GAMMA / CLASS WEIGHT DIAGNOSTIC (fold 0, 192f zero pad, 3 seeds)
# run in the same session, after the amended CV cell
# ============================================================
import datetime
GP=os.path.join(RES,'gamma_protocol.json'); GL=os.path.join(RES,'gamma_ablation.json')
ARMS={'H0_g4_cw':(4.0,True),'H1_g2_cw':(2.0,True),'H2_g2_nocw':(2.0,False)}

if not os.path.exists(GP):
    json.dump({
      'written_at':datetime.datetime.now().isoformat(),
      'question':'is the Normal vs Crackle boundary instability caused by focal gamma 4 stacked on balanced class weights',
      'setup':'fold 0, 192 frames, zero pad, 3 seeds, 40 epochs, final weights, no epoch selection on held out',
      'arms':{k:{'gamma':v[0],'class_weights':v[1]} for k,v in ARMS.items()},
      'primary':'SD of official ICBHI across the 3 seeds, then mean official',
      'note':'fold 0 only, a diagnostic and not proof; any better arm must be confirmed on all 5 folds before adoption',
      'commitment':'all arms reported, no arm dropped'}, open(GP,'w'), indent=2)
    print('protocol written')

def run_g(arm,T,k,seed):
    tag=f'{arm}_f{k}_s{seed}'
    L=json.load(open(GL)) if os.path.exists(GL) else {}
    if tag in L: print(f'  SKIP {tag} ({L[tag]["official"]*100:.2f}%)'); return
    gamma,use_cw=ARMS[arm]
    key=(T,'zero')
    if key not in XC:
        XC.clear(); print(f'  building X for {T}f zeropad...')
        XT=build_X(T,'zero',True); XC[key]=(XT,XT)
    XT,XE=XC[key]
    te=np.isin(pid,folds[k]); tr=~te
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    tf.keras.backend.clear_session()
    cw=[float(c) for c in compute_class_weight('balanced',classes=np.unique(ys[tr]),y=ys[tr])] if use_cw else None
    m=build(T); m.compile(optimizer=tf.keras.optimizers.Adam(LR),
        loss={'sound':focal(gamma,cw),'diagnosis':focal(2.0)},
        loss_weights={'sound':1.0,'diagnosis':DIAG_W},metrics={'sound':'accuracy'})
    t0=time.time()
    m.fit(XT[tr],{'sound':tf.keras.utils.to_categorical(ys[tr],NS),
                  'diagnosis':tf.keras.utils.to_categorical(yd[tr],ND)},
          batch_size=BATCH,epochs=EPOCHS,verbose=0,
          callbacks=[tf.keras.callbacks.LearningRateScheduler(
              lambda e: float(LR_MIN+0.5*(LR-LR_MIN)*(1+np.cos(np.pi*e/EPOCHS))))])
    yp=np.argmax(m.predict(XE[te],verbose=0)[0],axis=1)
    mac=macro(ys[te],yp); off,se,sp=official(ys[te],yp)
    rec={n:float((yp[ys[te]==i]==i).mean()) for i,n in enumerate(SOUND)}
    print(f'  {tag}: official {off*100:.2f}%  macro {mac*100:.2f}%  Normal recall {rec["Normal"]:.2f}  {(time.time()-t0)/60:.1f}min')
    L=json.load(open(GL)) if os.path.exists(GL) else {}
    L[tag]={'arm':arm,'gamma':gamma,'class_weights':use_cw,'fold':k,'seed':seed,
            'official':float(off),'macro':float(mac),'se':float(se),'sp':float(sp),
            'per_class_recall':rec,
            'confusion':confusion_matrix(ys[te],yp,labels=[0,1,2,3]).tolist()}
    json.dump(L,open(GL,'w'),indent=2)
    del m; gc.collect()

def summarize_g():
    L=json.load(open(GL))
    print('\n'+'='*72); print('GAMMA DIAGNOSTIC SUMMARY (fold 0)'); print('='*72)
    for arm in ARMS:
        r=[v for v in L.values() if v['arm']==arm]
        if not r: continue
        o=[v['official'] for v in r]; nr=[v['per_class_recall']['Normal'] for v in r]
        print(f'  {arm:12s} official mean {np.mean(o)*100:5.2f}  SD {np.std(o)*100:5.2f}  '
              f'per seed {[round(x*100,1) for x in o]}  Normal recall {[round(x,2) for x in nr]}  (n={len(r)})')

for arm in ARMS:
    for s in [0,1,2]:
        run_g(arm,192,0,s)
summarize_g()

protocol written
  building X for 192f zeropad...
  H0_g4_cw_f0_s0: official 42.99%  macro 57.83%  Normal recall 0.45  2.2min
  H0_g4_cw_f0_s1: official 46.77%  macro 60.06%  Normal recall 0.47  2.2min
  H0_g4_cw_f0_s2: official 47.72%  macro 61.04%  Normal recall 0.43  2.1min
  H1_g2_cw_f0_s0: official 43.59%  macro 58.21%  Normal recall 0.43  2.2min
  H1_g2_cw_f0_s1: official 48.24%  macro 59.53%  Normal recall 0.53  2.2min
  H1_g2_cw_f0_s2: official 48.40%  macro 60.54%  Normal recall 0.48  2.1min
  H2_g2_nocw_f0_s0: official 53.86%  macro 61.09%  Normal recall 0.65  2.1min
  H2_g2_nocw_f0_s1: official 55.29%  macro 60.20%  Normal recall 0.76  2.1min
  H2_g2_nocw_f0_s2: official 52.29%  macro 59.92%  Normal recall 0.66  2.1min

GAMMA DIAGNOSTIC SUMMARY (fold 0)
  H0_g4_cw     official mean 45.83  SD  2.04  per seed [43.0, 46.8, 47.7]  Normal recall [0.45, 0.47, 0.43]  (n=3)
  H1_g2_cw     official mean 46.74  SD  2.23  per seed [43.6, 48.2, 48.4]  Normal recall [0.43, 0.53, 0.48]  (

In [ ]:
# ============================================================
# CONFIRMATION: H0 vs H2 on folds 1 to 4 (fold 0 already done)
# same session as the previous cells; resumable
# ============================================================
FOLDS_TO_RUN=[1,2]        # first (about 26 min); then change to [3,4] and rerun
CONF=['H0_g4_cw','H2_g2_nocw']

d=json.load(open(GP))
if 'amendment_1' not in d:
    d['amendment_1']={
      'written_at':datetime.datetime.now().isoformat(),
      'written_after_fold0_results':True,
      'stage':'confirmation of H0 vs H2 on folds 1 to 4, 3 seeds each',
      'H1_dropped_because':'fold 0 showed no difference from H0 (46.74 vs 45.83, within seed SD); reported, not carried forward',
      'adoption_rule':['H2 beats H0 on at least 4 of 5 folds by fold mean official',
                       'mean paired fold difference exceeds H0 mean within fold seed SD',
                       'mean Wheeze and Both recall under H2 at least half of H0'],
      'otherwise':'keep H0 and report H2 as an operating point shift, not an improvement'}
    json.dump(d,open(GP,'w'),indent=2); print('amendment written')

for k in FOLDS_TO_RUN:
    for arm in CONF:
        for s in [0,1,2]:
            run_g(arm,192,k,s)

def summarize_confirm():
    L=json.load(open(GL)); A,B=CONF
    def runs(a,k): return [v for v in L.values() if v['arm']==a and v['fold']==k]
    print('\nfold  H0 off   H2 off    diff  H0 seedSD   H0 Se/Sp    H2 Se/Sp')
    diffs=[]; sds=[]; done=[]
    for k in range(5):
        ra,rb=runs(A,k),runs(B,k)
        if len(ra)<3 or len(rb)<3: continue
        oa=[v['official'] for v in ra]; ob=[v['official'] for v in rb]
        dd=(np.mean(ob)-np.mean(oa))*100
        diffs.append(dd); sds.append(np.std(oa)*100); done.append(k)
        sa=np.mean([v['se'] for v in ra])*100; pa=np.mean([v['sp'] for v in ra])*100
        sb=np.mean([v['se'] for v in rb])*100; pb=np.mean([v['sp'] for v in rb])*100
        print(f"  {k}   {np.mean(oa)*100:6.2f}  {np.mean(ob)*100:6.2f}  {dd:+6.2f}   {np.std(oa)*100:5.2f}     {sa:4.1f}/{pa:4.1f}   {sb:4.1f}/{pb:4.1f}")
    def rec(a,c): return np.nanmean([v['per_class_recall'][c] for k in done for v in runs(a,k)])
    print('\nmean recall over folds done:', done)
    for c in SOUND: print(f'  {c:8s} H0 {rec(A,c):.2f}   H2 {rec(B,c):.2f}')
    if len(done)==5:
        r1=sum(x>0 for x in diffs)>=4
        r2=np.mean(diffs)>np.mean(sds)
        r3=all(rec(B,c)>=0.5*rec(A,c) for c in ['Wheeze','Both'])
        print(f'\nRULE 1 wins>=4 of 5: {r1}  ({sum(x>0 for x in diffs)}/5)')
        print(f'RULE 2 mean diff {np.mean(diffs):+.2f} > seed SD {np.mean(sds):.2f}: {r2}')
        print(f'RULE 3 Wheeze and Both recall kept: {r3}')
        print('VERDICT:', 'ADOPT H2' if (r1 and r2 and r3) else 'KEEP H0, report H2 as operating point shift')
    else:
        print('\nnot all 5 folds done yet, no verdict')

summarize_confirm()

amendment written
  H0_g4_cw_f1_s0: official 49.72%  macro 64.91%  Normal recall 0.37  2.7min
  H0_g4_cw_f1_s1: official 51.25%  macro 63.44%  Normal recall 0.50  2.5min
  H0_g4_cw_f1_s2: official 49.14%  macro 64.40%  Normal recall 0.41  2.6min
  H2_g2_nocw_f1_s0: official 54.81%  macro 61.14%  Normal recall 0.76  2.6min
  H2_g2_nocw_f1_s1: official 54.74%  macro 61.93%  Normal recall 0.73  2.5min
  H2_g2_nocw_f1_s2: official 55.90%  macro 60.78%  Normal recall 0.79  2.6min
  H0_g4_cw_f2_s0: official 42.30%  macro 54.25%  Normal recall 0.62  2.5min
  H0_g4_cw_f2_s1: official 45.39%  macro 55.32%  Normal recall 0.72  2.5min
  H0_g4_cw_f2_s2: official 42.48%  macro 54.53%  Normal recall 0.63  2.5min
  H2_g2_nocw_f2_s0: official 47.00%  macro 52.80%  Normal recall 0.85  2.5min
  H2_g2_nocw_f2_s1: official 46.34%  macro 52.20%  Normal recall 0.86  2.5min
  H2_g2_nocw_f2_s2: official 46.59%  macro 52.23%  Normal recall 0.84  2.5min

fold  H0 off   H2 off    diff  H0 seedSD   H0 Se/Sp    H2

In [ ]:
# ============================================================
# FRESH SESSION: confirmation folds 3 and 4 + verdict + binary table
# self contained, resumable, skips runs already in gamma_ablation.json
# ============================================================
import os, json, time, random, gc, zipfile
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.metrics import confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

print("GPUs:", tf.config.list_physical_devices('GPU'))
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
PROJ='/content/drive/MyDrive/pulmoscan semester 7'; WORK='/content/work'
RES=os.path.join(PROJ,'results')
GP=os.path.join(RES,'gamma_protocol.json'); GL=os.path.join(RES,'gamma_ablation.json')
N_MELS=128; EPOCHS=40; BATCH=32; LR=1e-3; LR_MIN=1e-6; DIAG_W=0.1; NS=4; ND=7
SOUND=['Normal','Crackle','Wheeze','Both']
ARMS={'H0_g4_cw':(4.0,True),'H1_g2_cw':(2.0,True),'H2_g2_nocw':(2.0,False)}
CONF=['H0_g4_cw','H2_g2_nocw']
FOLDS_TO_RUN=[3,4]

# ---------- data ----------
os.makedirs(WORK,exist_ok=True)
if not os.path.isdir(os.path.join(WORK,'feat','processed','train')):
    with zipfile.ZipFile(os.path.join(PROJ,'data','pulmoscan_data.zip')) as z:
        z.extractall(os.path.join(WORK,'feat'))
with zipfile.ZipFile(os.path.join(PROJ,'data','colab_training_package.zip')) as z:
    z.extract('data/processed/manifest.csv', WORK)
man=pd.read_csv(os.path.join(WORK,'data/processed/manifest.csv'))
dev=man[man.split.isin(['train','val'])].copy().reset_index(drop=True)
dev['fp']=dev.apply(lambda r: os.path.join(WORK,'feat','processed',r['split'],
                    r['audio_file'].replace('.wav','.npy')),axis=1)
dev=dev[dev.fp.map(os.path.exists)].reset_index(drop=True)
assert 'test' not in set(dev.split)
RAW={i:np.load(dev.fp[i]) for i in range(len(dev))}
print(f"{len(dev)} rows, {dev.patient_id.nunique()} patients loaded")   # expect 3690 and 82

def zero_pad(i,T):
    a=RAW[i]; c=a.shape[1]
    return np.pad(a,[(0,0),(0,T-c)]) if c<T else a[:,:T]
def build_X(T):
    X=np.empty((len(dev),N_MELS,T,1),np.float32)
    for i in range(len(dev)): X[i,...,0]=zero_pad(i,T)
    return X
ys=dev.sound_label.values.astype(np.int32)
yd=dev.diagnosis_label.values.astype(np.int32)
pid=dev.patient_id.values

rng=np.random.RandomState(42)
pdf=dev.groupby('patient_id')['diagnosis_label'].agg(lambda x:x.mode()[0]).reset_index()
folds=[[] for _ in range(5)]
for _,g in pdf.groupby('diagnosis_label'):
    ps=g.patient_id.values.copy(); rng.shuffle(ps)
    for j,p in enumerate(ps): folds[j%5].append(p)

# ---------- metrics, loss, model ----------
def macro(yt,yp,n=4):
    cm=confusion_matrix(yt,yp,labels=list(range(n))); se=[];sp=[]
    for i in range(n):
        tp=cm[i,i];fn=cm[i,:].sum()-tp;fp=cm[:,i].sum()-tp;tn=cm.sum()-tp-fn-fp
        se.append(tp/(tp+fn) if tp+fn else 0.); sp.append(tn/(tn+fp) if tn+fp else 0.)
    return (np.mean(se)+np.mean(sp))/2
def official(yt,yp):
    cm=confusion_matrix(yt,yp,labels=[0,1,2,3])
    sp=cm[0,0]/cm[0].sum() if cm[0].sum() else 0.
    se=sum(cm[i,i] for i in [1,2,3])/sum(cm[i].sum() for i in [1,2,3])
    return (se+sp)/2,se,sp
def focal(g=2.0,cw=None):
    def fn(yt,yp):
        yp=tf.clip_by_value(yp,1e-7,1.0); ce=-yt*tf.math.log(yp)
        p=tf.reduce_sum(yt*yp,axis=-1,keepdims=True); f=tf.pow(1.0-p,g)*ce
        if cw is not None: f=f*tf.reduce_sum(yt*tf.constant(cw,tf.float32),axis=-1,keepdims=True)
        return tf.reduce_mean(f)
    fn.__name__='focal'; return fn
def build(T):
    inp=tf.keras.Input(shape=(N_MELS,T,1)); x=inp
    for f in [32,64,128]:
        x=tf.keras.layers.Conv2D(f,3,padding='same',activation='relu')(x)
        x=tf.keras.layers.BatchNormalization()(x); x=tf.keras.layers.MaxPooling2D(2)(x)
    x=tf.keras.layers.Conv2D(256,3,padding='same',activation='relu')(x)
    x=tf.keras.layers.BatchNormalization()(x)
    x=tf.keras.layers.GlobalAveragePooling2D()(x)
    sh=tf.keras.layers.Dropout(0.5)(tf.keras.layers.Dense(256,activation='relu')(x))
    s=tf.keras.layers.Dropout(0.3)(tf.keras.layers.Dense(128,activation='relu')(sh))
    so=tf.keras.layers.Dense(NS,activation='softmax',name='sound')(s)
    d=tf.keras.layers.Dropout(0.3)(tf.keras.layers.Dense(128,activation='relu')(sh))
    do=tf.keras.layers.Dense(ND,activation='softmax',name='diagnosis')(d)
    return tf.keras.Model(inp,[so,do])

# ---------- run ----------
XC={}
def run_g(arm,T,k,seed):
    tag=f'{arm}_f{k}_s{seed}'
    L=json.load(open(GL)) if os.path.exists(GL) else {}
    if tag in L: print(f'  SKIP {tag} ({L[tag]["official"]*100:.2f}%)'); return
    gamma,use_cw=ARMS[arm]
    if T not in XC:
        XC.clear(); print(f'  building X for {T}f zeropad...'); XC[T]=build_X(T)
    X=XC[T]
    te=np.isin(pid,folds[k]); tr=~te
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    tf.keras.backend.clear_session()
    cw=[float(c) for c in compute_class_weight('balanced',classes=np.unique(ys[tr]),y=ys[tr])] if use_cw else None
    m=build(T); m.compile(optimizer=tf.keras.optimizers.Adam(LR),
        loss={'sound':focal(gamma,cw),'diagnosis':focal(2.0)},
        loss_weights={'sound':1.0,'diagnosis':DIAG_W},metrics={'sound':'accuracy'})
    t0=time.time()
    m.fit(X[tr],{'sound':tf.keras.utils.to_categorical(ys[tr],NS),
                 'diagnosis':tf.keras.utils.to_categorical(yd[tr],ND)},
          batch_size=BATCH,epochs=EPOCHS,verbose=0,
          callbacks=[tf.keras.callbacks.LearningRateScheduler(
              lambda e: float(LR_MIN+0.5*(LR-LR_MIN)*(1+np.cos(np.pi*e/EPOCHS))))])
    yp=np.argmax(m.predict(X[te],verbose=0)[0],axis=1)
    mac=macro(ys[te],yp); off,se,sp=official(ys[te],yp)
    rec={n:float((yp[ys[te]==i]==i).mean()) for i,n in enumerate(SOUND)}
    print(f'  {tag}: official {off*100:.2f}%  macro {mac*100:.2f}%  Normal recall {rec["Normal"]:.2f}  {(time.time()-t0)/60:.1f}min')
    L=json.load(open(GL)) if os.path.exists(GL) else {}
    L[tag]={'arm':arm,'gamma':gamma,'class_weights':use_cw,'fold':k,'seed':seed,
            'official':float(off),'macro':float(mac),'se':float(se),'sp':float(sp),
            'per_class_recall':rec,
            'confusion':confusion_matrix(ys[te],yp,labels=[0,1,2,3]).tolist()}
    json.dump(L,open(GL,'w'),indent=2)
    del m; gc.collect()

for k in FOLDS_TO_RUN:
    for arm in CONF:
        for s in [0,1,2]:
            run_g(arm,192,k,s)

# ---------- summary and verdict (rules fixed in amendment_1) ----------
def summarize_confirm():
    L=json.load(open(GL)); A,B=CONF
    def runs(a,k): return [v for v in L.values() if v['arm']==a and v['fold']==k]
    print('\nfold  H0 off   H2 off    diff  H0 seedSD   H0 Se/Sp    H2 Se/Sp')
    diffs=[]; sds=[]; done=[]
    for k in range(5):
        ra,rb=runs(A,k),runs(B,k)
        if len(ra)<3 or len(rb)<3: continue
        oa=[v['official'] for v in ra]; ob=[v['official'] for v in rb]
        dd=(np.mean(ob)-np.mean(oa))*100
        diffs.append(dd); sds.append(np.std(oa)*100); done.append(k)
        sa=np.mean([v['se'] for v in ra])*100; pa=np.mean([v['sp'] for v in ra])*100
        sb=np.mean([v['se'] for v in rb])*100; pb=np.mean([v['sp'] for v in rb])*100
        print(f"  {k}   {np.mean(oa)*100:6.2f}  {np.mean(ob)*100:6.2f}  {dd:+6.2f}   {np.std(oa)*100:5.2f}     {sa:4.1f}/{pa:4.1f}   {sb:4.1f}/{pb:4.1f}")
    def rec(a,c): return np.nanmean([v['per_class_recall'][c] for k in done for v in runs(a,k)])
    print('\nmean recall over folds done:', done)
    for c in SOUND: print(f'  {c:8s} H0 {rec(A,c):.2f}   H2 {rec(B,c):.2f}')
    if len(done)==5:
        r1=sum(x>0 for x in diffs)>=4
        r2=np.mean(diffs)>np.mean(sds)
        r3=all(rec(B,c)>=0.5*rec(A,c) for c in ['Wheeze','Both'])
        print(f'\nRULE 1 wins>=4 of 5: {r1}  ({sum(x>0 for x in diffs)}/5)')
        print(f'RULE 2 mean diff {np.mean(diffs):+.2f} > seed SD {np.mean(sds):.2f}: {r2}')
        print(f'RULE 3 Wheeze and Both recall kept: {r3}')
        print('VERDICT:', 'ADOPT H2' if (r1 and r2 and r3) else 'KEEP H0, report H2 as operating point shift')
    else:
        print('\nnot all 5 folds done yet, no verdict')
summarize_confirm()

# ---------- binary screening view (Normal vs abnormal) ----------
def binary(cm):
    cm=np.array(cm)
    return cm[1:,1:].sum()/cm[1:].sum(), cm[0,0]/cm[0].sum()
L=json.load(open(GL))
print("\nall-Normal floor: official 50.0 (Se 0, Sp 100)\n")
print("fold arm         official  4class Se/Sp    BINARY Se/Sp    binary mean")
for k in range(5):
    for arm in CONF:
        r=[v for v in L.values() if v['arm']==arm and v['fold']==k]
        if len(r)<3: continue
        o=np.mean([v['official'] for v in r])*100
        s4=np.mean([v['se'] for v in r])*100; p4=np.mean([v['sp'] for v in r])*100
        b=[binary(v['confusion']) for v in r]
        bs=np.mean([x[0] for x in b])*100; bp=np.mean([x[1] for x in b])*100
        print(f"  {k}  {arm:10s} {o:6.2f}   {s4:5.1f}/{p4:5.1f}     {bs:5.1f}/{bp:5.1f}     {(bs+bp)/2:5.1f}")

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
3690 rows, 82 patients loaded
  SKIP H0_g4_cw_f3_s0 (53.75%)
  SKIP H0_g4_cw_f3_s1 (57.87%)
  SKIP H0_g4_cw_f3_s2 (56.31%)
  SKIP H2_g2_nocw_f3_s0 (57.59%)
  SKIP H2_g2_nocw_f3_s1 (58.15%)
  SKIP H2_g2_nocw_f3_s2 (56.95%)
  SKIP H0_g4_cw_f4_s0 (50.74%)
  SKIP H0_g4_cw_f4_s1 (45.18%)
  SKIP H0_g4_cw_f4_s2 (49.67%)
  SKIP H2_g2_nocw_f4_s0 (59.00%)
  SKIP H2_g2_nocw_f4_s1 (59.63%)
  SKIP H2_g2_nocw_f4_s2 (59.78%)

fold  H0 off   H2 off    diff  H0 seedSD   H0 Se/Sp    H2 Se/Sp
  0    45.83   53.82   +7.99    2.04     46.7/45.0   38.6/69.0
  1    50.04   55.15   +5.11    0.89     57.5/42.6   34.3/76.0
  2    43.39   46.64   +3.25    1.42     21.5/65.3    8.3/85.0
  3    55.98   57.56   +1.59    1.70     53.7/58.2   19.0/96.1
  4    48.53   59.47  +10.95    2.41     52.4/44.7   32.6/86.4

mean recall over folds done: [0, 1, 2, 3, 4]
  Normal   H0 0.51   H2 0.83
  Crackle  H0 0.55   H2 0.30
  Wheeze   H0 0.48   H2 0.27


In [ ]:
# ============================================================
# OOF PROBABILITIES + THRESHOLD FREE CHECK (H0 vs H2, seed 0, 5 folds)
# self contained, resumable
# ============================================================
import os, json, time, random, gc, zipfile, datetime
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight

print("GPUs:", tf.config.list_physical_devices('GPU'))
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
PROJ='/content/drive/MyDrive/pulmoscan semester 7'; WORK='/content/work'
RES=os.path.join(PROJ,'results')
GP=os.path.join(RES,'gamma_protocol.json'); GL=os.path.join(RES,'gamma_ablation.json')
OOF=os.path.join(RES,'oof_probs'); os.makedirs(OOF,exist_ok=True)
N_MELS=128; EPOCHS=40; BATCH=32; LR=1e-3; LR_MIN=1e-6; DIAG_W=0.1; NS=4; ND=7
SOUND=['Normal','Crackle','Wheeze','Both']
ARMS={'H0_g4_cw':(4.0,True),'H2_g2_nocw':(2.0,False)}
ARMS2=list(ARMS)

# ---------- data ----------
os.makedirs(WORK,exist_ok=True)
if not os.path.isdir(os.path.join(WORK,'feat','processed','train')):
    with zipfile.ZipFile(os.path.join(PROJ,'data','pulmoscan_data.zip')) as z:
        z.extractall(os.path.join(WORK,'feat'))
with zipfile.ZipFile(os.path.join(PROJ,'data','colab_training_package.zip')) as z:
    z.extract('data/processed/manifest.csv', WORK)
man=pd.read_csv(os.path.join(WORK,'data/processed/manifest.csv'))
dev=man[man.split.isin(['train','val'])].copy().reset_index(drop=True)
dev['fp']=dev.apply(lambda r: os.path.join(WORK,'feat','processed',r['split'],
                    r['audio_file'].replace('.wav','.npy')),axis=1)
dev=dev[dev.fp.map(os.path.exists)].reset_index(drop=True)
assert 'test' not in set(dev.split)
RAW={i:np.load(dev.fp[i]) for i in range(len(dev))}
print(f"{len(dev)} rows, {dev.patient_id.nunique()} patients loaded")   # expect 3690 and 82

def zero_pad(i,T):
    a=RAW[i]; c=a.shape[1]
    return np.pad(a,[(0,0),(0,T-c)]) if c<T else a[:,:T]
def build_X(T):
    X=np.empty((len(dev),N_MELS,T,1),np.float32)
    for i in range(len(dev)): X[i,...,0]=zero_pad(i,T)
    return X
ys=dev.sound_label.values.astype(np.int32)
yd=dev.diagnosis_label.values.astype(np.int32)
pid=dev.patient_id.values

rng=np.random.RandomState(42)
pdf=dev.groupby('patient_id')['diagnosis_label'].agg(lambda x:x.mode()[0]).reset_index()
folds=[[] for _ in range(5)]
for _,g in pdf.groupby('diagnosis_label'):
    ps=g.patient_id.values.copy(); rng.shuffle(ps)
    for j,p in enumerate(ps): folds[j%5].append(p)

# ---------- metrics, loss, model ----------
def official(yt,yp):
    cm=confusion_matrix(yt,yp,labels=[0,1,2,3])
    sp=cm[0,0]/cm[0].sum() if cm[0].sum() else 0.
    se=sum(cm[i,i] for i in [1,2,3])/sum(cm[i].sum() for i in [1,2,3])
    return (se+sp)/2,se,sp
def binary(cm):
    cm=np.array(cm)
    return cm[1:,1:].sum()/cm[1:].sum(), cm[0,0]/cm[0].sum()
def focal(g=2.0,cw=None):
    def fn(yt,yp):
        yp=tf.clip_by_value(yp,1e-7,1.0); ce=-yt*tf.math.log(yp)
        p=tf.reduce_sum(yt*yp,axis=-1,keepdims=True); f=tf.pow(1.0-p,g)*ce
        if cw is not None: f=f*tf.reduce_sum(yt*tf.constant(cw,tf.float32),axis=-1,keepdims=True)
        return tf.reduce_mean(f)
    fn.__name__='focal'; return fn
def build(T):
    inp=tf.keras.Input(shape=(N_MELS,T,1)); x=inp
    for f in [32,64,128]:
        x=tf.keras.layers.Conv2D(f,3,padding='same',activation='relu')(x)
        x=tf.keras.layers.BatchNormalization()(x); x=tf.keras.layers.MaxPooling2D(2)(x)
    x=tf.keras.layers.Conv2D(256,3,padding='same',activation='relu')(x)
    x=tf.keras.layers.BatchNormalization()(x)
    x=tf.keras.layers.GlobalAveragePooling2D()(x)
    sh=tf.keras.layers.Dropout(0.5)(tf.keras.layers.Dense(256,activation='relu')(x))
    s=tf.keras.layers.Dropout(0.3)(tf.keras.layers.Dense(128,activation='relu')(sh))
    so=tf.keras.layers.Dense(NS,activation='softmax',name='sound')(s)
    d=tf.keras.layers.Dropout(0.3)(tf.keras.layers.Dense(128,activation='relu')(sh))
    do=tf.keras.layers.Dense(ND,activation='softmax',name='diagnosis')(d)
    return tf.keras.Model(inp,[so,do])

# ---------- record the verdict BEFORE the new analysis ----------
L=json.load(open(GL))
def fold_mean(arm,fn):
    return np.mean([np.mean([fn(v) for v in L.values() if v['arm']==arm and v['fold']==k]) for k in range(5)])*100
res={a:{'official':fold_mean(a,lambda v:v['official']),
        'macro':fold_mean(a,lambda v:v['macro']),
        'binary_se':fold_mean(a,lambda v:binary(v['confusion'])[0]),
        'binary_sp':fold_mean(a,lambda v:binary(v['confusion'])[1])} for a in ARMS2}
d=json.load(open(GP))
if 'confirmation_result' not in d:
    d['confirmation_result']={
      'written_at':datetime.datetime.now().isoformat(),
      'verdict_by_prewritten_rules':'ADOPT H2 (5/5 folds, mean diff +5.78 > seed SD 1.69, rule 3 passed)',
      'means_over_5_folds_3_seeds':res,
      'caveat':'H2 gain is an operating point shift: binary Se falls sharply, binary mean and macro do not improve; rule 3 was too weak to catch this',
      'next':'threshold free check (binary AUC, Normal bias sweep) on saved out of fold probabilities'}
    json.dump(d,open(GP,'w'),indent=2)
print({a:{k:round(v,1) for k,v in r.items()} for a,r in res.items()})

# ---------- runs that save probabilities ----------
XC={}
def run_p(arm,T,k,seed):
    path=os.path.join(OOF,f'{arm}_f{k}_s{seed}.npz')
    if os.path.exists(path): print('  SKIP',os.path.basename(path)); return
    gamma,use_cw=ARMS[arm]
    if T not in XC:
        XC.clear(); print(f'  building X for {T}f zeropad...'); XC[T]=build_X(T)
    X=XC[T]
    te=np.isin(pid,folds[k]); tr=~te
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    tf.keras.backend.clear_session()
    cw=[float(c) for c in compute_class_weight('balanced',classes=np.unique(ys[tr]),y=ys[tr])] if use_cw else None
    m=build(T); m.compile(optimizer=tf.keras.optimizers.Adam(LR),
        loss={'sound':focal(gamma,cw),'diagnosis':focal(2.0)},
        loss_weights={'sound':1.0,'diagnosis':DIAG_W},metrics={'sound':'accuracy'})
    t0=time.time()
    m.fit(X[tr],{'sound':tf.keras.utils.to_categorical(ys[tr],NS),
                 'diagnosis':tf.keras.utils.to_categorical(yd[tr],ND)},
          batch_size=BATCH,epochs=EPOCHS,verbose=0,
          callbacks=[tf.keras.callbacks.LearningRateScheduler(
              lambda e: float(LR_MIN+0.5*(LR-LR_MIN)*(1+np.cos(np.pi*e/EPOCHS))))])
    pr=m.predict(X[te],verbose=0)[0]
    off,se,sp=official(ys[te],np.argmax(pr,axis=1))
    print(f'  {arm}_f{k}_s{seed}: official {off*100:.2f}%  Se {se*100:.1f} Sp {sp*100:.1f}  {(time.time()-t0)/60:.1f}min')
    np.savez_compressed(path,p=pr.astype(np.float32),y=ys[te],pid=pid[te])
    del m; gc.collect()

for k in range(5):
    for arm in ARMS2:
        run_p(arm,192,k,0)

# ---------- threshold free analysis ----------
Dd={a:[np.load(os.path.join(OOF,f'{a}_f{k}_s0.npz')) for k in range(5)] for a in ARMS2}
print('\nBINARY AUC (Normal vs abnormal, threshold free), seed 0')
print('fold    H0     H2')
auc={a:[] for a in ARMS2}
for k in range(5):
    row=[]
    for a in ARMS2:
        z=Dd[a][k]; v=roc_auc_score(z['y']>0, 1-z['p'][:,0]); auc[a].append(v); row.append(v)
    print(f'  {k}   {row[0]:.3f}  {row[1]:.3f}')
print(f'mean  {np.mean(auc[ARMS2[0]]):.3f}  {np.mean(auc[ARMS2[1]]):.3f}')

shifts=np.arange(-3,3.01,0.25)
curve={a:[] for a in ARMS2}
for a in ARMS2:
    for s in shifts:
        w=np.array([np.exp(s),1,1,1])
        vals=[official(Dd[a][k]['y'],np.argmax(Dd[a][k]['p']*w,axis=1))[0] for k in range(5)]
        curve[a].append(np.mean(vals)*100)
print('\nOFFICIAL vs Normal bias shift (mean of 5 folds); s>0 favours Normal')
print('  shift     H0      H2')
for i,s in enumerate(shifts): print(f'  {s:+.2f}  {curve[ARMS2[0]][i]:6.2f}  {curve[ARMS2[1]][i]:6.2f}')
i0=int(np.argmin(np.abs(shifts)))
for a in ARMS2:
    i=int(np.argmax(curve[a]))
    print(f'{a}: as trained {curve[a][i0]:.2f} | best shift {shifts[i]:+.2f} -> {curve[a][i]:.2f}  (ceiling; picked using held out labels, never report as a result)')

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
3690 rows, 82 patients loaded
{'H0_g4_cw': {'official': np.float64(48.8), 'macro': np.float64(61.4), 'binary_se': np.float64(72.8), 'binary_sp': np.float64(51.1)}, 'H2_g2_nocw': {'official': np.float64(54.5), 'macro': np.float64(58.9), 'binary_se': np.float64(43.6), 'binary_sp': np.float64(82.5)}}
  building X for 192f zeropad...
  H0_g4_cw_f0_s0: official 44.57%  Se 42.9 Sp 46.3  2.2min
  H2_g2_nocw_f0_s0: official 52.34%  Se 40.3 Sp 64.4  2.1min
  H0_g4_cw_f1_s0: official 48.40%  Se 60.8 Sp 36.0  2.6min
  H2_g2_nocw_f1_s0: official 53.21%  Se 32.4 Sp 74.1  2.5min
  H0_g4_cw_f2_s0: official 44.05%  Se 28.0 Sp 60.1  2.5min
  H2_g2_nocw_f2_s0: official 47.70%  Se 10.3 Sp 85.1  2.5min
  H0_g4_cw_f3_s0: official 57.31%  Se 54.9 Sp 59.7  2.4min
  H2_g2_nocw_f3_s0: official 57.16%  Se 20.0 Sp 94.3  2.5min
  H0_g4_cw_f4_s0: official 44.31%  Se 54.5 Sp 34.1  2.4min
  H2_g2_nocw_f4_s0: official 59.40%  Se 27.5 Sp 91.3  2.

In [ ]:
# ============================================================
# AUGMENTATION ARM (A1: SpecAugment masks + level jitter) vs H2 baseline
# 5 folds x 3 seeds, same folds and seeds as H2; self contained, resumable
# ============================================================
import os, json, time, random, gc, zipfile, datetime
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.metrics import confusion_matrix, roc_auc_score

print("GPUs:", tf.config.list_physical_devices('GPU'))
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
PROJ='/content/drive/MyDrive/pulmoscan semester 7'; WORK='/content/work'
RES=os.path.join(PROJ,'results'); OOF=os.path.join(RES,'oof_probs'); os.makedirs(OOF,exist_ok=True)
GL=os.path.join(RES,'gamma_ablation.json')
AL=os.path.join(RES,'aug_ablation.json'); AP=os.path.join(RES,'aug_protocol.json')
N_MELS=128; EPOCHS=40; BATCH=32; LR=1e-3; LR_MIN=1e-6; DIAG_W=0.1; NS=4; ND=7
SOUND=['Normal','Crackle','Wheeze','Both']; BASE='H2_g2_nocw'; ARM='A1_specaug'

# ---------- protocol, written before any run ----------
if not os.path.exists(AP):
    json.dump({
      'written_at':datetime.datetime.now().isoformat(),
      'question':'does light SpecAugment style augmentation improve Normal vs abnormal separation and official ICBHI on the H2 baseline',
      'baseline':'H2_g2_nocw from gamma_ablation.json (192f zero pad, focal gamma 2, no class weights, 40 epochs, final weights)',
      'arm':'A1_specaug: same as baseline plus per sample 1 frequency mask (up to 20 bins), 1 time mask (up to 30 frames), level offset N(0,0.15) applied to non padded values only',
      'design':'5 patient grouped folds x 3 seeds, same folds and seeds as baseline',
      'adopt_only_if_all':[
        'mean official gain over H2 greater than 1.7 points (the measured H2 seed SD)',
        'official better on at least 4 of 5 folds',
        'mean binary Se not lower than H2 by more than 3 points',
        'seed 0 binary AUC not lower than H2 on at least 4 of 5 folds'],
      'otherwise':'report as no evidence of benefit',
      'commitment':'no rule changes after results; test set untouched'}, open(AP,'w'), indent=2)
    print('protocol written')

# ---------- data ----------
os.makedirs(WORK,exist_ok=True)
if not os.path.isdir(os.path.join(WORK,'feat','processed','train')):
    with zipfile.ZipFile(os.path.join(PROJ,'data','pulmoscan_data.zip')) as z:
        z.extractall(os.path.join(WORK,'feat'))
with zipfile.ZipFile(os.path.join(PROJ,'data','colab_training_package.zip')) as z:
    z.extract('data/processed/manifest.csv', WORK)
man=pd.read_csv(os.path.join(WORK,'data/processed/manifest.csv'))
dev=man[man.split.isin(['train','val'])].copy().reset_index(drop=True)
dev['fp']=dev.apply(lambda r: os.path.join(WORK,'feat','processed',r['split'],
                    r['audio_file'].replace('.wav','.npy')),axis=1)
dev=dev[dev.fp.map(os.path.exists)].reset_index(drop=True)
assert 'test' not in set(dev.split)
RAW={i:np.load(dev.fp[i]) for i in range(len(dev))}
print(f"{len(dev)} rows, {dev.patient_id.nunique()} patients loaded")   # expect 3690 and 82

def zero_pad(i,T):
    a=RAW[i]; c=a.shape[1]
    return np.pad(a,[(0,0),(0,T-c)]) if c<T else a[:,:T]
def build_X(T):
    X=np.empty((len(dev),N_MELS,T,1),np.float32)
    for i in range(len(dev)): X[i,...,0]=zero_pad(i,T)
    return X
ys=dev.sound_label.values.astype(np.int32)
yd=dev.diagnosis_label.values.astype(np.int32)
pid=dev.patient_id.values

rng=np.random.RandomState(42)
pdf=dev.groupby('patient_id')['diagnosis_label'].agg(lambda x:x.mode()[0]).reset_index()
folds=[[] for _ in range(5)]
for _,g in pdf.groupby('diagnosis_label'):
    ps=g.patient_id.values.copy(); rng.shuffle(ps)
    for j,p in enumerate(ps): folds[j%5].append(p)

# ---------- metrics, loss, model ----------
def official(yt,yp):
    cm=confusion_matrix(yt,yp,labels=[0,1,2,3])
    sp=cm[0,0]/cm[0].sum() if cm[0].sum() else 0.
    se=sum(cm[i,i] for i in [1,2,3])/sum(cm[i].sum() for i in [1,2,3])
    return (se+sp)/2,se,sp
def binary(cm):
    cm=np.array(cm)
    return cm[1:,1:].sum()/cm[1:].sum(), cm[0,0]/cm[0].sum()
def focal(g=2.0):
    def fn(yt,yp):
        yp=tf.clip_by_value(yp,1e-7,1.0); ce=-yt*tf.math.log(yp)
        p=tf.reduce_sum(yt*yp,axis=-1,keepdims=True)
        return tf.reduce_mean(tf.pow(1.0-p,g)*ce)
    fn.__name__='focal'; return fn
def build(T):
    inp=tf.keras.Input(shape=(N_MELS,T,1)); x=inp
    for f in [32,64,128]:
        x=tf.keras.layers.Conv2D(f,3,padding='same',activation='relu')(x)
        x=tf.keras.layers.BatchNormalization()(x); x=tf.keras.layers.MaxPooling2D(2)(x)
    x=tf.keras.layers.Conv2D(256,3,padding='same',activation='relu')(x)
    x=tf.keras.layers.BatchNormalization()(x)
    x=tf.keras.layers.GlobalAveragePooling2D()(x)
    sh=tf.keras.layers.Dropout(0.5)(tf.keras.layers.Dense(256,activation='relu')(x))
    s=tf.keras.layers.Dropout(0.3)(tf.keras.layers.Dense(128,activation='relu')(sh))
    so=tf.keras.layers.Dense(NS,activation='softmax',name='sound')(s)
    d=tf.keras.layers.Dropout(0.3)(tf.keras.layers.Dense(128,activation='relu')(sh))
    do=tf.keras.layers.Dense(ND,activation='softmax',name='diagnosis')(d)
    return tf.keras.Model(inp,[so,do])

# ---------- augmentation (training batches only) ----------
def augment(x,y):
    B=tf.shape(x)[0]; F=tf.shape(x)[1]; T=tf.shape(x)[2]
    real=tf.cast(tf.not_equal(x,0.0),tf.float32)
    x=x+tf.random.normal([B,1,1,1],0.0,0.15)*real            # level jitter, padding untouched
    fw=tf.random.uniform([B,1,1,1],0,21,dtype=tf.int32)
    f0=tf.random.uniform([B,1,1,1],0,1000000,dtype=tf.int32)%tf.maximum(F-fw,1)
    fi=tf.reshape(tf.range(F),[1,-1,1,1])
    fmask=tf.logical_and(fi>=f0,fi<f0+fw)
    tw=tf.random.uniform([B,1,1,1],0,31,dtype=tf.int32)
    t0=tf.random.uniform([B,1,1,1],0,1000000,dtype=tf.int32)%tf.maximum(T-tw,1)
    ti=tf.reshape(tf.range(T),[1,1,-1,1])
    tmask=tf.logical_and(ti>=t0,ti<t0+tw)
    x=tf.where(tf.logical_or(fmask,tmask),tf.zeros_like(x),x)  # masked area set to the feature mean (0)
    return x,y

# ---------- run ----------
XC={}
def run_a(k,seed,T=192):
    tag=f'{ARM}_f{k}_s{seed}'
    L=json.load(open(AL)) if os.path.exists(AL) else {}
    if tag in L: print(f'  SKIP {tag} ({L[tag]["official"]*100:.2f}%)'); return
    if T not in XC:
        XC.clear(); print(f'  building X for {T}f zeropad...'); XC[T]=build_X(T)
    X=XC[T]
    te=np.isin(pid,folds[k]); tr=~te
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    tf.keras.backend.clear_session()
    ds=tf.data.Dataset.from_tensor_slices((X[tr],{
        'sound':tf.keras.utils.to_categorical(ys[tr],NS),
        'diagnosis':tf.keras.utils.to_categorical(yd[tr],ND)}))
    ds=ds.shuffle(4096,seed=seed).batch(BATCH).map(augment,num_parallel_calls=tf.data.AUTOTUNE).prefetch(2)
    m=build(T); m.compile(optimizer=tf.keras.optimizers.Adam(LR),
        loss={'sound':focal(2.0),'diagnosis':focal(2.0)},
        loss_weights={'sound':1.0,'diagnosis':DIAG_W},metrics={'sound':'accuracy'})
    t0=time.time()
    m.fit(ds,epochs=EPOCHS,verbose=0,callbacks=[tf.keras.callbacks.LearningRateScheduler(
        lambda e: float(LR_MIN+0.5*(LR-LR_MIN)*(1+np.cos(np.pi*e/EPOCHS))))])
    pr=m.predict(X[te],verbose=0)[0]; yp=np.argmax(pr,axis=1)
    off,se,sp=official(ys[te],yp)
    print(f'  {tag}: official {off*100:.2f}%  Se {se*100:.1f} Sp {sp*100:.1f}  {(time.time()-t0)/60:.1f}min')
    np.savez_compressed(os.path.join(OOF,f'A1_f{k}_s{seed}.npz'),p=pr.astype(np.float32),y=ys[te],pid=pid[te])
    L=json.load(open(AL)) if os.path.exists(AL) else {}
    L[tag]={'arm':ARM,'fold':k,'seed':seed,'official':float(off),'se':float(se),'sp':float(sp),
            'per_class_recall':{n:float((yp[ys[te]==i]==i).mean()) for i,n in enumerate(SOUND)},
            'confusion':confusion_matrix(ys[te],yp,labels=[0,1,2,3]).tolist()}
    json.dump(L,open(AL,'w'),indent=2)
    del m; gc.collect()

for k in range(5):
    for s in [0,1,2]:
        run_a(k,s)

# ---------- summary and verdict ----------
A=json.load(open(AL)); B=json.load(open(GL))
def fr(Ld,arm,k): return [v for v in Ld.values() if v['arm']==arm and v['fold']==k]
print('\nfold  H2 off   A1 off    diff  | H2 binSe  A1 binSe | AUC H2  AUC A1 (seed 0)')
diffs=[]; se_h=[]; se_a=[]; auc_ok=[]; done=[]
for k in range(5):
    rb,ra=fr(B,BASE,k),fr(A,ARM,k)
    if len(rb)<3 or len(ra)<3: continue
    ob=np.mean([v['official'] for v in rb])*100; oa=np.mean([v['official'] for v in ra])*100
    sb=np.mean([binary(v['confusion'])[0] for v in rb])*100; sa=np.mean([binary(v['confusion'])[0] for v in ra])*100
    zb=np.load(os.path.join(OOF,f'{BASE}_f{k}_s0.npz')); za=np.load(os.path.join(OOF,f'A1_f{k}_s0.npz'))
    ab=roc_auc_score(zb['y']>0,1-zb['p'][:,0]); aa=roc_auc_score(za['y']>0,1-za['p'][:,0])
    diffs.append(oa-ob); se_h.append(sb); se_a.append(sa); auc_ok.append(aa>=ab); done.append(k)
    print(f'  {k}   {ob:6.2f}  {oa:6.2f}  {oa-ob:+6.2f}  |  {sb:5.1f}    {sa:5.1f}   |  {ab:.3f}   {aa:.3f}')
if len(done)==5:
    r1=np.mean(diffs)>1.7; r2=sum(x>0 for x in diffs)>=4
    r3=np.mean(se_a)>=np.mean(se_h)-3; r4=sum(auc_ok)>=4
    print(f'\nRULE 1 mean gain {np.mean(diffs):+.2f} > 1.7: {r1}')
    print(f'RULE 2 better on >=4 of 5 folds: {r2} ({sum(x>0 for x in diffs)}/5)')
    print(f'RULE 3 binary Se {np.mean(se_a):.1f} vs {np.mean(se_h):.1f}, drop at most 3: {r3}')
    print(f'RULE 4 AUC not lower on >=4 of 5 folds: {r4} ({sum(auc_ok)}/5)')
    print('VERDICT:', 'AUGMENTATION HELPS' if (r1 and r2 and r3 and r4) else 'NO EVIDENCE OF BENEFIT, report as negative')
else:
    print('\nnot all folds done, no verdict')

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Mounted at /content/drive
protocol written
3690 rows, 82 patients loaded
  building X for 192f zeropad...
  A1_specaug_f0_s0: official 53.45%  Se 29.8 Sp 77.1  2.2min
  A1_specaug_f0_s1: official 55.91%  Se 38.4 Sp 73.4  2.3min
  A1_specaug_f0_s2: official 52.10%  Se 30.8 Sp 73.4  2.3min
  A1_specaug_f1_s0: official 58.87%  Se 36.8 Sp 81.0  2.6min
  A1_specaug_f1_s1: official 58.37%  Se 34.3 Sp 82.4  2.6min
  A1_specaug_f1_s2: official 58.14%  Se 35.3 Sp 81.0  2.6min
  A1_specaug_f2_s0: official 49.08%  Se 10.3 Sp 87.9  2.5min
  A1_specaug_f2_s1: official 51.67%  Se 12.1 Sp 91.2  2.7min
  A1_specaug_f2_s2: official 51.35%  Se 12.6 Sp 90.1  2.6min
  A1_specaug_f3_s0: official 58.67%  Se 23.3 Sp 94.1  2.6min
  A1_specaug_f3_s1: official 57.00%  Se 17.6 Sp 96.4  2.5min
  A1_specaug_f3_s2: official 55.84%  Se 15.8 Sp 95.9  2.5min
  A1_specaug_f4_s0: official 51.96%  Se 18.4 Sp 85.5  2.5min
  A1_specaug_f4_s1: official

In [ ]:
# ============================================================
# CLOSE OUT: H2 seeds 1,2 with saved probabilities, then threshold free comparison
# self contained, resumable
# ============================================================
import os, json, time, random, gc, zipfile, datetime
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve

print("GPUs:", tf.config.list_physical_devices('GPU'))
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
PROJ='/content/drive/MyDrive/pulmoscan semester 7'; WORK='/content/work'
RES=os.path.join(PROJ,'results'); OOF=os.path.join(RES,'oof_probs'); AP=os.path.join(RES,'aug_protocol.json')
N_MELS=128; EPOCHS=40; BATCH=32; LR=1e-3; LR_MIN=1e-6; DIAG_W=0.1; NS=4; ND=7
H='H2_g2_nocw'; A='A1'

# ---------- amendment, written before any new run ----------
d=json.load(open(AP))
if 'amendment_1' not in d:
    d['amendment_1']={
      'written_at':datetime.datetime.now().isoformat(),
      'status':'exploratory, added after seeing seed 0 AUC; the primary verdict stays NO EVIDENCE OF BENEFIT (mean official gain +0.31, better on 3 of 5 folds)',
      'primary_result':{'mean_official_gain':0.31,'folds_better':'3/5','rule1':False,'rule2':False,'rule3':True,'rule4':True},
      'added_analysis':'rerun H2 seeds 1 and 2 saving probabilities; 3 seed mean binary AUC per fold; pooled out of fold Se at Sp 0.70 and Sp at Se 0.70',
      'freeze_rule':'freeze A1 only if 3 seed mean AUC is higher on at least 4 of 5 folds and mean gain exceeds 0.02; otherwise freeze H2',
      'commitment':'official results for both arms reported regardless; test set untouched'}
    json.dump(d,open(AP,'w'),indent=2); print('amendment written')

# ---------- data ----------
os.makedirs(WORK,exist_ok=True)
if not os.path.isdir(os.path.join(WORK,'feat','processed','train')):
    with zipfile.ZipFile(os.path.join(PROJ,'data','pulmoscan_data.zip')) as z:
        z.extractall(os.path.join(WORK,'feat'))
with zipfile.ZipFile(os.path.join(PROJ,'data','colab_training_package.zip')) as z:
    z.extract('data/processed/manifest.csv', WORK)
man=pd.read_csv(os.path.join(WORK,'data/processed/manifest.csv'))
dev=man[man.split.isin(['train','val'])].copy().reset_index(drop=True)
dev['fp']=dev.apply(lambda r: os.path.join(WORK,'feat','processed',r['split'],
                    r['audio_file'].replace('.wav','.npy')),axis=1)
dev=dev[dev.fp.map(os.path.exists)].reset_index(drop=True)
assert 'test' not in set(dev.split)
RAW={i:np.load(dev.fp[i]) for i in range(len(dev))}
print(f"{len(dev)} rows, {dev.patient_id.nunique()} patients loaded")   # expect 3690 and 82

def zero_pad(i,T):
    a=RAW[i]; c=a.shape[1]
    return np.pad(a,[(0,0),(0,T-c)]) if c<T else a[:,:T]
def build_X(T):
    X=np.empty((len(dev),N_MELS,T,1),np.float32)
    for i in range(len(dev)): X[i,...,0]=zero_pad(i,T)
    return X
ys=dev.sound_label.values.astype(np.int32)
yd=dev.diagnosis_label.values.astype(np.int32)
pid=dev.patient_id.values

rng=np.random.RandomState(42)
pdf=dev.groupby('patient_id')['diagnosis_label'].agg(lambda x:x.mode()[0]).reset_index()
folds=[[] for _ in range(5)]
for _,g in pdf.groupby('diagnosis_label'):
    ps=g.patient_id.values.copy(); rng.shuffle(ps)
    for j,p in enumerate(ps): folds[j%5].append(p)

def official(yt,yp):
    cm=confusion_matrix(yt,yp,labels=[0,1,2,3])
    sp=cm[0,0]/cm[0].sum() if cm[0].sum() else 0.
    se=sum(cm[i,i] for i in [1,2,3])/sum(cm[i].sum() for i in [1,2,3])
    return (se+sp)/2,se,sp
def focal(g=2.0):
    def fn(yt,yp):
        yp=tf.clip_by_value(yp,1e-7,1.0); ce=-yt*tf.math.log(yp)
        p=tf.reduce_sum(yt*yp,axis=-1,keepdims=True)
        return tf.reduce_mean(tf.pow(1.0-p,g)*ce)
    fn.__name__='focal'; return fn
def build(T):
    inp=tf.keras.Input(shape=(N_MELS,T,1)); x=inp
    for f in [32,64,128]:
        x=tf.keras.layers.Conv2D(f,3,padding='same',activation='relu')(x)
        x=tf.keras.layers.BatchNormalization()(x); x=tf.keras.layers.MaxPooling2D(2)(x)
    x=tf.keras.layers.Conv2D(256,3,padding='same',activation='relu')(x)
    x=tf.keras.layers.BatchNormalization()(x)
    x=tf.keras.layers.GlobalAveragePooling2D()(x)
    sh=tf.keras.layers.Dropout(0.5)(tf.keras.layers.Dense(256,activation='relu')(x))
    s=tf.keras.layers.Dropout(0.3)(tf.keras.layers.Dense(128,activation='relu')(sh))
    so=tf.keras.layers.Dense(NS,activation='softmax',name='sound')(s)
    d=tf.keras.layers.Dropout(0.3)(tf.keras.layers.Dense(128,activation='relu')(sh))
    do=tf.keras.layers.Dense(ND,activation='softmax',name='diagnosis')(d)
    return tf.keras.Model(inp,[so,do])

# ---------- H2 reruns that save probabilities ----------
XC={}
def run_h2(k,seed,T=192):
    path=os.path.join(OOF,f'{H}_f{k}_s{seed}.npz')
    if os.path.exists(path): print('  SKIP',os.path.basename(path)); return
    if T not in XC:
        XC.clear(); print(f'  building X for {T}f zeropad...'); XC[T]=build_X(T)
    X=XC[T]
    te=np.isin(pid,folds[k]); tr=~te
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    tf.keras.backend.clear_session()
    m=build(T); m.compile(optimizer=tf.keras.optimizers.Adam(LR),
        loss={'sound':focal(2.0),'diagnosis':focal(2.0)},
        loss_weights={'sound':1.0,'diagnosis':DIAG_W},metrics={'sound':'accuracy'})
    t0=time.time()
    m.fit(X[tr],{'sound':tf.keras.utils.to_categorical(ys[tr],NS),
                 'diagnosis':tf.keras.utils.to_categorical(yd[tr],ND)},
          batch_size=BATCH,epochs=EPOCHS,verbose=0,
          callbacks=[tf.keras.callbacks.LearningRateScheduler(
              lambda e: float(LR_MIN+0.5*(LR-LR_MIN)*(1+np.cos(np.pi*e/EPOCHS))))])
    pr=m.predict(X[te],verbose=0)[0]
    off,se,sp=official(ys[te],np.argmax(pr,axis=1))
    print(f'  {H}_f{k}_s{seed}: official {off*100:.2f}%  Se {se*100:.1f} Sp {sp*100:.1f}  {(time.time()-t0)/60:.1f}min')
    np.savez_compressed(path,p=pr.astype(np.float32),y=ys[te],pid=pid[te])
    del m; gc.collect()

for k in range(5):
    for s in [1,2]:
        run_h2(k,s)

# ---------- threshold free comparison ----------
def load(prefix,k,s): return np.load(os.path.join(OOF,f'{prefix}_f{k}_s{s}.npz'))
def auc_of(z): return roc_auc_score(z['y']>0,1-z['p'][:,0])
print('\nBINARY AUC per fold, mean of 3 seeds (SD across seeds)')
print('fold   H2 mean (sd)     A1 mean (sd)     diff')
diffs=[]
for k in range(5):
    h=[auc_of(load(H,k,s)) for s in range(3)]; a=[auc_of(load(A,k,s)) for s in range(3)]
    diffs.append(np.mean(a)-np.mean(h))
    print(f'  {k}   {np.mean(h):.3f} ({np.std(h):.3f})   {np.mean(a):.3f} ({np.std(a):.3f})   {diffs[-1]:+.3f}')
wins=sum(x>0 for x in diffs)
print(f'\nA1 higher on {wins}/5 folds, mean AUC gain {np.mean(diffs):+.3f}')
print('FREEZE RULE:', 'FREEZE A1' if (wins>=4 and np.mean(diffs)>0.02) else 'FREEZE H2')

def pooled(prefix,s):
    y=np.concatenate([load(prefix,k,s)['y'] for k in range(5)])>0
    sc=np.concatenate([1-load(prefix,k,s)['p'][:,0] for k in range(5)])
    return y,sc
def ops(y,sc):
    fpr,tpr,_=roc_curve(y,sc); sp=1-fpr
    return roc_auc_score(y,sc), tpr[sp>=0.70].max(), sp[tpr>=0.70].max()
print('\nPOOLED OUT OF FOLD, screening view (mean of 3 seeds)')
for name,pre in [('H2',H),('A1',A)]:
    r=np.array([ops(*pooled(pre,s)) for s in range(3)])
    print(f'  {name}: AUC {r[:,0].mean():.3f}   Se at Sp>=0.70: {r[:,1].mean():.2f}   Sp at Se>=0.70: {r[:,2].mean():.2f}')

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
amendment written
3690 rows, 82 patients loaded
  building X for 192f zeropad...
  H2_g2_nocw_f0_s1: official 50.85%  Se 31.1 Sp 70.6  2.2min
  H2_g2_nocw_f0_s2: official 50.89%  Se 36.2 Sp 65.6  2.1min
  H2_g2_nocw_f1_s1: official 54.37%  Se 28.9 Sp 79.8  2.5min
  H2_g2_nocw_f1_s2: official 58.14%  Se 35.3 Sp 81.0  2.5min
  H2_g2_nocw_f2_s1: official 46.46%  Se 10.3 Sp 82.6  2.5min
  H2_g2_nocw_f2_s2: official 47.91%  Se 12.6 Sp 83.2  2.6min
  H2_g2_nocw_f3_s1: official 56.86%  Se 15.5 Sp 98.2  2.4min
  H2_g2_nocw_f3_s2: official 58.64%  Se 24.8 Sp 92.5  2.5min
  H2_g2_nocw_f4_s1: official 54.54%  Se 25.2 Sp 83.9  2.4min
  H2_g2_nocw_f4_s2: official 60.36%  Se 35.5 Sp 85.2  2.3min

BINARY AUC per fold, mean of 3 seeds (SD across seeds)
fold   H2 mean (sd)     A1 mean (sd)     diff
  0   0.681 (0.010)   0.735 (0.003)   +0.054
  1   0.723 (0.009)   0.733 (0.002)   +0.010
  2   0.621 (0.018)   0.656 (0.019)   +0.035

In [ ]:
import os, json, time
import pandas as pd
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
root="/content/drive/MyDrive"; P=os.path.join(root,"processed")

print("===== folders =====")
for d in ["train","train_aug","train_aug_v2","val","test"]:
    p=os.path.join(P,d)
    if os.path.isdir(p):
        fs=sorted(os.listdir(p))
        print(f"{d:14s} {len(fs):6d} files  modified {time.strftime('%Y-%m-%d',time.localtime(os.path.getmtime(p)))}  sample {fs[:2]}")

for f in ["manifest.csv","manifest_aug.csv"]:
    df=pd.read_csv(os.path.join(P,f))
    print(f"\n===== {f} {df.shape} =====")
    print(list(df.columns))
    for c in df.columns:
        if 1<df[c].nunique()<=12:
            print(f"  {c}: {df[c].value_counts().to_dict()}")

print("\n===== augmentation code in notebooks =====")
keys=["train_aug","manifest_aug","augment","pitch","time_stretch","add_noise","mixup","SpecAug"]
nbs=[os.path.join(root,"Colab Notebooks",f) for f in os.listdir(os.path.join(root,"Colab Notebooks")) if f.endswith(".ipynb")]
nbs+=[os.path.join(root,"pulmoscan semester 7","notebooks",f) for f in os.listdir(os.path.join(root,"pulmoscan semester 7","notebooks")) if f.endswith(".ipynb")]
for p in nbs:
    try: nb=json.load(open(p))
    except Exception: continue
    shown=0
    for i,c in enumerate(nb["cells"]):
        if c["cell_type"]!="code": continue
        for line in "".join(c["source"]).splitlines():
            if any(k in line for k in keys) and shown<12:
                if shown==0: print("\n--",os.path.basename(p))
                print(f"  [cell {i}] {line.strip()[:140]}"); shown+=1

print("\n===== SE attention: last scores in outputs =====")
p=os.path.join(root,"pulmoscan semester 7","notebooks","se_attention_eval.ipynb")
nb=json.load(open(p)); lines=[]
for c in nb["cells"]:
    for o in c.get("outputs",[]):
        t=o.get("text")
        if t: lines+= "".join(t).splitlines()
hits=[l for l in lines if any(k in l.lower() for k in ["icbhi","official","macro","baseline","se_","attention"])]
for l in hits[-15:]: print("  ",l[:150])

===== folders =====
train            2891 files  modified 2026-03-18  sample ['102_1b1_Ar_sc_Meditron_cycle0_normal.npy', '102_1b1_Ar_sc_Meditron_cycle10_normal.npy']
train_aug        2276 files  modified 2026-03-18  sample ['103_2b2_Ar_mc_LittC2SE_cycle0_wheeze_aug0.npy', '103_2b2_Ar_mc_LittC2SE_cycle0_wheeze_aug1.npy']
train_aug_v2    18621 files  modified 2026-03-19  sample ['102_1b1_Ar_sc_Meditron_cycle0_normal_aug0.npy', '102_1b1_Ar_sc_Meditron_cycle0_normal_aug1.npy']
val               799 files  modified 2026-03-18  sample ['110_1b1_Pr_sc_Meditron_cycle0_crackle.npy', '110_1b1_Pr_sc_Meditron_cycle10_normal.npy']
test             1629 files  modified 2026-03-18  sample ['109_1b1_Al_sc_Litt3200_cycle0_normal.npy', '109_1b1_Al_sc_Litt3200_cycle1_normal.npy']

===== manifest.csv (5319, 7) =====
['audio_file', 'audio_path', 'features_path', 'sound_label', 'diagnosis_label', 'patient_id', 'split']
  sound_label: {0: 2920, 1: 1402, 2: 662, 3: 335}
  diagnosis_label: {1: 4409, 0: 238, 4

In [ ]:
import os, numpy as np
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
OOF="/content/drive/MyDrive/pulmoscan semester 7/results/oof_probs"
def load(pre,k,s): return np.load(os.path.join(OOF,f"{pre}_f{k}_s{s}.npz"))
def pooled(pre,s):
    y=[];sc=[];am=[];fo=[]
    for k in range(5):
        z=load(pre,k,s); y.append(z["y"]>0); sc.append(1-z["p"][:,0])
        am.append(np.argmax(z["p"],axis=1)!=0); fo.append(np.full(len(z["y"]),k))
    return np.concatenate(y),np.concatenate(sc),np.concatenate(am),np.concatenate(fo)

TARGETS=[0.60,0.70,0.80,0.90]
for pre in ["A1","H2_g2_nocw"]:
    print(f"\n=== {pre}: pooled out of fold, mean of 3 seeds ===")
    rows={t:[] for t in TARGETS}; arg=[]
    for s in range(3):
        y,sc,am,fo=pooled(pre,s)
        arg.append((am[y].mean(), (~am[~y]).mean()))
        for t in TARGETS:
            th=np.quantile(sc[~y],t)
            rows[t].append((th,(sc[y]>th).mean(),(sc[~y]<=th).mean()))
    a=np.mean(arg,axis=0)
    print(f"  argmax rule (what the official score uses): Se {a[0]:.2f}  Sp {a[1]:.2f}")
    for t in TARGETS:
        r=np.mean(rows[t],axis=0)
        print(f"  Sp target {t:.2f}: threshold {r[0]:.3f}  Se {r[1]:.2f}  Sp {r[2]:.2f}")

print("\n=== A1: ONE fixed threshold (Sp target 0.70, mean over seeds) applied per fold ===")
ths=[]
for s in range(3):
    y,sc,am,fo=pooled("A1",s); ths.append(np.quantile(sc[~y],0.70))
tb=float(np.mean(ths)); print(f"fixed threshold on P(abnormal): {tb:.3f}\n")
print("fold   Se     Sp   (mean of 3 seeds)")
for k in range(5):
    se=[];sp=[]
    for s in range(3):
        z=load("A1",k,s); yy=z["y"]>0; ss=1-z["p"][:,0]
        se.append((ss[yy]>tb).mean()); sp.append((ss[~yy]<=tb).mean())
    print(f"  {k}   {np.mean(se):.2f}  {np.mean(sp):.2f}")

print("\n=== transforms in the offline augmentation code ===")
for p in ["/content/drive/MyDrive/pulmoscan drive/src/augmentation.py"]:
    if os.path.exists(p):
        n=0
        for line in open(p).read().splitlines():
            if any(k in line for k in ["def ","librosa","shift","stretch","noise","gain","pitch"]) and n<25:
                print(" ",line.strip()[:130]); n+=1
    else:
        print("not found:",p)


=== A1: pooled out of fold, mean of 3 seeds ===
  argmax rule (what the official score uses): Se 0.41  Sp 0.84
  Sp target 0.60: threshold 0.452  Se 0.71  Sp 0.60
  Sp target 0.70: threshold 0.480  Se 0.61  Sp 0.70
  Sp target 0.80: threshold 0.521  Se 0.49  Sp 0.80
  Sp target 0.90: threshold 0.606  Se 0.33  Sp 0.90

=== H2_g2_nocw: pooled out of fold, mean of 3 seeds ===
  argmax rule (what the official score uses): Se 0.43  Sp 0.79
  Sp target 0.60: threshold 0.409  Se 0.65  Sp 0.60
  Sp target 0.70: threshold 0.459  Se 0.54  Sp 0.70
  Sp target 0.80: threshold 0.519  Se 0.42  Sp 0.80
  Sp target 0.90: threshold 0.637  Se 0.30  Sp 0.90

=== A1: ONE fixed threshold (Sp target 0.70, mean over seeds) applied per fold ===
fixed threshold on P(abnormal): 0.480

fold   Se     Sp   (mean of 3 seeds)
  0   0.74  0.60
  1   0.66  0.66
  2   0.45  0.78
  3   0.49  0.88
  4   0.67  0.67

=== transforms in the offline augmentation code ===
  import librosa
  def time_stretch(audio, rate=None):

In [ ]:
# ============================================================
# FINAL TRAINING: A1 on all 82 development patients. NO TEST DATA IS LOADED.
# ============================================================
import os, json, time, random, gc, zipfile, datetime
import numpy as np, pandas as pd, tensorflow as tf

print("GPUs:", tf.config.list_physical_devices('GPU'))
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
PROJ='/content/drive/MyDrive/pulmoscan semester 7'; WORK='/content/work'
RES=os.path.join(PROJ,'results'); CK=os.path.join(PROJ,'checkpoints')
N_MELS=128; T=192; EPOCHS=40; BATCH=32; LR=1e-3; LR_MIN=1e-6; DIAG_W=0.1; NS=4; ND=7
THRESH=0.480     # threshold on P(abnormal)=1-p[Normal]; 0.452 -> Se .71/Sp .60, 0.521 -> Se .49/Sp .80

# ---------- protocol, written before training and before any test use ----------
TP=os.path.join(RES,'test_protocol_final.json')
if not os.path.exists(TP):
    json.dump({
      'written_at':datetime.datetime.now().isoformat(),
      'frozen_config':'A1: 192f zero pad, focal gamma 2 no class weights, 40 epochs final weights, SpecAugment masks + level jitter',
      'freeze_reason':'3 seed mean binary AUC higher than H2 on 4/5 folds, mean gain +0.030 (aug_protocol amendment_1)',
      'training_data':'all 82 development patients (train+val, 3690 cycles); test split never loaded in this cell',
      'seeds':[0,1,2],'deployed_model':'seed 0, fixed in advance',
      'operating_point':{'threshold_on_P_abnormal':THRESH,'source':'pooled out of fold scores, A1, mean over 3 seeds'},
      'reference_out_of_fold':'threshold 0.480 gave pooled Se 0.61 Sp 0.70; per fold Se 0.45 to 0.74, Sp 0.60 to 0.88',
      'test_use':'second disclosed use of the test set (first: 3 seed B_192f ensemble on 2026-09-06, official 40.2)',
      'report_on_test':['binary Se and Sp at the fixed threshold, each seed individually and mean',
                        'binary AUC','argmax 4 class official ICBHI for continuity'],
      'evaluations_permitted':1,
      'commitment':'no threshold change, no seed selection, no rerun after seeing the test result'}, open(TP,'w'), indent=2)
    print('test protocol written')

# ---------- data (development only) ----------
os.makedirs(WORK,exist_ok=True)
if not os.path.isdir(os.path.join(WORK,'feat','processed','train')):
    with zipfile.ZipFile(os.path.join(PROJ,'data','pulmoscan_data.zip')) as z:
        z.extractall(os.path.join(WORK,'feat'))
with zipfile.ZipFile(os.path.join(PROJ,'data','colab_training_package.zip')) as z:
    z.extract('data/processed/manifest.csv', WORK)
man=pd.read_csv(os.path.join(WORK,'data/processed/manifest.csv'))
dev=man[man.split.isin(['train','val'])].copy().reset_index(drop=True)
dev['fp']=dev.apply(lambda r: os.path.join(WORK,'feat','processed',r['split'],
                    r['audio_file'].replace('.wav','.npy')),axis=1)
dev=dev[dev.fp.map(os.path.exists)].reset_index(drop=True)
assert 'test' not in set(dev.split)
print(f"{len(dev)} rows, {dev.patient_id.nunique()} patients loaded")   # expect 3690 and 82
X=np.empty((len(dev),N_MELS,T,1),np.float32)
for i in range(len(dev)):
    a=np.load(dev.fp[i]); X[i,...,0]=np.pad(a,[(0,0),(0,T-a.shape[1])]) if a.shape[1]<T else a[:,:T]
ys=dev.sound_label.values.astype(np.int32); yd=dev.diagnosis_label.values.astype(np.int32)

# ---------- loss, model, augmentation (identical to the A1 CV runs) ----------
def focal(g=2.0):
    def fn(yt,yp):
        yp=tf.clip_by_value(yp,1e-7,1.0); ce=-yt*tf.math.log(yp)
        p=tf.reduce_sum(yt*yp,axis=-1,keepdims=True)
        return tf.reduce_mean(tf.pow(1.0-p,g)*ce)
    fn.__name__='focal'; return fn
def build(T):
    inp=tf.keras.Input(shape=(N_MELS,T,1)); x=inp
    for f in [32,64,128]:
        x=tf.keras.layers.Conv2D(f,3,padding='same',activation='relu')(x)
        x=tf.keras.layers.BatchNormalization()(x); x=tf.keras.layers.MaxPooling2D(2)(x)
    x=tf.keras.layers.Conv2D(256,3,padding='same',activation='relu')(x)
    x=tf.keras.layers.BatchNormalization()(x)
    x=tf.keras.layers.GlobalAveragePooling2D()(x)
    sh=tf.keras.layers.Dropout(0.5)(tf.keras.layers.Dense(256,activation='relu')(x))
    s=tf.keras.layers.Dropout(0.3)(tf.keras.layers.Dense(128,activation='relu')(sh))
    so=tf.keras.layers.Dense(NS,activation='softmax',name='sound')(s)
    d=tf.keras.layers.Dropout(0.3)(tf.keras.layers.Dense(128,activation='relu')(sh))
    do=tf.keras.layers.Dense(ND,activation='softmax',name='diagnosis')(d)
    return tf.keras.Model(inp,[so,do])
def augment(x,y):
    B=tf.shape(x)[0]; F=tf.shape(x)[1]; Tt=tf.shape(x)[2]
    real=tf.cast(tf.not_equal(x,0.0),tf.float32)
    x=x+tf.random.normal([B,1,1,1],0.0,0.15)*real
    fw=tf.random.uniform([B,1,1,1],0,21,dtype=tf.int32)
    f0=tf.random.uniform([B,1,1,1],0,1000000,dtype=tf.int32)%tf.maximum(F-fw,1)
    fi=tf.reshape(tf.range(F),[1,-1,1,1]); fmask=tf.logical_and(fi>=f0,fi<f0+fw)
    tw=tf.random.uniform([B,1,1,1],0,31,dtype=tf.int32)
    t0=tf.random.uniform([B,1,1,1],0,1000000,dtype=tf.int32)%tf.maximum(Tt-tw,1)
    ti=tf.reshape(tf.range(Tt),[1,1,-1,1]); tmask=tf.logical_and(ti>=t0,ti<t0+tw)
    return tf.where(tf.logical_or(fmask,tmask),tf.zeros_like(x),x),y

# ---------- train and save ----------
os.makedirs(CK,exist_ok=True)
for seed in [0,1,2]:
    path=os.path.join(CK,f'final_A1_s{seed}.keras')
    if os.path.exists(path): print('  SKIP',os.path.basename(path)); continue
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    tf.keras.backend.clear_session()
    ds=tf.data.Dataset.from_tensor_slices((X,{'sound':tf.keras.utils.to_categorical(ys,NS),
                                               'diagnosis':tf.keras.utils.to_categorical(yd,ND)}))
    ds=ds.shuffle(4096,seed=seed).batch(BATCH).map(augment,num_parallel_calls=tf.data.AUTOTUNE).prefetch(2)
    m=build(T); m.compile(optimizer=tf.keras.optimizers.Adam(LR),
        loss={'sound':focal(2.0),'diagnosis':focal(2.0)},
        loss_weights={'sound':1.0,'diagnosis':DIAG_W},metrics={'sound':'accuracy'})
    t0=time.time()
    m.fit(ds,epochs=EPOCHS,verbose=0,callbacks=[tf.keras.callbacks.LearningRateScheduler(
        lambda e: float(LR_MIN+0.5*(LR-LR_MIN)*(1+np.cos(np.pi*e/EPOCHS))))])
    try: m.save(path)
    except Exception as e:
        print('  full save failed:',str(e)[:100],'-> saving weights only')
        m.save_weights(path.replace('.keras','.weights.h5'))
    print(f'  final_A1_s{seed}: trained in {(time.time()-t0)/60:.1f}min, params {m.count_params()}')
    del m; gc.collect()

print('\ncheckpoints on Drive:')
for f in sorted(os.listdir(CK)):
    if f.startswith('final_A1'): print(f'  {f}  {os.path.getsize(os.path.join(CK,f))/1e6:.1f} MB')

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
test protocol written
3690 rows, 82 patients loaded
  final_A1_s0: trained in 2.9min, params 522763
  final_A1_s1: trained in 3.0min, params 522763
  final_A1_s2: trained in 3.0min, params 522763

checkpoints on Drive:
  final_A1_s0.keras  6.4 MB
  final_A1_s1.keras  6.4 MB
  final_A1_s2.keras  6.4 MB


In [ ]:
# ============================================================
# FINAL TEST EVALUATION - ONE USE ONLY (second disclosed use of the test set)
# ============================================================
import os, json, datetime, zipfile
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.metrics import roc_auc_score, confusion_matrix

REASON = ""     # REQUIRED: one sentence, why this operating point. The cell stops if empty.

REASON = "Chose 0.480 because it gave pooled out of fold Se 0.61 and Sp 0.70, a balance between missing abnormal cycles and false alarms for a screening prototype."
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
PROJ='/content/drive/MyDrive/pulmoscan semester 7'; WORK='/content/work'
RES=os.path.join(PROJ,'results'); CK=os.path.join(PROJ,'checkpoints')
TP=os.path.join(RES,'test_protocol_final.json'); RESF=os.path.join(RES,'test_result_A1_FINAL.json')
N_MELS=128; T=192; SOUND=['Normal','Crackle','Wheeze','Both']

if os.path.exists(RESF):
    print("Already evaluated. The test set may be used once. Stored result:\n")
    print(json.dumps(json.load(open(RESF)),indent=1)[:4000]); raise SystemExit

tp=json.load(open(TP))
THRESH=tp['operating_point']['threshold_on_P_abnormal']
tp['operating_point']['reason']=REASON.strip()
tp['evaluation_started_at']=datetime.datetime.now().isoformat()
json.dump(tp,open(TP,'w'),indent=2)
print("threshold from protocol file:",THRESH)

# ---------- load test features ----------
os.makedirs(WORK,exist_ok=True)
with zipfile.ZipFile(os.path.join(PROJ,'data','colab_training_package.zip')) as z:
    z.extract('data/processed/manifest.csv', WORK)
man=pd.read_csv(os.path.join(WORK,'data/processed/manifest.csv'))
test=man[man.split=='test'].reset_index(drop=True)
print("test rows:",len(test),"patients:",test.patient_id.nunique())     # expect 1629

def find_npys(root):
    d={}
    for dp,_,fs in os.walk(root):
        for f in fs:
            if f.endswith('.npy'): d[f]=os.path.join(dp,f)
    return d
names=test.audio_file.str.replace('.wav','.npy',regex=False)
found=find_npys(os.path.join(WORK,'feat')) if os.path.isdir(os.path.join(WORK,'feat')) else {}
if not all(n in found for n in names):
    tz=os.path.join(PROJ,'data','test_features.zip')
    with zipfile.ZipFile(tz) as z: z.extractall(os.path.join(WORK,'testfeat'))
    found.update(find_npys(os.path.join(WORK,'testfeat')))
missing=[n for n in names if n not in found]
assert not missing, f"{len(missing)} test features not found, e.g. {missing[:3]}"

X=np.empty((len(test),N_MELS,T,1),np.float32)
for i,n in enumerate(names):
    a=np.load(found[n]); X[i,...,0]=np.pad(a,[(0,0),(0,T-a.shape[1])]) if a.shape[1]<T else a[:,:T]
ys=test.sound_label.values.astype(np.int32); yb=ys>0; pid=test.patient_id.values
print("test class counts:",{SOUND[i]:int((ys==i).sum()) for i in range(4)})

# ---------- evaluate the 3 frozen models ----------
def official(yt,yp):
    cm=confusion_matrix(yt,yp,labels=[0,1,2,3])
    sp=cm[0,0]/cm[0].sum(); se=sum(cm[i,i] for i in [1,2,3])/sum(cm[i].sum() for i in [1,2,3])
    return (se+sp)/2,se,sp
per={}; scores={}
for s in [0,1,2]:
    m=tf.keras.models.load_model(os.path.join(CK,f'final_A1_s{s}.keras'),compile=False)
    p=m.predict(X,batch_size=64,verbose=0)[0]; sc=1-p[:,0]; scores[s]=sc
    pred=sc>THRESH
    off,se4,sp4=official(ys,np.argmax(p,axis=1))
    per[s]={'binary_se':float(pred[yb].mean()),'binary_sp':float((~pred[~yb]).mean()),
            'auc':float(roc_auc_score(yb,sc)),'official_argmax':float(off),
            'confusion_argmax':confusion_matrix(ys,np.argmax(p,axis=1),labels=[0,1,2,3]).tolist()}

# ---------- patient level bootstrap for the deployed model (seed 0), descriptive only ----------
rng=np.random.RandomState(0); upid=np.unique(pid); idx={u:np.where(pid==u)[0] for u in upid}
bs=[]
for _ in range(1000):
    ii=np.concatenate([idx[u] for u in rng.choice(upid,len(upid),replace=True)])
    y_=yb[ii]; s_=scores[0][ii]
    if y_.sum()==0 or (~y_).sum()==0: continue
    pr=s_>THRESH; bs.append((pr[y_].mean(),(~pr[~y_]).mean(),roc_auc_score(y_,s_)))
ci=np.percentile(np.array(bs),[2.5,97.5],axis=0)

out={'threshold':THRESH,'reason':REASON.strip(),'n_test':len(test),'n_patients':int(test.patient_id.nunique()),
     'per_seed':per,
     'mean_over_seeds':{k:float(np.mean([per[s][k] for s in per])) for k in ['binary_se','binary_sp','auc','official_argmax']},
     'seed0_patient_bootstrap_95ci':{'binary_se':ci[:,0].tolist(),'binary_sp':ci[:,1].tolist(),'auc':ci[:,2].tolist()},
     'written_at':datetime.datetime.now().isoformat()}
json.dump(out,open(RESF,'w'),indent=2)

print("\n============ TEST RESULT (final, do not rerun) ============")
print(f"threshold {THRESH} on P(abnormal)\n")
print("seed   binary Se   binary Sp   AUC     official(argmax)")
for s in per:
    r=per[s]; print(f"  {s}     {r['binary_se']:.3f}       {r['binary_sp']:.3f}     {r['auc']:.3f}   {r['official_argmax']*100:.2f}")
m=out['mean_over_seeds']
print(f"mean   {m['binary_se']:.3f}       {m['binary_sp']:.3f}     {m['auc']:.3f}   {m['official_argmax']*100:.2f}")
print(f"\ndeployed model (seed 0), patient bootstrap 95% CI:")
print(f"  Se {ci[0,0]:.2f} to {ci[1,0]:.2f}   Sp {ci[0,1]:.2f} to {ci[1,1]:.2f}   AUC {ci[0,2]:.3f} to {ci[1,2]:.3f}")
print("\nreference: out of fold Se 0.61 Sp 0.70 AUC 0.715 | all Normal floor official 50.0 | earlier test result 40.2")

threshold from protocol file: 0.48
test rows: 1629 patients: 19
test class counts: {'Normal': 771, 'Crackle': 572, 'Wheeze': 183, 'Both': 103}

============ TEST RESULT (final, do not rerun) ============
threshold 0.48 on P(abnormal)

seed   binary Se   binary Sp   AUC     official(argmax)
  0     0.479       0.598     0.548   46.67
  1     0.492       0.598     0.560   47.54
  2     0.508       0.604     0.575   50.09
mean   0.493       0.600     0.561   48.10

deployed model (seed 0), patient bootstrap 95% CI:
  Se 0.39 to 0.68   Sp 0.51 to 0.69   AUC 0.489 to 0.690

reference: out of fold Se 0.61 Sp 0.70 AUC 0.715 | all Normal floor official 50.0 | earlier test result 40.2


In [ ]:
import os, zipfile
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
PROJ='/content/drive/MyDrive/pulmoscan semester 7'; WORK='/content/work'
OOF=os.path.join(PROJ,'results','oof_probs'); os.makedirs(WORK,exist_ok=True)
with zipfile.ZipFile(os.path.join(PROJ,'data','colab_training_package.zip')) as z:
    z.extract('data/processed/manifest.csv', WORK)
man=pd.read_csv(os.path.join(WORK,'data/processed/manifest.csv'))
man['device']=man.audio_file.str.split('_').str[4]
dev=man[man.split.isin(['train','val'])].reset_index(drop=True)
test=man[man.split=='test'].reset_index(drop=True)

print("=== device composition (rows | patients) ===")
for name,df in [('dev',dev),('test',test)]:
    g=df.groupby('device').agg(rows=('audio_file','size'),patients=('patient_id','nunique'))
    g['row_share']=(g.rows/g.rows.sum()).round(2)
    print(f"\n{name}"); print(g.to_string())

# rebuild the exact folds used in CV
pid=dev.patient_id.values; ys=dev.sound_label.values
rng=np.random.RandomState(42)
pdf=dev.groupby('patient_id')['diagnosis_label'].agg(lambda x:x.mode()[0]).reset_index()
folds=[[] for _ in range(5)]
for _,g in pdf.groupby('diagnosis_label'):
    ps=g.patient_id.values.copy(); rng.shuffle(ps)
    for j,p in enumerate(ps): folds[j%5].append(p)

print("\n=== A1 out of fold binary AUC by device (dev only, mean of 3 seeds) ===")
res={}
for s in range(3):
    dv=[];yy=[];sc=[]
    for k in range(5):
        te=np.isin(pid,folds[k]); z=np.load(os.path.join(OOF,f'A1_f{k}_s{s}.npz'))
        assert len(z['y'])==te.sum() and (z['y']==ys[te]).all(), "fold reconstruction mismatch"
        dv.append(dev.device.values[te]); yy.append(z['y']>0); sc.append(1-z['p'][:,0])
    dv=np.concatenate(dv); yy=np.concatenate(yy); sc=np.concatenate(sc)
    for d in np.unique(dv):
        m=dv==d
        if yy[m].sum()>0 and (~yy[m]).sum()>0:
            res.setdefault(d,[]).append(roc_auc_score(yy[m],sc[m]))
n_by=dev.groupby('device').size()
for d,v in res.items():
    print(f"  {d:10s} AUC {np.mean(v):.3f}   rows {n_by[d]}")

=== device composition (rows | patients) ===

dev
          rows  patients  row_share
device                             
AKGC417L  2015        18       0.55
Litt3200   342         7       0.09
LittC2SE   455        18       0.12
Meditron   878        41       0.24

test
          rows  patients  row_share
device                             
AKGC417L  1314         7       0.81
Litt3200    61         1       0.04
LittC2SE    58         2       0.04
Meditron   196         9       0.12

=== A1 out of fold binary AUC by device (dev only, mean of 3 seeds) ===
  AKGC417L   AUC 0.677   rows 2015
  Litt3200   AUC 0.584   rows 342
  LittC2SE   AUC 0.809   rows 455
  Meditron   AUC 0.597   rows 878


In [ ]:
# ============================================================
# TFLITE INT8 CONVERSION of final_A1_s0 (sound head only). No test data touched.
# ============================================================
import os, zipfile, json
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.metrics import roc_auc_score

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
PROJ='/content/drive/MyDrive/pulmoscan semester 7'; WORK='/content/work'
CK=os.path.join(PROJ,'checkpoints'); OUT=os.path.join(PROJ,'tflite'); os.makedirs(OUT,exist_ok=True)
N_MELS=128; T=192; THRESH=0.48
print("TF", tf.__version__)

os.makedirs(WORK,exist_ok=True)
if not os.path.isdir(os.path.join(WORK,'feat','processed','train')):
    with zipfile.ZipFile(os.path.join(PROJ,'data','pulmoscan_data.zip')) as z:
        z.extractall(os.path.join(WORK,'feat'))
with zipfile.ZipFile(os.path.join(PROJ,'data','colab_training_package.zip')) as z:
    z.extract('data/processed/manifest.csv', WORK)
man=pd.read_csv(os.path.join(WORK,'data/processed/manifest.csv'))
dev=man[man.split.isin(['train','val'])].copy().reset_index(drop=True)
dev['fp']=dev.apply(lambda r: os.path.join(WORK,'feat','processed',r['split'],
                    r['audio_file'].replace('.wav','.npy')),axis=1)
dev=dev[dev.fp.map(os.path.exists)].reset_index(drop=True)
assert 'test' not in set(dev.split)
X=np.empty((len(dev),N_MELS,T,1),np.float32)
for i in range(len(dev)):
    a=np.load(dev.fp[i]); X[i,...,0]=np.pad(a,[(0,0),(0,T-a.shape[1])]) if a.shape[1]<T else a[:,:T]
yb=(dev.sound_label.values>0)
print("dev rows:",len(dev))                                     # expect 3690

full=tf.keras.models.load_model(os.path.join(CK,'final_A1_s0.keras'),compile=False)
model=tf.keras.Model(full.input,full.outputs[0])                # sound head only
print("sound model params:",model.count_params())

rng=np.random.RandomState(0)
cal=rng.choice(len(dev),300,replace=False)
def rep():
    for i in cal: yield [X[i:i+1]]

try:
    conv=tf.lite.TFLiteConverter.from_keras_model(model)
except Exception as e:
    print("from_keras_model failed, using exported SavedModel:",str(e)[:100])
    model.export(os.path.join(WORK,'sm')); conv=tf.lite.TFLiteConverter.from_saved_model(os.path.join(WORK,'sm'))
conv.optimizations=[tf.lite.Optimize.DEFAULT]
conv.representative_dataset=rep
conv.target_spec.supported_ops=[tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv.inference_input_type=tf.int8; conv.inference_output_type=tf.int8
tfl=conv.convert()
path=os.path.join(OUT,'pulmoscan_A1_s0_int8.tflite'); open(path,'wb').write(tfl)
print(f"\nint8 model size: {len(tfl)/1e3:.0f} KB   saved: {path}")

# ---------- float vs int8 on 800 dev rows ----------
ev=rng.choice(len(dev),800,replace=False)
pf=1-model.predict(X[ev],batch_size=64,verbose=0)[:,0]
itp=tf.lite.Interpreter(model_content=tfl); itp.allocate_tensors()
inp=itp.get_input_details()[0]; out=itp.get_output_details()[0]
si,zi=inp['quantization']; so,zo=out['quantization']
pq=np.empty(len(ev))
for j,i in enumerate(ev):
    q=np.clip(np.round(X[i:i+1]/si+zi),-128,127).astype(np.int8)
    itp.set_tensor(inp['index'],q); itp.invoke()
    o=(itp.get_tensor(out['index']).astype(np.float32)-zo)*so
    pq[j]=1-o[0,0]
print("\n=== float vs int8, P(abnormal) on 800 dev rows ===")
print(f"mean abs diff {np.mean(np.abs(pf-pq)):.4f}   max abs diff {np.max(np.abs(pf-pq)):.4f}   corr {np.corrcoef(pf,pq)[0,1]:.4f}")
print(f"decision agreement at {THRESH}: {np.mean((pf>THRESH)==(pq>THRESH)):.4f}")
print(f"dev AUC float {roc_auc_score(yb[ev],pf):.3f}   int8 {roc_auc_score(yb[ev],pq):.3f}   (in sample, compare the two only)")
print(f"output quantization step: {so:.5f}   input scale {si:.4f} zero point {zi}")

# ---------- rough activation sizes (bytes, int8) ----------
print("\nlargest tensors in the int8 graph:")
td=sorted(itp.get_tensor_details(),key=lambda d:-np.prod(d['shape']))[:5]
for d in td: print(f"  {str(d['shape']):22s} {int(np.prod(d['shape']))/1e3:7.0f} KB  {d['name'][:50]}")

TF 2.20.0
dev rows: 3690
sound model params: 488964
Saved artifact at '/tmp/tmpmin4ayai'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 192, 1), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  137192996361936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137193098734160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137193098735888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137193098736272: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137193098736080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137193098733008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137193098733968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137193098739152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137193098734736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137193098735312: TensorSpec(shape

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(



int8 model size: 520 KB   saved: /content/drive/MyDrive/pulmoscan semester 7/tflite/pulmoscan_A1_s0_int8.tflite


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



=== float vs int8, P(abnormal) on 800 dev rows ===
mean abs diff 0.0809   max abs diff 0.3603   corr 0.9285
decision agreement at 0.48: 0.8125
dev AUC float 0.901   int8 0.852   (in sample, compare the two only)
output quantization step: 0.00391   input scale 0.0480 zero point -1

largest tensors in the int8 graph:
  [  1 128 192  32]          786 KB  functional_1_1/conv2d_1/Relu;functional_1_1/conv2d
  [  1 128 192  32]          786 KB  functional_1_1/batch_normalization_1/batchnorm/mul
  [  1 128 192  32]          786 KB  functional_1_1/batch_normalization_1/batchnorm/add
  [ 1 64 96 64]              393 KB  functional_1_1/conv2d_1_2/Relu;functional_1_1/conv
  [ 1 64 96 64]              393 KB  functional_1_1/batch_normalization_1_2/batchnorm/m


In [ ]:
# ============================================================
# RECALIBRATION CHECK for int8 (CPU is fine). No test data touched.
# ============================================================
import os, zipfile, json
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.metrics import roc_auc_score

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
PROJ='/content/drive/MyDrive/pulmoscan semester 7'; WORK='/content/work'
CK=os.path.join(PROJ,'checkpoints'); OUT=os.path.join(PROJ,'tflite'); os.makedirs(OUT,exist_ok=True)
N_MELS=128; T=192; THRESH=0.48

os.makedirs(WORK,exist_ok=True)
if not os.path.isdir(os.path.join(WORK,'feat','processed','train')):
    with zipfile.ZipFile(os.path.join(PROJ,'data','pulmoscan_data.zip')) as z:
        z.extractall(os.path.join(WORK,'feat'))
with zipfile.ZipFile(os.path.join(PROJ,'data','colab_training_package.zip')) as z:
    z.extract('data/processed/manifest.csv', WORK)
man=pd.read_csv(os.path.join(WORK,'data/processed/manifest.csv'))
dev=man[man.split.isin(['train','val'])].copy().reset_index(drop=True)
dev['fp']=dev.apply(lambda r: os.path.join(WORK,'feat','processed',r['split'],
                    r['audio_file'].replace('.wav','.npy')),axis=1)
dev=dev[dev.fp.map(os.path.exists)].reset_index(drop=True)
assert 'test' not in set(dev.split)
X=np.empty((len(dev),N_MELS,T,1),np.float32)
for i in range(len(dev)):
    a=np.load(dev.fp[i]); X[i,...,0]=np.pad(a,[(0,0),(0,T-a.shape[1])]) if a.shape[1]<T else a[:,:T]
ys=dev.sound_label.values; yb=ys>0
print("dev rows:",len(dev))                                   # expect 3690

full=tf.keras.models.load_model(os.path.join(CK,'final_A1_s0.keras'),compile=False)
model=tf.keras.Model(full.input,full.outputs[0])

print("\nlayer order (first 12):")
for l in model.layers[:12]:
    print(" ",l.__class__.__name__, l.get_config().get('activation',''))

rng=np.random.RandomState(0)
_=rng.choice(len(dev),300,replace=False)                     # same draws as before, so eval rows are identical
ev=rng.choice(len(dev),800,replace=False)
pf=1-model.predict(X[ev],batch_size=64,verbose=0)[:,0]

def convert(cal_idx):
    def rep():
        for i in cal_idx: yield [X[i:i+1]]
    conv=tf.lite.TFLiteConverter.from_keras_model(model)
    conv.optimizations=[tf.lite.Optimize.DEFAULT]
    conv.representative_dataset=rep
    conv.target_spec.supported_ops=[tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    conv.inference_input_type=tf.int8; conv.inference_output_type=tf.int8
    return conv.convert()

Mounted at /content/drive
dev rows: 3690

layer order (first 12):
  InputLayer 
  Conv2D relu
  BatchNormalization 
  MaxPooling2D 
  Conv2D relu
  BatchNormalization 
  MaxPooling2D 
  Conv2D relu
  BatchNormalization 
  MaxPooling2D 
  Conv2D relu
  BatchNormalization 


In [ ]:
# ============================================================
# CONTINUATION of the recalibration check (uses variables from the previous cell)
# ============================================================
assert all(k in globals() for k in ['convert','pf','ev','X','ys','yb','THRESH','OUT','PROJ']), \
    "Session was reset: rerun the previous cell first, then this one."

def evaluate(tfl):
    itp=tf.lite.Interpreter(model_content=tfl); itp.allocate_tensors()
    inp=itp.get_input_details()[0]; out=itp.get_output_details()[0]
    si,zi=inp['quantization']; so,zo=out['quantization']; pq=np.empty(len(ev))
    for j,i in enumerate(ev):
        q=np.clip(np.round(X[i:i+1]/si+zi),-128,127).astype(np.int8)
        itp.set_tensor(inp['index'],q); itp.invoke()
        pq[j]=1-(itp.get_tensor(out['index']).astype(np.float32)-zo)[0,0]*so
    return pq

r2=np.random.RandomState(1)
strat=np.concatenate([r2.choice(np.where(ys==c)[0],min(375,int((ys==c).sum())),replace=False) for c in range(4)])
variants={'rand1500':r2.choice(len(dev),1500,replace=False),'balanced':strat}

rows=[]
for name,idx in variants.items():
    tfl=convert(idx); open(os.path.join(OUT,f'pulmoscan_A1_s0_int8_{name}.tflite'),'wb').write(tfl)
    pq=evaluate(tfl)
    r=dict(variant=name,n_cal=len(idx),size_kb=len(tfl)/1e3,mad=float(np.mean(np.abs(pf-pq))),
           corr=float(np.corrcoef(pf,pq)[0,1]),agree=float(np.mean((pf>THRESH)==(pq>THRESH))),
           auc_float=float(roc_auc_score(yb[ev],pf)),auc_int8=float(roc_auc_score(yb[ev],pq)))
    rows.append(r)
    print(f"\n{name}: cal rows {r['n_cal']}, size {r['size_kb']:.0f} KB, mean abs diff {r['mad']:.4f}, "
          f"corr {r['corr']:.4f}, agreement {r['agree']:.4f}, AUC float {r['auc_float']:.3f} int8 {r['auc_int8']:.3f}")
    print("  ", "PASS" if (r['agree']>=0.95 and r['auc_float']-r['auc_int8']<=0.02) else "FAIL",
          "(bar: agreement 0.95, AUC drop at most 0.02)")
json.dump(rows,open(os.path.join(PROJ,'results','tflite_recalibration.json'),'w'),indent=2)
print("\nfor reference, first attempt (300 random rows): agreement 0.8125, AUC 0.901 to 0.852")

Saved artifact at '/tmp/tmpmcvwrxa8'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 192, 1), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  133154706569552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133154706570512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133154706569936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133154706570320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133154706569360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133154706570128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133154706570704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133154706569744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133154706568208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133154684830160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133154706571088

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



rand1500: cal rows 1500, size 520 KB, mean abs diff 0.0886, corr 0.8856, agreement 0.7900, AUC float 0.901 int8 0.828
   FAIL (bar: agreement 0.95, AUC drop at most 0.02)
Saved artifact at '/tmp/tmphkj6ttfv'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 192, 1), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  133154706569552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133154706570512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133154706569936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133154706570320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133154706569360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133154706570128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133154706570704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133154706569744: TensorSpec(shape=(), dtype=tf.resour

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



balanced: cal rows 1357, size 520 KB, mean abs diff 0.1060, corr 0.8424, agreement 0.8075, AUC float 0.901 int8 0.814
   FAIL (bar: agreement 0.95, AUC drop at most 0.02)

for reference, first attempt (300 random rows): agreement 0.8125, AUC 0.901 to 0.852


In [ ]:
# ============================================================
# BN ORDER TEST: B1 (Conv -> BN -> ReLU) trained on all 82 dev patients, int8 check
# No test data. T4 GPU, about 10 minutes.
# ============================================================
import os, json, time, random, zipfile, datetime
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.metrics import roc_auc_score

print("GPUs:", tf.config.list_physical_devices('GPU'))
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
PROJ='/content/drive/MyDrive/pulmoscan semester 7'; WORK='/content/work'
RES=os.path.join(PROJ,'results'); CK=os.path.join(PROJ,'checkpoints'); OUT=os.path.join(PROJ,'tflite')
os.makedirs(OUT,exist_ok=True); os.makedirs(CK,exist_ok=True)
N_MELS=128; T=192; EPOCHS=40; BATCH=32; LR=1e-3; LR_MIN=1e-6; DIAG_W=0.1; NS=4; ND=7; THRESH=0.48; SEED=0

# ---------- protocol, written before training ----------
BP=os.path.join(RES,'bn_order_protocol.json')
if not os.path.exists(BP):
    json.dump({
      'written_at':datetime.datetime.now().isoformat(),
      'hypothesis':'int8 failure of A1 (agreement 0.79 to 0.81, AUC drop 0.05 to 0.09 on dev rows) is caused by BatchNorm placed after ReLU, which cannot fold into the conv weights',
      'evidence_so_far':'calibration sets of 300, 1500 and 1357 balanced rows all failed; layer order confirmed Conv(relu) then BN',
      'step1':'train B1 = A1 with Conv, BN, ReLU order (nothing else changed), seed 0, all 82 dev patients, same augmentation and recipe',
      'step1_check':'int8 conversion with ONE pre registered calibration set (1500 random dev rows, RandomState(1)); same 800 evaluation rows as before',
      'step1_bar':'decision agreement at 0.48 at least 0.95 and dev AUC drop (float minus int8) at most 0.02',
      'if_step1_fails':'BN placement is not the cause; stop retraining; use float32 on the board and measure latency',
      'if_step1_passes':'run 15 CV runs of B1 against A1 (5 folds x 3 seeds); adopt B1 only if mean fold binary AUC is at least A1 mean minus 0.01 (A1 reference 0.721)',
      'threshold_if_adopted':'re-derived from B1 pooled out of fold scores at Sp 0.70, same rule as A1; not tuned on test',
      'test_set':'not used; if B1 is deployed the report states that the test result belongs to the float A1 model',
      'commitment':'one calibration set, no retries with other sets after seeing the result'}, open(BP,'w'), indent=2)
    print('protocol written')

# ---------- data (development only) ----------
os.makedirs(WORK,exist_ok=True)
if not os.path.isdir(os.path.join(WORK,'feat','processed','train')):
    with zipfile.ZipFile(os.path.join(PROJ,'data','pulmoscan_data.zip')) as z:
        z.extractall(os.path.join(WORK,'feat'))
with zipfile.ZipFile(os.path.join(PROJ,'data','colab_training_package.zip')) as z:
    z.extract('data/processed/manifest.csv', WORK)
man=pd.read_csv(os.path.join(WORK,'data/processed/manifest.csv'))
dev=man[man.split.isin(['train','val'])].copy().reset_index(drop=True)
dev['fp']=dev.apply(lambda r: os.path.join(WORK,'feat','processed',r['split'],
                    r['audio_file'].replace('.wav','.npy')),axis=1)
dev=dev[dev.fp.map(os.path.exists)].reset_index(drop=True)
assert 'test' not in set(dev.split)
X=np.empty((len(dev),N_MELS,T,1),np.float32)
for i in range(len(dev)):
    a=np.load(dev.fp[i]); X[i,...,0]=np.pad(a,[(0,0),(0,T-a.shape[1])]) if a.shape[1]<T else a[:,:T]
ys=dev.sound_label.values.astype(np.int32); yd=dev.diagnosis_label.values.astype(np.int32); yb=ys>0
print(f"{len(dev)} rows, {dev.patient_id.nunique()} patients loaded")      # expect 3690 and 82

# ---------- loss, B1 model, augmentation ----------
def focal(g=2.0):
    def fn(yt,yp):
        yp=tf.clip_by_value(yp,1e-7,1.0); ce=-yt*tf.math.log(yp)
        p=tf.reduce_sum(yt*yp,axis=-1,keepdims=True)
        return tf.reduce_mean(tf.pow(1.0-p,g)*ce)
    fn.__name__='focal'; return fn
def build_b1(T):
    L=tf.keras.layers; inp=tf.keras.Input(shape=(N_MELS,T,1)); x=inp
    for f in [32,64,128]:
        x=L.Conv2D(f,3,padding='same')(x); x=L.BatchNormalization()(x)
        x=L.Activation('relu')(x); x=L.MaxPooling2D(2)(x)
    x=L.Conv2D(256,3,padding='same')(x); x=L.BatchNormalization()(x); x=L.Activation('relu')(x)
    x=L.GlobalAveragePooling2D()(x)
    sh=L.Dropout(0.5)(L.Dense(256,activation='relu')(x))
    s=L.Dropout(0.3)(L.Dense(128,activation='relu')(sh))
    so=L.Dense(NS,activation='softmax',name='sound')(s)
    d=L.Dropout(0.3)(L.Dense(128,activation='relu')(sh))
    do=L.Dense(ND,activation='softmax',name='diagnosis')(d)
    return tf.keras.Model(inp,[so,do])
def augment(x,y):
    B=tf.shape(x)[0]; F=tf.shape(x)[1]; Tt=tf.shape(x)[2]
    real=tf.cast(tf.not_equal(x,0.0),tf.float32)
    x=x+tf.random.normal([B,1,1,1],0.0,0.15)*real
    fw=tf.random.uniform([B,1,1,1],0,21,dtype=tf.int32)
    f0=tf.random.uniform([B,1,1,1],0,1000000,dtype=tf.int32)%tf.maximum(F-fw,1)
    fi=tf.reshape(tf.range(F),[1,-1,1,1]); fmask=tf.logical_and(fi>=f0,fi<f0+fw)
    tw=tf.random.uniform([B,1,1,1],0,31,dtype=tf.int32)
    t0=tf.random.uniform([B,1,1,1],0,1000000,dtype=tf.int32)%tf.maximum(Tt-tw,1)
    ti=tf.reshape(tf.range(Tt),[1,1,-1,1]); tmask=tf.logical_and(ti>=t0,ti<t0+tw)
    return tf.where(tf.logical_or(fmask,tmask),tf.zeros_like(x),x),y

# ---------- train B1 ----------
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED); tf.keras.backend.clear_session()
ds=tf.data.Dataset.from_tensor_slices((X,{'sound':tf.keras.utils.to_categorical(ys,NS),
                                           'diagnosis':tf.keras.utils.to_categorical(yd,ND)}))
ds=ds.shuffle(4096,seed=SEED).batch(BATCH).map(augment,num_parallel_calls=tf.data.AUTOTUNE).prefetch(2)
m=build_b1(T)
print("layer order:",[l.__class__.__name__ for l in m.layers[1:8]])
print("params:",m.count_params())                                   # A1 had 522763
m.compile(optimizer=tf.keras.optimizers.Adam(LR),
    loss={'sound':focal(2.0),'diagnosis':focal(2.0)},
    loss_weights={'sound':1.0,'diagnosis':DIAG_W},metrics={'sound':'accuracy'})
t0=time.time()
m.fit(ds,epochs=EPOCHS,verbose=0,callbacks=[tf.keras.callbacks.LearningRateScheduler(
    lambda e: float(LR_MIN+0.5*(LR-LR_MIN)*(1+np.cos(np.pi*e/EPOCHS))))])
print(f"trained in {(time.time()-t0)/60:.1f} min")
m.save(os.path.join(CK,'final_B1_s0.keras'))
model=tf.keras.Model(m.input,m.outputs[0])                          # sound head only

# ---------- int8 conversion, one pre registered calibration set ----------
cal=np.random.RandomState(1).choice(len(dev),1500,replace=False)
def rep():
    for i in cal: yield [X[i:i+1]]
conv=tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations=[tf.lite.Optimize.DEFAULT]; conv.representative_dataset=rep
conv.target_spec.supported_ops=[tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv.inference_input_type=tf.int8; conv.inference_output_type=tf.int8
tfl=conv.convert(); open(os.path.join(OUT,'pulmoscan_B1_s0_int8.tflite'),'wb').write(tfl)

# ---------- same 800 evaluation rows as the A1 check ----------
rng=np.random.RandomState(0); _=rng.choice(len(dev),300,replace=False); ev=rng.choice(len(dev),800,replace=False)
pf=1-model.predict(X[ev],batch_size=64,verbose=0)[:,0]
itp=tf.lite.Interpreter(model_content=tfl); itp.allocate_tensors()
inp=itp.get_input_details()[0]; out=itp.get_output_details()[0]
si,zi=inp['quantization']; so,zo=out['quantization']; pq=np.empty(len(ev))
for j,i in enumerate(ev):
    q=np.clip(np.round(X[i:i+1]/si+zi),-128,127).astype(np.int8)
    itp.set_tensor(inp['index'],q); itp.invoke()
    pq[j]=1-(itp.get_tensor(out['index']).astype(np.float32)-zo)[0,0]*so
agree=float(np.mean((pf>THRESH)==(pq>THRESH))); a_f=float(roc_auc_score(yb[ev],pf)); a_q=float(roc_auc_score(yb[ev],pq))
ok=(agree>=0.95 and a_f-a_q<=0.02)
print(f"\nB1 int8: size {len(tfl)/1e3:.0f} KB, mean abs diff {np.mean(np.abs(pf-pq)):.4f}, corr {np.corrcoef(pf,pq)[0,1]:.4f}, "
      f"agreement {agree:.4f}, AUC float {a_f:.3f} int8 {a_q:.3f}")
print("RESULT:", "PASS" if ok else "FAIL", "(bar: agreement 0.95, AUC drop at most 0.02)")
print("for reference, A1: agreement 0.79 to 0.81, AUC 0.901 to 0.81 to 0.85")
d=json.load(open(BP))
d['step1_result']={'written_at':datetime.datetime.now().isoformat(),'agreement':agree,'auc_float':a_f,'auc_int8':a_q,
                   'size_kb':len(tfl)/1e3,'passed':bool(ok)}
json.dump(d,open(BP,'w'),indent=2)

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Mounted at /content/drive
protocol written
3690 rows, 82 patients loaded
layer order: ['Conv2D', 'BatchNormalization', 'Activation', 'MaxPooling2D', 'Conv2D', 'BatchNormalization', 'Activation']
params: 522763
trained in 3.6 min
Saved artifact at '/tmp/tmpzi2cwxzq'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 192, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  132996579828240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132996579821712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132996579821136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132996579821328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132996579827664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132996579820944: TensorSpec(shape=(), dtype=tf.resource, name=

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



B1 int8: size 517 KB, mean abs diff 0.0046, corr 0.9994, agreement 0.9762, AUC float 0.847 int8 0.847
RESULT: PASS (bar: agreement 0.95, AUC drop at most 0.02)
for reference, A1: agreement 0.79 to 0.81, AUC 0.901 to 0.81 to 0.85


In [ ]:
# ============================================================
# B1 CROSS VALIDATION: 5 folds x 3 seeds, same recipe and folds as A1. No test data.
# self contained, resumable
# ============================================================
import os, json, time, random, gc, zipfile, datetime
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.metrics import roc_auc_score, roc_curve

print("GPUs:", tf.config.list_physical_devices('GPU'))
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
PROJ='/content/drive/MyDrive/pulmoscan semester 7'; WORK='/content/work'
RES=os.path.join(PROJ,'results'); OOF=os.path.join(RES,'oof_probs'); BP=os.path.join(RES,'bn_order_protocol.json')
N_MELS=128; T=192; EPOCHS=40; BATCH=32; LR=1e-3; LR_MIN=1e-6; DIAG_W=0.1; NS=4; ND=7

d=json.load(open(BP))
if 'step2_started' not in d:
    d['step2_started']={'written_at':datetime.datetime.now().isoformat(),
        'design':'B1 5 fold patient grouped CV, 3 seeds, same folds and recipe as A1, probabilities saved as B1_f{k}_s{seed}.npz',
        'rule':'adopt B1 only if mean fold binary AUC (3 seed mean per fold) >= A1 mean 0.721 minus 0.01',
        'threshold':'if adopted, from B1 pooled out of fold scores at Sp 0.70, mean over 3 seeds'}
    json.dump(d,open(BP,'w'),indent=2); print('step 2 recorded')

# ---------- data ----------
os.makedirs(WORK,exist_ok=True)
if not os.path.isdir(os.path.join(WORK,'feat','processed','train')):
    with zipfile.ZipFile(os.path.join(PROJ,'data','pulmoscan_data.zip')) as z:
        z.extractall(os.path.join(WORK,'feat'))
with zipfile.ZipFile(os.path.join(PROJ,'data','colab_training_package.zip')) as z:
    z.extract('data/processed/manifest.csv', WORK)
man=pd.read_csv(os.path.join(WORK,'data/processed/manifest.csv'))
dev=man[man.split.isin(['train','val'])].copy().reset_index(drop=True)
dev['fp']=dev.apply(lambda r: os.path.join(WORK,'feat','processed',r['split'],
                    r['audio_file'].replace('.wav','.npy')),axis=1)
dev=dev[dev.fp.map(os.path.exists)].reset_index(drop=True)
assert 'test' not in set(dev.split)
X=np.empty((len(dev),N_MELS,T,1),np.float32)
for i in range(len(dev)):
    a=np.load(dev.fp[i]); X[i,...,0]=np.pad(a,[(0,0),(0,T-a.shape[1])]) if a.shape[1]<T else a[:,:T]
ys=dev.sound_label.values.astype(np.int32); yd=dev.diagnosis_label.values.astype(np.int32); pid=dev.patient_id.values
print(f"{len(dev)} rows, {dev.patient_id.nunique()} patients loaded")      # expect 3690 and 82

rng=np.random.RandomState(42)
pdf=dev.groupby('patient_id')['diagnosis_label'].agg(lambda x:x.mode()[0]).reset_index()
folds=[[] for _ in range(5)]
for _,g in pdf.groupby('diagnosis_label'):
    ps=g.patient_id.values.copy(); rng.shuffle(ps)
    for j,p in enumerate(ps): folds[j%5].append(p)

# ---------- loss, B1 model, augmentation (same as the final training cell) ----------
def focal(g=2.0):
    def fn(yt,yp):
        yp=tf.clip_by_value(yp,1e-7,1.0); ce=-yt*tf.math.log(yp)
        p=tf.reduce_sum(yt*yp,axis=-1,keepdims=True)
        return tf.reduce_mean(tf.pow(1.0-p,g)*ce)
    fn.__name__='focal'; return fn
def build_b1(T):
    L=tf.keras.layers; inp=tf.keras.Input(shape=(N_MELS,T,1)); x=inp
    for f in [32,64,128]:
        x=L.Conv2D(f,3,padding='same')(x); x=L.BatchNormalization()(x)
        x=L.Activation('relu')(x); x=L.MaxPooling2D(2)(x)
    x=L.Conv2D(256,3,padding='same')(x); x=L.BatchNormalization()(x); x=L.Activation('relu')(x)
    x=L.GlobalAveragePooling2D()(x)
    sh=L.Dropout(0.5)(L.Dense(256,activation='relu')(x))
    s=L.Dropout(0.3)(L.Dense(128,activation='relu')(sh))
    so=L.Dense(NS,activation='softmax',name='sound')(s)
    dd=L.Dropout(0.3)(L.Dense(128,activation='relu')(sh))
    do=L.Dense(ND,activation='softmax',name='diagnosis')(dd)
    return tf.keras.Model(inp,[so,do])
def augment(x,y):
    B=tf.shape(x)[0]; F=tf.shape(x)[1]; Tt=tf.shape(x)[2]
    real=tf.cast(tf.not_equal(x,0.0),tf.float32)
    x=x+tf.random.normal([B,1,1,1],0.0,0.15)*real
    fw=tf.random.uniform([B,1,1,1],0,21,dtype=tf.int32)
    f0=tf.random.uniform([B,1,1,1],0,1000000,dtype=tf.int32)%tf.maximum(F-fw,1)
    fi=tf.reshape(tf.range(F),[1,-1,1,1]); fmask=tf.logical_and(fi>=f0,fi<f0+fw)
    tw=tf.random.uniform([B,1,1,1],0,31,dtype=tf.int32)
    t0=tf.random.uniform([B,1,1,1],0,1000000,dtype=tf.int32)%tf.maximum(Tt-tw,1)
    ti=tf.reshape(tf.range(Tt),[1,1,-1,1]); tmask=tf.logical_and(ti>=t0,ti<t0+tw)
    return tf.where(tf.logical_or(fmask,tmask),tf.zeros_like(x),x),y

# ---------- runs ----------
def run_b(k,seed):
    path=os.path.join(OOF,f'B1_f{k}_s{seed}.npz')
    if os.path.exists(path): print('  SKIP',os.path.basename(path)); return
    te=np.isin(pid,folds[k]); tr=~te
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed); tf.keras.backend.clear_session()
    ds=tf.data.Dataset.from_tensor_slices((X[tr],{'sound':tf.keras.utils.to_categorical(ys[tr],NS),
                                                   'diagnosis':tf.keras.utils.to_categorical(yd[tr],ND)}))
    ds=ds.shuffle(4096,seed=seed).batch(BATCH).map(augment,num_parallel_calls=tf.data.AUTOTUNE).prefetch(2)
    m=build_b1(T); m.compile(optimizer=tf.keras.optimizers.Adam(LR),
        loss={'sound':focal(2.0),'diagnosis':focal(2.0)},
        loss_weights={'sound':1.0,'diagnosis':DIAG_W},metrics={'sound':'accuracy'})
    t0=time.time()
    m.fit(ds,epochs=EPOCHS,verbose=0,callbacks=[tf.keras.callbacks.LearningRateScheduler(
        lambda e: float(LR_MIN+0.5*(LR-LR_MIN)*(1+np.cos(np.pi*e/EPOCHS))))])
    pr=m.predict(X[te],verbose=0)[0]
    auc=roc_auc_score(ys[te]>0,1-pr[:,0])
    print(f'  B1_f{k}_s{seed}: AUC {auc:.3f}  {(time.time()-t0)/60:.1f}min')
    np.savez_compressed(path,p=pr.astype(np.float32),y=ys[te],pid=pid[te])
    del m; gc.collect()
for k in range(5):
    for s in [0,1,2]: run_b(k,s)

# ---------- comparison with A1 (rule fixed in step 1) ----------
def load(pre,k,s): return np.load(os.path.join(OOF,f'{pre}_f{k}_s{s}.npz'))
def auc_of(z): return roc_auc_score(z['y']>0,1-z['p'][:,0])
print('\nfold   A1 AUC   B1 AUC   diff   (3 seed means)')
a1=[];b1=[]
for k in range(5):
    a=np.mean([auc_of(load('A1',k,s)) for s in range(3)]); b=np.mean([auc_of(load('B1',k,s)) for s in range(3)])
    a1.append(a); b1.append(b); print(f'  {k}    {a:.3f}    {b:.3f}   {b-a:+.3f}')
ma,mb=float(np.mean(a1)),float(np.mean(b1))
print(f'\nmean fold AUC: A1 {ma:.3f}   B1 {mb:.3f}   B1 needs at least {ma-0.01:.3f}')
verdict='ADOPT B1' if mb>=ma-0.01 else 'KEEP FLOAT A1 for deployment'
print('VERDICT:',verdict)

def pooled(pre,s):
    y=np.concatenate([load(pre,k,s)['y'] for k in range(5)])>0
    sc=np.concatenate([1-load(pre,k,s)['p'][:,0] for k in range(5)]); return y,sc
print('\nB1 pooled out of fold (mean of 3 seeds)')
print('  pooled AUC:',round(float(np.mean([roc_auc_score(*pooled('B1',s)) for s in range(3)])),3),
      ' (A1 reference 0.715)')
TG=[0.60,0.70,0.80]; thr={}
for t in TG:
    r=[]
    for s in range(3):
        y,sc=pooled('B1',s); th=np.quantile(sc[~y],t); r.append((th,(sc[y]>th).mean(),(sc[~y]<=th).mean()))
    r=np.mean(r,axis=0); thr[t]=float(r[0])
    print(f'  Sp target {t:.2f}: threshold {r[0]:.3f}  Se {r[1]:.2f}  Sp {r[2]:.2f}')
tb=thr[0.70]; print(f'\nB1 fixed threshold {tb:.3f} per fold (mean of 3 seeds):')
for k in range(5):
    se=[];sp=[]
    for s in range(3):
        z=load('B1',k,s); yy=z['y']>0; ss=1-z['p'][:,0]
        se.append((ss[yy]>tb).mean()); sp.append((ss[~yy]<=tb).mean())
    print(f'  fold {k}: Se {np.mean(se):.2f}  Sp {np.mean(sp):.2f}')
d=json.load(open(BP))
d['step2_result']={'written_at':datetime.datetime.now().isoformat(),'mean_fold_auc_A1':ma,'mean_fold_auc_B1':mb,
                   'verdict':verdict,'threshold_Sp070':tb}
json.dump(d,open(BP,'w'),indent=2)

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
step 2 recorded
3690 rows, 82 patients loaded
  B1_f0_s0: AUC 0.704  2.8min
  B1_f0_s1: AUC 0.731  2.6min
  B1_f0_s2: AUC 0.709  2.6min
  B1_f1_s0: AUC 0.648  3.1min
  B1_f1_s1: AUC 0.717  3.1min
  B1_f1_s2: AUC 0.740  3.0min
  B1_f2_s0: AUC 0.724  2.9min
  B1_f2_s1: AUC 0.738  2.9min
  B1_f2_s2: AUC 0.711  2.9min
  B1_f3_s0: AUC 0.806  2.9min
  B1_f3_s1: AUC 0.773  2.9min
  B1_f3_s2: AUC 0.776  2.8min
  B1_f4_s0: AUC 0.736  2.9min
  B1_f4_s1: AUC 0.760  2.7min
  B1_f4_s2: AUC 0.709  2.8min

fold   A1 AUC   B1 AUC   diff   (3 seed means)
  0    0.735    0.715   -0.020
  1    0.733    0.701   -0.032
  2    0.656    0.724   +0.068
  3    0.749    0.785   +0.036
  4    0.733    0.735   +0.001

mean fold AUC: A1 0.721   B1 0.732   B1 needs at least 0.711
VERDICT: ADOPT B1

B1 pooled out of fold (mean of 3 seeds)
  pooled AUC: 0.713  (A1 reference 0.715)
  Sp target 0.60: threshold 0.479  Se 0.72  Sp 0.60
  Sp target 0

In [ ]:
# ============================================================
# B1 int8 DEPLOYMENT CHECK at threshold 0.505 + firmware constants. No test data. CPU is fine.
# ============================================================
import os, json, zipfile
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.metrics import roc_auc_score

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
PROJ='/content/drive/MyDrive/pulmoscan semester 7'; WORK='/content/work'
CK=os.path.join(PROJ,'checkpoints'); TF_DIR=os.path.join(PROJ,'tflite'); RES=os.path.join(PROJ,'results')
N_MELS=128; T=192; THRESH=0.505

os.makedirs(WORK,exist_ok=True)
if not os.path.isdir(os.path.join(WORK,'feat','processed','train')):
    with zipfile.ZipFile(os.path.join(PROJ,'data','pulmoscan_data.zip')) as z:
        z.extractall(os.path.join(WORK,'feat'))
with zipfile.ZipFile(os.path.join(PROJ,'data','colab_training_package.zip')) as z:
    z.extract('data/processed/manifest.csv', WORK)
man=pd.read_csv(os.path.join(WORK,'data/processed/manifest.csv'))
dev=man[man.split.isin(['train','val'])].copy().reset_index(drop=True)
dev['fp']=dev.apply(lambda r: os.path.join(WORK,'feat','processed',r['split'],
                    r['audio_file'].replace('.wav','.npy')),axis=1)
dev=dev[dev.fp.map(os.path.exists)].reset_index(drop=True)
assert 'test' not in set(dev.split)
X=np.empty((len(dev),N_MELS,T,1),np.float32)
for i in range(len(dev)):
    a=np.load(dev.fp[i]); X[i,...,0]=np.pad(a,[(0,0),(0,T-a.shape[1])]) if a.shape[1]<T else a[:,:T]
yb=dev.sound_label.values>0
print("dev rows:",len(dev))                                            # expect 3690

full=tf.keras.models.load_model(os.path.join(CK,'final_B1_s0.keras'),compile=False)
model=tf.keras.Model(full.input,full.outputs[0])
rng=np.random.RandomState(0); _=rng.choice(len(dev),300,replace=False); ev=rng.choice(len(dev),800,replace=False)
pf=1-model.predict(X[ev],batch_size=64,verbose=0)[:,0]

tp=os.path.join(TF_DIR,'pulmoscan_B1_s0_int8.tflite'); tfl=open(tp,'rb').read()
itp=tf.lite.Interpreter(model_content=tfl); itp.allocate_tensors()
inp=itp.get_input_details()[0]; out=itp.get_output_details()[0]
si,zi=inp['quantization']; so,zo=out['quantization']; pq=np.empty(len(ev)); q0=np.empty(len(ev))
for j,i in enumerate(ev):
    q=np.clip(np.round(X[i:i+1]/si+zi),-128,127).astype(np.int8)
    itp.set_tensor(inp['index'],q); itp.invoke()
    raw=itp.get_tensor(out['index'])[0]; q0[j]=raw[0]; pq[j]=1-(float(raw[0])-zo)*so
agree=float(np.mean((pf>THRESH)==(pq>THRESH)))
cut=(1-THRESH)/so+zo            # flag abnormal when int8 output[0] (Normal) < cut
agree_int=float(np.mean((q0<cut)==(pf>THRESH)))
print(f"\nat threshold {THRESH}: float vs int8 agreement {agree:.4f}  |  agreement using the integer cut {agree_int:.4f}")
print(f"AUC float {roc_auc_score(yb[ev],pf):.3f}   int8 {roc_auc_score(yb[ev],pq):.3f}")
print(f"flagged abnormal on these rows: float {np.mean(pf>THRESH):.2f}  int8 {np.mean(pq>THRESH):.2f}")

print("\n=== constants for the firmware ===")
print(f"input : int8 {tuple(inp['shape'])}  scale {si:.6f}  zero_point {int(zi)}   q = round(x/scale)+zero_point, clip to [-128,127]")
print(f"output: int8 {tuple(out['shape'])}  scale {so:.6f}  zero_point {int(zo)}   index 0 = Normal")
print(f"decision: abnormal if output[0] < {cut:.2f}   (equivalent to P(abnormal) > {THRESH})")

ops=sorted({d['op_name'] for d in itp._get_ops_details()})
print("\noperators the TFLM resolver must register:",ops)
big=sorted(itp.get_tensor_details(),key=lambda d:-int(np.prod(d['shape'])))[:4]
print("largest tensors:",[(str(d['shape']),f"{int(np.prod(d['shape']))/1e3:.0f} KB") for d in big])

# C header with the model bytes
hdr=os.path.join(TF_DIR,'pulmoscan_model_data.h')
with open(hdr,'w') as f:
    f.write('// PulmoScan B1 int8, generated from pulmoscan_B1_s0_int8.tflite\n#include <stdint.h>\n')
    f.write('alignas(16) const unsigned char pulmoscan_model[] = {\n')
    for k in range(0,len(tfl),24): f.write(', '.join(str(b) for b in tfl[k:k+24])+',\n')
    f.write(f'}};\nconst unsigned int pulmoscan_model_len = {len(tfl)};\n')
json.dump({'input_scale':float(si),'input_zero_point':int(zi),'output_scale':float(so),'output_zero_point':int(zo),
           'threshold_p_abnormal':THRESH,'abnormal_if_output0_below':float(cut),'ops':ops,'size_bytes':len(tfl)},
          open(os.path.join(TF_DIR,'firmware_constants.json'),'w'),indent=2)
print(f"\nwritten: {hdr}  ({os.path.getsize(hdr)/1e6:.1f} MB) and firmware_constants.json")

Mounted at /content/drive
dev rows: 3690


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



at threshold 0.505: float vs int8 agreement 0.9850  |  agreement using the integer cut 0.9850
AUC float 0.847   int8 0.847
flagged abnormal on these rows: float 0.36  int8 0.36

=== constants for the firmware ===
input : int8 (np.int32(1), np.int32(128), np.int32(192), np.int32(1))  scale 0.048760  zero_point 0   q = round(x/scale)+zero_point, clip to [-128,127]
output: int8 (np.int32(1), np.int32(4))  scale 0.003906  zero_point -128   index 0 = Normal
decision: abnormal if output[0] < -1.28   (equivalent to P(abnormal) > 0.505)

operators the TFLM resolver must register: ['CONV_2D', 'DELEGATE', 'FULLY_CONNECTED', 'MAX_POOL_2D', 'MEAN', 'SOFTMAX']
largest tensors: [('[  1 128 192  32]', '786 KB'), ('[ 1 64 96 64]', '393 KB'), ('[256   3   3 128]', '295 KB'), ('[ 1 64 96 32]', '197 KB')]

written: /content/drive/MyDrive/pulmoscan semester 7/tflite/pulmoscan_model_data.h  (2.3 MB) and firmware_constants.json


In [ ]:
import zipfile, os, re
PROJ='/content/drive/MyDrive/pulmoscan semester 7'
keys=['n_fft','hop_length','n_mels','win_length','sr=','sample_rate','target_sr','fmin','fmax','power_to_db',
      'amplitude_to_db','normalize','.mean(','.std(','resample','duration','butter','filtfilt','highpass','bandpass','MAX_FRAMES','np.pad']
with zipfile.ZipFile(os.path.join(PROJ,'data','colab_training_package.zip')) as z:
    pys=[n for n in z.namelist() if n.endswith('.py')]
    print("python files in the package:",pys)
    for n in pys:
        txt=z.read(n).decode('utf8','ignore').splitlines()
        hits=[(i+1,l.strip()) for i,l in enumerate(txt) if any(k in l for k in keys)]
        if hits:
            print(f"\n-- {n}")
            for i,l in hits[:40]: print(f"  {i:4d}: {l[:150]}")

python files in the package: ['scripts/train_multitask.py', 'src/augmentation.py', 'src/data_loader.py', 'src/evaluation.py', 'src/export_tflite.py', 'src/hf_lung_labels.py', 'src/hf_lung_split.py', 'src/models.py', 'src/multitask_model.py', 'src/preprocessing.py', 'src/synthetic_generator.py', 'src/training.py', 'src/__init__.py']

-- scripts/train_multitask.py
    40: feat = np.pad(feat, [(0,0), (0, t-c)])
    67: return (np.mean(se)+np.mean(sp))/2.0

-- src/augmentation.py
    49: return librosa.effects.pitch_shift(audio, sr=sr, n_steps=n_steps)
    65: signal_power = np.mean(audio ** 2)

-- src/export_tflite.py
   163: float_acc = np.mean(y_pred == y_test)

-- src/hf_lung_labels.py
    29: WINDOW_DUR = 1.0          # seconds — derived to match TARGET_FRAMES=63 @ hop_length=256, SR=16000

-- src/preprocessing.py
    12: from scipy.signal import butter, filtfilt
    15: SR           = 16000   # resample everything to 16kHz (ICBHI has mixed rates)
    19: # Butterworth bandpass (lung 

In [ ]:
import zipfile, os, re
import numpy as np, pandas as pd, tensorflow as tf
PROJ='/content/drive/MyDrive/pulmoscan semester 7'
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')

with zipfile.ZipFile(os.path.join(PROJ,'data','colab_training_package.zip')) as z:
    src=z.read('src/preprocessing.py').decode('utf8','ignore').splitlines()
    z.extract('data/processed/manifest.csv','/content/work')
print("=== preprocessing.py lines 1 to 60 ===")
for i,l in enumerate(src[:60],1): print(f"{i:3d}: {l[:140]}")
print("\n=== lines 94 to 113 (mel) and 196 to 215 (frames) ===")
for i in list(range(94,114))+list(range(196,216)):
    if i<=len(src): print(f"{i:3d}: {src[i-1][:140]}")

man=pd.read_csv('/content/work/data/processed/manifest.csv')
pat=re.compile(r'^\d+_\w+_\w+_\w+_\w+_cycle\d+_[a-z]+\.wav$',re.I)
bad=man[~man.audio_file.map(lambda f: bool(pat.match(f)))]
print(f"\nmanifest rows {len(man)}; rows not matching the ICBHI cycle name pattern: {len(bad)}")
if len(bad): print(bad.audio_file.head(5).tolist())

tfl=open(os.path.join(PROJ,'tflite','pulmoscan_B1_s0_int8.tflite'),'rb').read()
try:
    it=tf.lite.Interpreter(model_content=tfl,
        experimental_op_resolver_type=tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES)
    it.allocate_tensors()
    print("\ntrue operators in the model file:",sorted({d['op_name'] for d in it._get_ops_details()}))
except Exception as e:
    print("\nop listing without delegates failed:",str(e)[:150])

=== preprocessing.py lines 1 to 60 ===
  1: """
  2: preprocessing.py
  3: ----------------
  4: Audio preprocessing: Butterworth filter, mel-spectrogram, MFCC extraction.
  5: Features are cached to disk as .npy files to avoid re-computation.
  6: """
  7: 
  8: import os
  9: import numpy as np
 10: import librosa
 11: import pandas as pd
 12: from scipy.signal import butter, filtfilt
 13: from tqdm import tqdm
 14: 
 15: SR           = 16000   # resample everything to 16kHz (ICBHI has mixed rates)
 16: MIN_DURATION = 2.0     # pad cycles shorter than this (seconds)
 17: MAX_DURATION = 8.0     # truncate cycles longer than this (seconds)
 18: 
 19: # Butterworth bandpass (lung sounds are 100–2000 Hz)
 20: LOW_CUT      = 100
 21: HIGH_CUT     = 2000
 22: FILTER_ORDER = 4
 23: 
 24: # Mel spectrogram
 25: # hop_length=256 (was 512) — doubles temporal resolution so crackle bursts
 26: # (5–100ms) span 3+ frames instead of 1, giving the model more signal to learn from
 27: N_FFT      = 2

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [ ]:
# ============================================================
# FIRMWARE REFERENCE EXPORT + GOLDEN VECTORS (CPU is fine, no test data)
# ============================================================
import os, io, json, zipfile, inspect
import numpy as np, pandas as pd, tensorflow as tf, librosa, scipy
from scipy.signal import butter, filtfilt, sosfilt, sosfiltfilt, get_window

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
PROJ='/content/drive/MyDrive/pulmoscan semester 7'
OUT=os.path.join(PROJ,'tflite','firmware_ref'); os.makedirs(OUT,exist_ok=True)
SR=16000; LOW=100; HIGH=2000; ORDER=4; NFFT=2048; HOP=256; NMELS=128; FMIN=50; FMAX=2000; FRAMES=192
N=(FRAMES-1)*HOP                                   # 48896 samples gives exactly 192 frames
nyq=0.5*SR
SOS=butter(ORDER,[LOW/nyq,HIGH/nyq],btype='band',output='sos')

print("librosa",librosa.__version__,"scipy",scipy.__version__)
p=inspect.signature(librosa.feature.melspectrogram).parameters
print("melspectrogram defaults: window",p['window'].default,"center",p['center'].default,"pad_mode",p['pad_mode'].default)
p2=inspect.signature(librosa.power_to_db).parameters
print("power_to_db defaults: amin",p2['amin'].default,"top_db",p2['top_db'].default)

# ---------- deterministic test signal, stored as int16 so C sees identical samples ----------
rng=np.random.RandomState(0); t=np.arange(N)/SR
x=rng.randn(N)*0.02
x+=0.05*np.sin(2*np.pi*400*t)*((t>0.8)&(t<1.8))
for pos in rng.randint(2000,N-2000,12):
    n=np.arange(160); x[pos:pos+160]+=0.15*np.exp(-n/30.0)*np.sin(2*np.pi*rng.uniform(600,1200)*n/SR)
a16=np.clip(np.round(x*32768),-32768,32767).astype(np.int16)
audio=(a16.astype(np.float32)/32768.0)

def feats(a,mode):
    if mode=='filtfilt':
        b,aa=butter(ORDER,[LOW/nyq,HIGH/nyq],btype='band'); y=filtfilt(b,aa,a)
    elif mode=='sosfiltfilt': y=sosfiltfilt(SOS,a)
    else: y=sosfilt(SOS,a)
    mel=librosa.feature.melspectrogram(y=y,sr=SR,n_fft=NFFT,hop_length=HOP,n_mels=NMELS,fmin=FMIN,fmax=FMAX,power=2.0)
    db=librosa.power_to_db(mel,ref=np.max)
    return ((db-db.mean())/(db.std()+1e-8)).astype(np.float32), y

# ---------- int8 model with reference kernels ----------
tfl=open(os.path.join(PROJ,'tflite','pulmoscan_B1_s0_int8.tflite'),'rb').read()
try:
    itp=tf.lite.Interpreter(model_content=tfl,
        experimental_op_resolver_type=tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES)
except Exception:
    itp=tf.lite.Interpreter(model_content=tfl)
itp.allocate_tensors()
inp=itp.get_input_details()[0]; out=itp.get_output_details()[0]; si,zi=inp['quantization']
def run(z):
    q=np.clip(np.round(z[None,:,:,None]/si+zi),-128,127).astype(np.int8)
    itp.set_tensor(inp['index'],q); itp.invoke()
    return q[0,:,:,0], itp.get_tensor(out['index'])[0].copy()
def verdict(o): return "ABNORMAL" if o[0]<-1.28 else "normal"

z0,y0=feats(audio,'filtfilt'); assert z0.shape==(NMELS,FRAMES), z0.shape
q0,o0=run(z0)
print(f"\nGOLDEN (scipy filtfilt): output int8 {o0.tolist()}  -> {verdict(o0)}   P(abnormal) {1-(float(o0[0])+128)/256:.3f}")
print("firmware filter options, difference from golden:")
for mode in ['sosfiltfilt','sosfilt']:
    z,_=feats(audio,mode); _,o=run(z)
    print(f"  {mode:12s}: max |z diff| {np.max(np.abs(z-z0)):.3f}  mean |z diff| {np.mean(np.abs(z-z0)):.4f}  output {o.tolist()}  -> {verdict(o)}")

# ---------- real feature examples (model only, no feature extraction) ----------
with zipfile.ZipFile(os.path.join(PROJ,'data','colab_training_package.zip')) as z:
    man=pd.read_csv(io.BytesIO(z.read('data/processed/manifest.csv')))
dev=man[man.split.isin(['train','val'])].reset_index(drop=True)
real={}
with zipfile.ZipFile(os.path.join(PROJ,'data','pulmoscan_data.zip')) as z:
    names={os.path.basename(n):n for n in z.namelist() if n.endswith('.npy')}
    for lab,tag in [(0,'normal'),(1,'crackle')]:
        r=dev[dev.sound_label==lab].iloc[0]; f=r.audio_file.replace('.wav','.npy')
        a=np.load(io.BytesIO(z.read(names[f])))
        a=np.pad(a,[(0,0),(0,FRAMES-a.shape[1])]) if a.shape[1]<FRAMES else a[:,:FRAMES]
        qq,oo=run(a.astype(np.float32)); real[tag]=(qq,oo)
        print(f"real {tag:8s} ({f[:34]}): output {oo.tolist()} -> {verdict(oo)}")

# ---------- firmware tables ----------
fb=librosa.filters.mel(sr=SR,n_fft=NFFT,n_mels=NMELS,fmin=FMIN,fmax=FMAX).astype(np.float32)
starts=[];lens=[];w=[]
for r in fb:
    nz=np.nonzero(r)[0]
    if len(nz)==0: starts.append(0); lens.append(0); continue
    s,e=int(nz[0]),int(nz[-1])+1; starts.append(s); lens.append(e-s); w.extend(r[s:e].tolist())
print(f"\nmel filterbank: {fb.shape}, empty filters {sum(l==0 for l in lens)}, stored weights {len(w)} (full table would be {fb.size})")

def arr(name,ctype,vals,fmt):
    return f"const {ctype} {name}[{len(vals)}] = {{\n"+",\n".join(", ".join(fmt.format(v) for v in vals[i:i+8]) for i in range(0,len(vals),8))+"\n};\n"
with open(os.path.join(OUT,'mel_and_filter_tables.h'),'w') as f:
    f.write(f"// generated: sr {SR}, n_fft {NFFT}, hop {HOP}, {NMELS} mels, fmin {FMIN}, fmax {FMAX}, Slaney mel and norm\n")
    f.write("// Hann window is periodic: w[n] = 0.5 - 0.5*cos(2*pi*n/2048)\n#include <stdint.h>\n")
    f.write(arr('bp_sos','float',SOS.flatten().tolist(),'{:.9e}f')+"// 4 sections x [b0 b1 b2 a0 a1 a2]\n")
    f.write(arr('mel_start','uint16_t',starts,'{}')); f.write(arr('mel_len','uint16_t',lens,'{}'))
    f.write(arr('mel_weights','float',w,'{:.9e}f'))
st=lambda v,k=8: [float(x) for x in np.asarray(v).flatten()[:k]]
with open(os.path.join(OUT,'golden_test_data.h'),'w') as f:
    f.write(f"#include <stdint.h>\n#define GOLDEN_N {N}\n")
    f.write(arr('golden_audio','int16_t',a16.tolist(),'{}'))
    f.write(arr('golden_output_int8','int8_t',[int(v) for v in o0],'{}'))
    f.write(arr('golden_filtered_1000_1007','float',st(y0[1000:1008]),'{:.7e}f'))
    for row in [0,64,127]: f.write(arr(f'golden_z_row{row}_first6','float',st(z0[row,:6],6),'{:.6e}f'))
np.savez(os.path.join(OUT,'golden.npz'),audio_int16=a16,z=z0,q_input=q0,out=o0,
         real_normal_q=real['normal'][0],real_normal_out=real['normal'][1],
         real_crackle_q=real['crackle'][0],real_crackle_out=real['crackle'][1])
json.dump({'sr':SR,'n_fft':NFFT,'hop':HOP,'n_mels':NMELS,'fmin':FMIN,'fmax':FMAX,'frames':FRAMES,'samples':N,
           'input_scale':float(si),'input_zero_point':int(zi),'abnormal_if_output0_below':-1.28},
          open(os.path.join(OUT,'pipeline_spec.json'),'w'),indent=2)
print("\nfiles in",OUT)
for fn in sorted(os.listdir(OUT)): print(f"  {fn}  {os.path.getsize(os.path.join(OUT,fn))/1e3:.0f} KB")
print("\nfirst filtered samples (golden):",np.round(y0[1000:1004],6).tolist())
print("golden z row0 first 4:",np.round(z0[0,:4],4).tolist())

librosa 0.11.0 scipy 1.16.3
melspectrogram defaults: window hann center True pad_mode constant
power_to_db defaults: amin 1e-10 top_db 80.0

GOLDEN (scipy filtfilt): output int8 [52, -118, -62, -128]  -> normal   P(abnormal) 0.297
firmware filter options, difference from golden:


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


  sosfiltfilt : max |z diff| 0.000  mean |z diff| 0.0000  output [52, -118, -62, -128]  -> normal
  sosfilt     : max |z diff| 6.337  mean |z diff| 0.1398  output [27, -126, -29, -128]  -> normal
real normal   (102_1b1_Ar_sc_Meditron_cycle0_norm): output [17, -30, -118, -126] -> normal
real crackle  (104_1b1_Ll_sc_Litt3200_cycle4_crac): output [-5, -114, -18, -119] -> ABNORMAL

mel filterbank: (128, 1025), empty filters 0, stored weights 493 (full table would be 131200)

files in /content/drive/MyDrive/pulmoscan semester 7/tflite/firmware_ref
  golden.npz  272 KB
  golden_test_data.h  278 KB
  mel_and_filter_tables.h  11 KB
  pipeline_spec.json  0 KB

first filtered samples (golden): [-0.002596, 0.001068, 0.003329, 0.003031]
golden z row0 first 4: [0.30219998955726624, 0.08420000225305557, -0.6899999976158142, -2.4274001121520996]


In [ ]:
# ============================================================
# EMULATE THE FIRMWARE ALGORITHM from the exported header tables (CPU, no test data)
# ============================================================
import os, re, numpy as np, tensorflow as tf, librosa
from scipy.signal import sosfilt, sosfiltfilt

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
PROJ='/content/drive/MyDrive/pulmoscan semester 7'; REF=os.path.join(PROJ,'tflite','firmware_ref')
SR=16000; NFFT=2048; HOP=256; NM=128; FR=192; N=(FR-1)*HOP
g=np.load(os.path.join(REF,'golden.npz')); a16=g['audio_int16']; z_gold=g['z']; q_gold=g['q_input']; o_gold=g['out']
txt=open(os.path.join(REF,'mel_and_filter_tables.h')).read()
def arr(name):
    m=re.search(rf'{name}\[\d+\]\s*=\s*\{{(.*?)\}};',txt,re.S)
    return np.array([float(v.strip().rstrip('f')) for v in m.group(1).replace('\n',' ').split(',') if v.strip()])
sos=arr('bp_sos').astype(np.float32).reshape(-1,6); starts=arr('mel_start').astype(int); lens=arr('mel_len').astype(int); wts=arr('mel_weights')
print("tables parsed: sos",sos.shape,"filters",len(starts),"weights",len(wts))

tfl=open(os.path.join(PROJ,'tflite','pulmoscan_B1_s0_int8.tflite'),'rb').read()
itp=tf.lite.Interpreter(model_content=tfl); itp.allocate_tensors()
inp=itp.get_input_details()[0]; out=itp.get_output_details()[0]; si,zi=inp['quantization']
def model(z):
    q=np.clip(np.round(z[None,:,:,None]/si+zi),-128,127).astype(np.int8)
    itp.set_tensor(inp['index'],q); itp.invoke(); return q[0,:,:,0], itp.get_tensor(out['index'])[0].copy()
verdict=lambda o:"ABNORMAL" if o[0]<-1.28 else "normal"

def firmware_front_end(x, filt='forward_backward', pad='constant'):
    if filt=='forward_backward':
        y=sosfilt(sos,x); y=sosfilt(sos,y[::-1])[::-1]          # zero initial state, no edge padding
    else: y=sosfiltfilt(sos,x)                                   # scipy reference
    y=y.astype(np.float32)
    P=np.pad(y,NFFT//2,mode=pad)                                 # centred frames
    w=(0.5-0.5*np.cos(2*np.pi*np.arange(NFFT)/NFFT)).astype(np.float32)
    spec=np.empty((NFFT//2+1,FR),np.float32)
    for t in range(FR):
        spec[:,t]=np.abs(np.fft.rfft(P[t*HOP:t*HOP+NFFT]*w))**2
    mel=np.zeros((NM,FR)); pos=0
    for m in range(NM):
        s,L=starts[m],lens[m]; mel[m]=wts[pos:pos+L]@spec[s:s+L]; pos+=L
    S=np.maximum(mel,1e-10); db=10*np.log10(S)-10*np.log10(max(1e-10,S.max())); db=np.maximum(db,db.max()-80.0)
    return ((db-db.mean())/(db.std()+1e-8)).astype(np.float32), y

x=a16.astype(np.float32)/32768.0
print(f"\ngolden (scipy filtfilt, librosa): output {o_gold.tolist()} -> {verdict(o_gold)}")
for name,kw in [('firmware recipe, zero padded frames',dict(filt='forward_backward',pad='constant')),
                ('firmware recipe, reflect padded frames',dict(filt='forward_backward',pad='reflect')),
                ('scipy sosfiltfilt filter, zero padded frames',dict(filt='scipy',pad='constant'))]:
    z,y=firmware_front_end(x,**kw); q,o=model(z)
    print(f"\n{name}:")
    print(f"  mel z score vs golden: max |diff| {np.max(np.abs(z-z_gold)):.4f}  mean |diff| {np.mean(np.abs(z-z_gold)):.5f}")
    print(f"  int8 input cells different from golden: {int(np.sum(q!=q_gold))} of {q.size}")
    print(f"  model output {o.tolist()} -> {verdict(o)}   (golden {o_gold.tolist()})  max output diff {int(np.max(np.abs(o.astype(int)-o_gold.astype(int))))}")

Mounted at /content/drive
tables parsed: sos (4, 6) filters 128 weights 493

golden (scipy filtfilt, librosa): output [52, -118, -62, -128] -> normal

firmware recipe, zero padded frames:
  mel z score vs golden: max |diff| 8.1144  mean |diff| 0.01830
  int8 input cells different from golden: 4603 of 24576
  model output [51, -117, -63, -128] -> normal   (golden [52, -118, -62, -128])  max output diff 1

firmware recipe, reflect padded frames:
  mel z score vs golden: max |diff| 7.4144  mean |diff| 0.01935
  int8 input cells different from golden: 3377 of 24576
  model output [51, -117, -63, -128] -> normal   (golden [52, -118, -62, -128])  max output diff 1

scipy sosfiltfilt filter, zero padded frames:
  mel z score vs golden: max |diff| 0.0060  mean |diff| 0.00002
  int8 input cells different from golden: 7 of 24576
  model output [52, -118, -62, -128] -> normal   (golden [52, -118, -62, -128])  max output diff 0


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [ ]:
# ============================================================
# EDGE HANDLING TEST: plain forward backward vs margin, with microphone DC offset
# uses the variables from the previous cell; CPU is fine
# ============================================================
assert all(k in globals() for k in ['sos','starts','lens','wts','model','verdict','sosfilt','sosfiltfilt']), \
    "Session was reset: rerun the previous cell first, then this one."
M=4000

def mel_z(y):
    y=y.astype(np.float32); P=np.pad(y,NFFT//2,mode='constant')
    w=(0.5-0.5*np.cos(2*np.pi*np.arange(NFFT)/NFFT)).astype(np.float32)
    spec=np.empty((NFFT//2+1,FR),np.float32)
    for t in range(FR): spec[:,t]=np.abs(np.fft.rfft(P[t*HOP:t*HOP+NFFT]*w))**2
    mel=np.zeros((NM,FR)); pos=0
    for m in range(NM):
        s,L=starts[m],lens[m]; mel[m]=wts[pos:pos+L]@spec[s:s+L]; pos+=L
    S=np.maximum(mel,1e-10); db=10*np.log10(S)-10*np.log10(max(1e-10,S.max())); db=np.maximum(db,db.max()-80.0)
    return ((db-db.mean())/(db.std()+1e-8)).astype(np.float32)
fb=lambda x: sosfilt(sos,sosfilt(sos,x)[::-1])[::-1]

def make(L,seed):
    r=np.random.RandomState(seed); t=np.arange(L)/SR
    x=r.randn(L)*0.02+0.05*np.sin(2*np.pi*400*t)*((t>0.8)&(t<1.8))
    for pos in r.randint(2000,L-2000,14):
        n=np.arange(160); x[pos:pos+160]+=0.15*np.exp(-n/30.0)*np.sin(2*np.pi*r.uniform(600,1200)*n/SR)
    return x

print("DC offset | variant      | worst |z diff| | mean |z diff| | worst output diff | verdict flips (of 6)")
for dc in [0.0,0.005,0.02]:
    stats={'plain':[], 'margin':[]}
    for seed in range(6):
        xl=np.round((make(N+2*M,seed)+dc)*32768)/32768
        xw=xl[M:M+N]
        zr=mel_z(sosfiltfilt(sos,xw)); _,orf=model(zr)
        for name,y in [('plain',fb(xw)),('margin',fb(xl)[M:M+N])]:
            z=mel_z(y); _,o=model(z); d=np.abs(z-zr)
            stats[name].append((d.max(),d.mean(),int(np.max(np.abs(o.astype(int)-orf.astype(int)))),
                                verdict(o)!=verdict(orf),int(np.unravel_index(d.argmax(),d.shape)[1])))
    for name in ['plain','margin']:
        s=stats[name]
        print(f"  {dc:5.3f}   | {name:12s} | {max(v[0] for v in s):13.3f} | {np.mean([v[1] for v in s]):13.5f} | "
              f"{max(v[2] for v in s):17d} | {sum(v[3] for v in s)}   (worst frame index {s[int(np.argmax([v[0] for v in s]))][4]})")

DC offset | variant      | worst |z diff| | mean |z diff| | worst output diff | verdict flips (of 6)
  0.000   | plain        |         6.031 |       0.01175 |                 3 | 0   (worst frame index 191)
  0.000   | margin       |         5.363 |       0.01040 |                 4 | 0   (worst frame index 0)
  0.005   | plain        |         6.031 |       0.01116 |                 6 | 0   (worst frame index 191)
  0.005   | margin       |         5.365 |       0.01040 |                 6 | 0   (worst frame index 0)
  0.020   | plain        |         6.040 |       0.00991 |                 3 | 0   (worst frame index 191)
  0.020   | margin       |         5.361 |       0.01040 |                 3 | 0   (worst frame index 0)


In [ ]:
# ============================================================
# EXACT scipy filtfilt edge handling (odd extension 27 + steady state), float64 and float32
# ============================================================
assert all(k in globals() for k in ['sos','mel_z','model','verdict','make','M','N','sosfilt','sosfiltfilt','REF']), \
    "Session was reset: rerun the two previous cells first."
from scipy.signal import sosfilt_zi
PAD=27                                         # scipy default: 3*(2*4 sections + 1)
zi64=sosfilt_zi(sos)                           # (4,2)

def exact_filtfilt(x,dtype):
    s=sos.astype(dtype); zi=zi64.astype(dtype); x=x.astype(dtype)
    ext=np.concatenate([2*x[0]-x[PAD:0:-1], x, 2*x[-1]-x[-2:-PAD-2:-1]])
    y,_=sosfilt(s,ext,zi=zi*ext[0])
    y,_=sosfilt(s,y[::-1],zi=zi*y[-1])
    return y[::-1][PAD:-PAD]

r=np.random.RandomState(0); xt=(r.randn(N)*0.02).astype(np.float64)
ref=sosfiltfilt(sos,xt)
for dt in [np.float64,np.float32]:
    print(f"{dt.__name__}: max |filtered - scipy| {np.max(np.abs(exact_filtfilt(xt,dt)-ref)):.2e}  (signal scale {np.std(ref):.3f})")

print("\nDC offset | filter       | worst |z diff| | mean |z diff| | worst output diff | verdict flips (of 6)")
for dc in [0.0,0.005,0.02]:
    stats={'float64':[], 'float32':[]}
    for seed in range(6):
        xw=(np.round((make(N+2*M,seed)+dc)*32768)/32768)[M:M+N]
        zr=mel_z(sosfiltfilt(sos,xw)); _,orf=model(zr)
        for dt in [np.float64,np.float32]:
            z=mel_z(exact_filtfilt(xw,dt)); _,o=model(z); d=np.abs(z-zr)
            stats[dt.__name__].append((d.max(),d.mean(),int(np.max(np.abs(o.astype(int)-orf.astype(int)))),verdict(o)!=verdict(orf)))
    for name in ['float64','float32']:
        s=stats[name]
        print(f"  {dc:5.3f}   | {name:12s} | {max(v[0] for v in s):13.4f} | {np.mean([v[1] for v in s]):13.6f} | {max(v[2] for v in s):17d} | {sum(v[3] for v in s)}")

with open(os.path.join(REF,'filter_init.h'),'w') as f:
    f.write("// steady state start values for the 4 bandpass sections, 2 per section\n#define BP_PADLEN 27\n")
    f.write("const float bp_zi[8] = {"+", ".join(f"{v:.9e}f" for v in zi64.flatten())+"};\n")
print("\nwritten:",os.path.join(REF,'filter_init.h'))

float64: max |filtered - scipy| 0.00e+00  (signal scale 0.009)
float32: max |filtered - scipy| 2.88e-07  (signal scale 0.009)

DC offset | filter       | worst |z diff| | mean |z diff| | worst output diff | verdict flips (of 6)
  0.000   | float64      |        0.0000 |      0.000000 |                 0 | 0
  0.000   | float32      |        0.0608 |      0.000020 |                 0 | 0
  0.005   | float64      |        0.0000 |      0.000000 |                 0 | 0
  0.005   | float32      |        0.0153 |      0.000020 |                 0 | 0
  0.020   | float64      |        0.0000 |      0.000000 |                 0 | 0
  0.020   | float32      |        0.0209 |      0.000020 |                 4 | 0

written: /content/drive/MyDrive/pulmoscan semester 7/tflite/firmware_ref/filter_init.h


In [ ]:
import os, io, sys, zipfile, numpy as np, pandas as pd, librosa
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
PROJ='/content/drive/MyDrive/pulmoscan semester 7'; WORK='/content/work'; os.makedirs(WORK+'/src',exist_ok=True)
with zipfile.ZipFile(os.path.join(PROJ,'data','colab_training_package.zip')) as z:
    open(WORK+'/src/preprocessing.py','wb').write(z.read('src/preprocessing.py'))
    man=pd.read_csv(io.BytesIO(z.read('data/processed/manifest.csv')))
sys.path.insert(0,WORK+'/src'); import preprocessing as P
dev=man[man.split.isin(['train','val'])].reset_index(drop=True); pick=dev.sample(12,random_state=0)
zr=zipfile.ZipFile(os.path.join(PROJ,'data','pulmoscan_raw_audio.zip')); zf=zipfile.ZipFile(os.path.join(PROJ,'data','pulmoscan_data.zip'))
wavs={os.path.basename(n):n for n in zr.namelist() if n.endswith('.wav')}
npys={os.path.basename(n):n for n in zf.namelist() if n.endswith('.npy')}
print("cycle                                   cached shape  new shape   max|diff|  mean|diff|")
for r in pick.itertuples():
    y,_=librosa.load(io.BytesIO(zr.read(wavs[r.audio_file])),sr=P.SR,mono=True)
    f=P.extract_mel_spectrogram(P.normalise_duration(P.butterworth_filter(y)))
    c=np.load(io.BytesIO(zf.read(npys[r.audio_file.replace('.wav','.npy')])))
    same=c.shape==f.shape; d=np.abs(c-f) if same else None
    print(f"{r.audio_file[:38]:38s}  {str(c.shape):12s}  {str(f.shape):10s}  "+(f"{d.max():.5f}   {d.mean():.6f}" if same else "SHAPES DIFFER"))

Mounted at /content/drive
cycle                                   cached shape  new shape   max|diff|  mean|diff|
226_1b1_Al_sc_Meditron_cycle4_normal.w  (128, 135)    (128, 135)  0.00000   0.000000
120_1b1_Lr_sc_Meditron_cycle8_normal.w  (128, 126)    (128, 126)  0.00000   0.000000
156_5b3_Pr_mc_AKGC417L_cycle1_both.wav  (128, 136)    (128, 136)  0.00000   0.000000
120_1b1_Pr_sc_Meditron_cycle17_normal.  (128, 218)    (128, 218)  0.00000   0.000000
133_3p2_Al_mc_AKGC417L_cycle7_wheeze.w  (128, 158)    (128, 158)  0.00000   0.000000
138_1p3_Pr_mc_AKGC417L_cycle3_crackle.  (128, 259)    (128, 259)  0.00000   0.000000
114_1b4_Pr_mc_AKGC417L_cycle4_crackle.  (128, 191)    (128, 191)  0.00000   0.000000
138_2p2_Al_mc_AKGC417L_cycle3_crackle.  (128, 181)    (128, 181)  0.00000   0.000000
193_7b3_Ll_mc_AKGC417L_cycle6_normal.w  (128, 163)    (128, 163)  0.00000   0.000000
170_1b2_Al_mc_AKGC417L_cycle4_normal.w  (128, 192)    (128, 192)  0.00000   0.000000
118_1b1_Pl_sc_Litt3200_cycle1_normal

In [ ]:
import os, subprocess, glob
from google.colab import files, drive
drive.mount('/content/drive')
os.makedirs('/root/.kaggle', exist_ok=True)
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Upload kaggle.json when the file picker appears")
    up = files.upload()
    open('/root/.kaggle/kaggle.json','wb').write(up['kaggle.json'])
os.chmod('/root/.kaggle/kaggle.json', 0o600)
subprocess.run(['pip','install','-q','kaggle'], check=True)
DEST='/content/icbhi'; os.makedirs(DEST, exist_ok=True)
r = subprocess.run(['kaggle','datasets','download','-d','vbookshelf/respiratory-sound-database','-p',DEST,'--unzip'],
                   capture_output=True, text=True)
print((r.stdout+r.stderr)[-600:])
wav=glob.glob(DEST+'/**/*.wav',recursive=True); txt=glob.glob(DEST+'/**/*.txt',recursive=True)
print(len(wav),'wav files,',len(txt),'txt files,', glob.glob(DEST+'/**/patient_diagnosis.csv',recursive=True))

Mounted at /content/drive
Upload kaggle.json when the file picker appears


Saving kaggle.json to kaggle.json
 [00:29<00:01, 211MB/s]
 94%|█████████▍| 3.47G/3.69G [00:29<00:00, 235MB/s]
 95%|█████████▍| 3.50G/3.69G [00:30<00:00, 230MB/s]
 95%|█████████▌| 3.52G/3.69G [00:30<00:00, 206MB/s]
 96%|█████████▌| 3.54G/3.69G [00:30<00:00, 216MB/s]
 97%|█████████▋| 3.57G/3.69G [00:30<00:00, 239MB/s]
 98%|█████████▊| 3.59G/3.69G [00:31<00:01, 58.1MB/s]
 98%|█████████▊| 3.62G/3.69G [00:31<00:01, 73.7MB/s]
 99%|█████████▊| 3.64G/3.69G [00:31<00:00, 85.4MB/s]
 99%|█████████▉| 3.66G/3.69G [00:31<00:00, 107MB/s] 
100%|█████████▉| 3.68G/3.69G [00:32<00:00, 126MB/s]
100%|██████████| 3.69G/3.69G [00:32<00:00, 123MB/s]

1840 wav files, 1845 txt files, ['/content/icbhi/Respiratory_Sound_Database/Respiratory_Sound_Database/patient_diagnosis.csv', '/content/icbhi/respiratory_sound_database/Respiratory_Sound_Database/patient_diagnosis.csv']


In [ ]:
import os, glob, shutil, zipfile, subprocess
from google.colab import files, drive
drive.mount('/content/drive')
PROJ='/content/drive/MyDrive/pulmoscan semester 7'; ZIP=os.path.join(PROJ,'data','icbhi_kaggle.zip'); DEST='/content/icbhi'
os.makedirs(DEST,exist_ok=True)

if os.path.exists(ZIP):
    print("Using the copy on Drive, no Kaggle needed")
elif glob.glob(DEST+'/**/*.wav',recursive=True):
    print("Already downloaded this session, saving a zip to Drive...")
    with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_STORED) as z:          # wav files do not compress, so store only
        for p in glob.glob(DEST+'/**/*',recursive=True):
            if os.path.isfile(p): z.write(p, os.path.relpath(p,DEST))
else:
    os.makedirs('/root/.kaggle',exist_ok=True)
    if not os.path.exists('/root/.kaggle/kaggle.json'):
        print("Upload kaggle.json when the file picker appears")
        up=files.upload(); open('/root/.kaggle/kaggle.json','wb').write(up['kaggle.json'])
    os.chmod('/root/.kaggle/kaggle.json',0o600)
    subprocess.run(['pip','install','-q','kaggle'],check=True)
    r=subprocess.run(['kaggle','datasets','download','-d','vbookshelf/respiratory-sound-database','-p','/content/dl'],capture_output=True,text=True)
    print((r.stdout+r.stderr)[-400:])
    shutil.copy(glob.glob('/content/dl/*.zip')[0],ZIP)               # the secret kaggle.json is never saved to Drive

if not glob.glob(DEST+'/**/*.wav',recursive=True):
    with zipfile.ZipFile(ZIP) as z: z.extractall(DEST)
print(f"zip on Drive: {os.path.getsize(ZIP)/1e9:.2f} GB")
wav=glob.glob(DEST+'/**/*.wav',recursive=True); txt=glob.glob(DEST+'/**/*.txt',recursive=True)
print(len(wav),"wav,",len(txt),"txt,",glob.glob(DEST+'/**/patient_diagnosis.csv',recursive=True))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Already downloaded this session, saving a zip to Drive...
zip on Drive: 4.36 GB
1840 wav, 1845 txt, ['/content/icbhi/Respiratory_Sound_Database/Respiratory_Sound_Database/patient_diagnosis.csv', '/content/icbhi/respiratory_sound_database/Respiratory_Sound_Database/patient_diagnosis.csv']


In [ ]:
import os, glob, shutil, zipfile, hashlib, re
DEST='/content/icbhi'; ZIP='/content/drive/MyDrive/pulmoscan semester 7/data/icbhi_kaggle.zip'
A=os.path.join(DEST,'Respiratory_Sound_Database'); B=os.path.join(DEST,'respiratory_sound_database')
assert os.path.isdir(A) and os.path.isdir(B), "expected both folders, stop and paste the output of: os.listdir('/content/icbhi')"
def listing(root): return {os.path.relpath(p,root):os.path.getsize(p) for p in glob.glob(root+'/**/*',recursive=True) if os.path.isfile(p)}
la,lb=listing(A),listing(B)
print("files in each copy:",len(la),len(lb),"| same names and sizes:",la==lb)
def md5(p):
    h=hashlib.md5()
    with open(p,'rb') as f:
        for c in iter(lambda:f.read(1<<20),b''): h.update(c)
    return h.hexdigest()
sample=sorted(la)[::max(1,len(la)//60)]
same=all(md5(os.path.join(A,r))==md5(os.path.join(B,r)) for r in sample if r in lb)
print(f"checksums identical on {len(sample)} sampled files:",same)
if not (la==lb and same):
    print("STOP: the copies are not identical, nothing was deleted. Paste me these lines.")
else:
    shutil.rmtree(B)
    tmp=ZIP+'.tmp'
    with zipfile.ZipFile(tmp,'w',zipfile.ZIP_STORED) as z:
        for p in glob.glob(DEST+'/**/*',recursive=True):
            if os.path.isfile(p): z.write(p,os.path.relpath(p,DEST))
    os.replace(tmp,ZIP)
    print(f"Drive zip rebuilt: {os.path.getsize(ZIP)/1e9:.2f} GB")
wav=glob.glob(DEST+'/**/*.wav',recursive=True); txt=glob.glob(DEST+'/**/*.txt',recursive=True)
print(len(wav),"wav,",len(txt),"txt")
extra=[os.path.basename(t) for t in txt if not re.match(r'^\d+_',os.path.basename(t))]
print("non annotation text files:",extra)
print("diagnosis file:",glob.glob(DEST+'/**/patient_diagnosis.csv',recursive=True))

files in each copy: 1843 1843 | same names and sizes: True
checksums identical on 62 sampled files: True
Drive zip rebuilt: 2.18 GB
920 wav, 923 txt
non annotation text files: ['demographic_info.txt', 'filename_differences.txt', 'filename_format.txt']
diagnosis file: ['/content/icbhi/Respiratory_Sound_Database/Respiratory_Sound_Database/patient_diagnosis.csv']


In [ ]:
import os, io, re, glob, zipfile, numpy as np, pandas as pd, soundfile as sf
PROJ='/content/drive/MyDrive/pulmoscan semester 7'; DEST='/content/icbhi'
with zipfile.ZipFile(os.path.join(PROJ,'data','colab_training_package.zip')) as z:
    man=pd.read_csv(io.BytesIO(z.read('data/processed/manifest.csv')))
man['rec']=man.audio_file.str.extract(r'^(.*)_cycle\d+_[a-z]+\.wav$')[0]
man['cyc']=man.audio_file.str.extract(r'_cycle(\d+)_')[0].astype(int)

wav=glob.glob(DEST+'/**/*.wav',recursive=True); rows=[]; ann={}
for p in wav:
    name=os.path.basename(p)[:-4]; t=os.path.join(os.path.dirname(p),name+'.txt')
    if not os.path.exists(t): continue
    ann[name]=np.loadtxt(t,ndmin=2); info=sf.info(p)
    rows.append(dict(rec=name,patient=int(name.split('_')[0]),sr=info.samplerate,dur=info.duration,cycles=len(ann[name])))
R=pd.DataFrame(rows)
print(f"recordings {len(R)}, patients {R.patient.nunique()}, hours {R.dur.sum()/3600:.2f}, annotated cycles {R.cycles.sum()}")
print("sample rates:",R.sr.value_counts().to_dict())
print("recording length s: min %.1f median %.1f max %.1f"%(R.dur.min(),R.dur.median(),R.dur.max()))
end_ok=np.mean([a[:,1].max()<=d+0.05 for a,d in zip((ann[r] for r in R.rec),R.dur)])
print(f"recordings whose last cycle ends inside the audio: {end_ok:.3f}")

fd=glob.glob(DEST+'/**/filename_differences.txt',recursive=True)
if fd: print("\nfilename_differences.txt (first lines):\n"+"".join(open(fd[0]).readlines()[:12]))

mp=set(man.patient_id.astype(int)); sub=R[R.patient.isin(mp)]
print(f"manifest patients {len(mp)}, present in the Kaggle data: {len(mp & set(R.patient))}")
print(f"annotated cycles for those patients: {int(sub.cycles.sum())}   manifest cycles: {len(man)}")
ok=bad=missing=0; miss_recs=set()
for r in man.itertuples():
    a=ann.get(r.rec)
    if a is None or r.cyc>=len(a): missing+=1; miss_recs.add(r.rec); continue
    c,w=int(a[r.cyc,2]),int(a[r.cyc,3]); lab=1 if (c and not w) else 2 if (w and not c) else 3 if (c and w) else 0
    ok+=(lab==r.sound_label); bad+=(lab!=r.sound_label)
print(f"\ncycle label check: agree {ok}, disagree {bad}, not found {missing} (of {len(man)})")
if miss_recs: print("recordings with unmatched cycles:",sorted(miss_recs)[:8])

# preview: windows of 3.056 s every 1.5 s, labelled by the kind covering at least half of the window
WIN=3.056; STEP=1.5
split=man.groupby('patient_id').split.first().map(lambda s:'test' if s=='test' else 'dev').to_dict()
cnt={'dev':[0,0,0],'test':[0,0,0]}
for r in R.itertuples():
    g=split.get(r.patient)
    if g is None: continue
    a=ann[r.rec]
    for ws in np.arange(0,r.dur-WIN+1e-9,STEP):
        ov=np.clip(np.minimum(a[:,1],ws+WIN)-np.maximum(a[:,0],ws),0,None); ab=(a[:,2]+a[:,3])>0
        ca,cn=ov[ab].sum(),ov[~ab].sum()
        k=0 if cn>=WIN/2 else 1 if ca>=WIN/2 else 2
        cnt[g][k]+=1
for g,(n,a_,d) in cnt.items():
    t=n+a_+d; print(f"{g:4s}: windows {t}  normal {n} ({n/t:.0%})  abnormal {a_} ({a_/t:.0%})  dropped as mixed or uncovered {d} ({d/t:.0%})")

recordings 920, patients 126, hours 5.49, annotated cycles 6898
sample rates: {44100: 824, 4000: 90, 10000: 6}
recording length s: min 7.9 median 20.0 max 86.2
recordings whose last cycle ends inside the audio: 1.000

filename_differences.txt (first lines):
'101_1b1_Al_sc_AKGC417L'
'101_1b1_Pr_sc_AKGC417L'
'102_1b1_Ar_sc_AKGC417L'
'105_1b1_Tc_sc_LittC2SE'
'108_1b1_Al_sc_LittC2SE'
'111_1b2_Tc_sc_LittC2SE'
'111_1b3_Tc_sc_LittC2SE'
'115_1b1_Ar_sc_LittC2SE'
'116_1b2_Pl_sc_LittC2SE'
'116_1b2_Tc_sc_LittC2SE'
'119_1b1_Ar_sc_AKGC417L'
'121_1b1_Tc_sc_LittC2SE'

manifest patients 101, present in the Kaggle data: 101
annotated cycles for those patients: 5319   manifest cycles: 5319

cycle label check: agree 5319, disagree 0, not found 0 (of 5319)
dev : windows 6396  normal 3350 (52%)  abnormal 2871 (45%)  dropped as mixed or uncovered 175 (3%)
test: windows 2813  normal 1344 (48%)  abnormal 1423 (51%)  dropped as mixed or uncovered 46 (2%)


In [ ]:
import os, io, glob, json, sys, time, zipfile, datetime
import numpy as np, pandas as pd, librosa
PROJ='/content/drive/MyDrive/pulmoscan semester 7'; DEST='/content/icbhi'; WORK='/content/work'
os.makedirs(WORK+'/src',exist_ok=True)
SR=16000; NWIN=48896; STEP=24000; WSEC=NWIN/SR          # 3.056 s, windows start every 1.5 s

PP=os.path.join(PROJ,'results','window_protocol.json')
if not os.path.exists(PP):
    json.dump({'written_at':datetime.datetime.now().isoformat(),
      'question':'how much accuracy does the cycle trained model lose on device style fixed windows, and does training on windows recover it',
      'data':'920 ICBHI recordings (Kaggle copy); windows from the 82 development patients only; the 19 test patients are not used or built',
      'window':'48896 samples at 16 kHz (3.056 s, 192 frames), start every 24000 samples, fully inside the recording',
      'label':'abnormal if annotated abnormal cycles cover at least half the window, normal if normal cycles cover at least half, otherwise dropped; coverage fractions stored per window',
      'features':'src/preprocessing.py butterworth_filter then extract_mel_spectrogram on each window alone (same as the device front end)',
      'arms':{'A':'B1 trained on annotated cycles (existing recipe), evaluated on windows of held out patients',
              'B':'B1 trained on windows of the training patients, evaluated on windows of held out patients'},
      'design':'same 5 patient grouped folds as all earlier CV, 3 seeds per fold, binary labels',
      'metrics':'binary AUC per fold (3 seed mean), pooled sensitivity at specificity 0.70',
      'rule':'adopt window training only if mean fold AUC of B exceeds A by more than 0.02 and B is better on at least 4 of 5 folds',
      'test_set':'untouched; any later test use is a disclosed third use',
      'commitment':'all arms reported regardless of outcome; rule not changed after results'},open(PP,'w'),indent=2)
    print('protocol written')

with zipfile.ZipFile(os.path.join(PROJ,'data','colab_training_package.zip')) as z:
    open(WORK+'/src/preprocessing.py','wb').write(z.read('src/preprocessing.py'))
    man=pd.read_csv(io.BytesIO(z.read('data/processed/manifest.csv')))
sys.path.insert(0,WORK+'/src'); import preprocessing as P
dev_pat=set(man[man.split.isin(['train','val'])].patient_id.astype(int)); print("development patients:",len(dev_pat))

wavs={os.path.basename(p)[:-4]:p for p in glob.glob(DEST+'/**/*.wav',recursive=True)}
X=[];Y=[];PID=[];REC=[];ST=[];COV=[]; t0=time.time(); n_rec=0
for name,path in sorted(wavs.items()):
    pat=int(name.split('_')[0])
    if pat not in dev_pat: continue
    a=np.loadtxt(os.path.join(os.path.dirname(path),name+'.txt'),ndmin=2)
    y,_=librosa.load(path,sr=SR,mono=True)
    for s in range(0,len(y)-NWIN+1,STEP):
        ws=s/SR; ov=np.clip(np.minimum(a[:,1],ws+WSEC)-np.maximum(a[:,0],ws),0,None)
        cr,wh=a[:,2]>0,a[:,3]>0
        cov=np.array([ov[~cr&~wh].sum(),ov[cr&~wh].sum(),ov[~cr&wh].sum(),ov[cr&wh].sum()])   # normal, crackle, wheeze, both
        if cov[0]>=WSEC/2: lab=0
        elif cov[1:].sum()>=WSEC/2: lab=1
        else: continue
        f=P.extract_mel_spectrogram(P.butterworth_filter(y[s:s+NWIN]))
        assert f.shape==(128,192),f.shape
        X.append(f.astype(np.float16)); Y.append(lab); PID.append(pat); REC.append(name); ST.append(ws); COV.append(cov/WSEC)
    n_rec+=1
    if n_rec%20==0: print(f"  {n_rec} recordings, {len(Y)} windows, {(time.time()-t0)/60:.1f} min",flush=True)

X=np.stack(X); Y=np.array(Y,np.int8); PID=np.array(PID); COV=np.array(COV,np.float32)
out=os.path.join(PROJ,'data','windows_dev.npz')
np.savez(out,X=X,y=Y,patient=PID,rec=np.array(REC),start=np.array(ST,np.float32),cov=COV)
print(f"\nsaved {out}  ({os.path.getsize(out)/1e6:.0f} MB)")
print(f"windows {len(Y)}  normal {int((Y==0).sum())}  abnormal {int((Y==1).sum())}  patients {len(set(PID))}")
Xf=X[::50].astype(np.float32); print("feature check: mean %.3f std %.3f min %.2f max %.2f"%(Xf.mean(),Xf.std(),Xf.min(),Xf.max()))
cnt=pd.Series(PID).value_counts(); print("windows per patient: min %d median %d max %d"%(cnt.min(),cnt.median(),cnt.max()))
pure=((COV[:,0]>=0.9)|(COV[:,1:].sum(1)>=0.9)).mean(); print(f"windows that are at least 90% one kind: {pure:.0%}")

protocol written
development patients: 82
  20 recordings, 374 windows, 0.7 min
  40 recordings, 604 windows, 0.8 min
  60 recordings, 966 windows, 0.9 min
  80 recordings, 1204 windows, 0.9 min
  100 recordings, 1444 windows, 1.0 min
  120 recordings, 1684 windows, 1.1 min
  140 recordings, 1900 windows, 1.1 min
  160 recordings, 2140 windows, 1.2 min
  180 recordings, 2357 windows, 1.3 min
  200 recordings, 2558 windows, 1.3 min
  220 recordings, 2920 windows, 1.4 min
  240 recordings, 3160 windows, 1.5 min
  260 recordings, 3392 windows, 1.5 min
  280 recordings, 3629 windows, 1.6 min
  300 recordings, 3867 windows, 1.6 min
  320 recordings, 4107 windows, 1.7 min
  340 recordings, 4347 windows, 1.8 min
  360 recordings, 4602 windows, 1.8 min
  380 recordings, 4840 windows, 1.9 min
  400 recordings, 5069 windows, 2.0 min
  420 recordings, 5298 windows, 2.0 min
  440 recordings, 5537 windows, 2.1 min
  460 recordings, 5775 windows, 2.2 min
  480 recordings, 5993 windows, 2.2 min

save

In [ ]:
# ============================================================
# WINDOW EXPERIMENT: arm A (B1 trained on cycles) vs arm B (B1 trained on windows), both scored on windows
# GPU. self contained, resumable. No test data.
# ============================================================
import os, io, json, time, random, gc, zipfile, datetime
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.metrics import roc_auc_score, roc_curve

print("GPUs:", tf.config.list_physical_devices('GPU'))
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
PROJ='/content/drive/MyDrive/pulmoscan semester 7'; WORK='/content/work'
RES=os.path.join(PROJ,'results'); OOF=os.path.join(RES,'oof_probs'); os.makedirs(OOF,exist_ok=True)
PP=os.path.join(RES,'window_protocol.json')
N_MELS=128; T=192; EPOCHS=40; BATCH=32; LR=1e-3; LR_MIN=1e-6; DIAG_W=0.1; NS=4; ND=7
SEEDS=[0]            # stage 1; change to [0,1,2] for stage 2 (finished runs are skipped)

d=json.load(open(PP))
if 'amendment_1' not in d:
    d['amendment_1']={'written_at':datetime.datetime.now().isoformat(),'made_before_any_run':True,
      'change':'arm B trains on four class labels derived from window coverage (Normal, or the largest of crackle only, wheeze only, both) with the same architecture, loss and diagnosis head as arm A, so only the training data differs; evaluation stays binary on P(abnormal)=1-p[Normal]',
      'staging':'stage 1 runs seed 0 for both arms (10 runs); stage 2 adds seeds 1 and 2; the adoption rule is applied only when all 3 seeds are complete',
      'extra_output':'arm A is also scored on the held out cycles, giving the cycle versus window gap for the same model'}
    json.dump(d,open(PP,'w'),indent=2); print('amendment written')

# ---------- folds (identical to every earlier CV) ----------
with zipfile.ZipFile(os.path.join(PROJ,'data','colab_training_package.zip')) as z:
    man=pd.read_csv(io.BytesIO(z.read('data/processed/manifest.csv')))
man['patient_id']=man.patient_id.astype(int)
devm=man[man.split.isin(['train','val'])].copy().reset_index(drop=True)
rng=np.random.RandomState(42)
pdf=devm.groupby('patient_id')['diagnosis_label'].agg(lambda x:x.mode()[0]).reset_index()
folds=[[] for _ in range(5)]
for _,g in pdf.groupby('diagnosis_label'):
    ps=g.patient_id.values.copy(); rng.shuffle(ps)
    for j,p in enumerate(ps): folds[j%5].append(int(p))
pc=devm.patient_id.values
cyc_per_fold=[int(np.isin(pc,f).sum()) for f in folds]
print("patients per fold:",[len(f) for f in folds],"| cycles per fold:",cyc_per_fold)
assert cyc_per_fold[:3]==[1056,551,577], "folds differ from the earlier CV, stop and paste this output"
diag_of=dict(zip(pdf.patient_id.astype(int),pdf.diagnosis_label.astype(int)))

# ---------- windows ----------
W=np.load(os.path.join(PROJ,'data','windows_dev.npz'))
Xw=W['X']; yw=W['y'].astype(int); pw=W['patient'].astype(int); cov=W['cov']
ym=np.where(yw==0,0,1+np.argmax(cov[:,1:4],axis=1)); ydw=np.array([diag_of[p] for p in pw])
print("windows",len(yw),"| per fold:",[int(np.isin(pw,f).sum()) for f in folds],"| 4 class counts:",np.bincount(ym).tolist())

# ---------- cycle features (arm A training and the cycle versus window gap) ----------
os.makedirs(WORK,exist_ok=True)
if not os.path.isdir(os.path.join(WORK,'feat','processed','train')):
    with zipfile.ZipFile(os.path.join(PROJ,'data','pulmoscan_data.zip')) as z: z.extractall(os.path.join(WORK,'feat'))
devm['fp']=devm.apply(lambda r: os.path.join(WORK,'feat','processed',r['split'],r['audio_file'].replace('.wav','.npy')),axis=1)
Xc=np.empty((len(devm),N_MELS,T,1),np.float32)
for i,fp in enumerate(devm.fp):
    a=np.load(fp); Xc[i,...,0]=np.pad(a,[(0,0),(0,T-a.shape[1])]) if a.shape[1]<T else a[:,:T]
ysc=devm.sound_label.values.astype(np.int32); ydc=devm.diagnosis_label.values.astype(np.int32)
print("cycles",len(ysc))

# ---------- loss, B1 model, augmentation (same as the B1 cross validation) ----------
def focal(g=2.0):
    def fn(yt,yp):
        yp=tf.clip_by_value(yp,1e-7,1.0); ce=-yt*tf.math.log(yp)
        p=tf.reduce_sum(yt*yp,axis=-1,keepdims=True)
        return tf.reduce_mean(tf.pow(1.0-p,g)*ce)
    fn.__name__='focal'; return fn
def build_b1(T):
    L=tf.keras.layers; inp=tf.keras.Input(shape=(N_MELS,T,1)); x=inp
    for f in [32,64,128]:
        x=L.Conv2D(f,3,padding='same')(x); x=L.BatchNormalization()(x); x=L.Activation('relu')(x); x=L.MaxPooling2D(2)(x)
    x=L.Conv2D(256,3,padding='same')(x); x=L.BatchNormalization()(x); x=L.Activation('relu')(x)
    x=L.GlobalAveragePooling2D()(x)
    sh=L.Dropout(0.5)(L.Dense(256,activation='relu')(x))
    s=L.Dropout(0.3)(L.Dense(128,activation='relu')(sh)); so=L.Dense(NS,activation='softmax',name='sound')(s)
    dd=L.Dropout(0.3)(L.Dense(128,activation='relu')(sh)); do=L.Dense(ND,activation='softmax',name='diagnosis')(dd)
    return tf.keras.Model(inp,[so,do])
def augment(x,y):
    B=tf.shape(x)[0]; F=tf.shape(x)[1]; Tt=tf.shape(x)[2]
    real=tf.cast(tf.not_equal(x,0.0),tf.float32)
    x=x+tf.random.normal([B,1,1,1],0.0,0.15)*real
    fw=tf.random.uniform([B,1,1,1],0,21,dtype=tf.int32)
    f0=tf.random.uniform([B,1,1,1],0,1000000,dtype=tf.int32)%tf.maximum(F-fw,1)
    fi=tf.reshape(tf.range(F),[1,-1,1,1]); fmask=tf.logical_and(fi>=f0,fi<f0+fw)
    tw=tf.random.uniform([B,1,1,1],0,31,dtype=tf.int32)
    t0=tf.random.uniform([B,1,1,1],0,1000000,dtype=tf.int32)%tf.maximum(Tt-tw,1)
    ti=tf.reshape(tf.range(Tt),[1,1,-1,1]); tmask=tf.logical_and(ti>=t0,ti<t0+tw)
    return tf.where(tf.logical_or(fmask,tmask),tf.zeros_like(x),x),y

# ---------- runs ----------
def run(arm,k,seed):
    path=os.path.join(OOF,f'W_{arm}_f{k}_s{seed}.npz')
    if os.path.exists(path): print('  SKIP',os.path.basename(path)); return
    tew=np.isin(pw,folds[k]); tec=np.isin(pc,folds[k])
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed); tf.keras.backend.clear_session()
    if arm=='A': Xtr,ytr,ydtr=Xc[~tec],ysc[~tec],ydc[~tec]
    else:        trw=~tew; Xtr,ytr,ydtr=Xw[trw].astype(np.float32)[...,None],ym[trw],ydw[trw]
    ds=tf.data.Dataset.from_tensor_slices((Xtr,{'sound':tf.keras.utils.to_categorical(ytr,NS),'diagnosis':tf.keras.utils.to_categorical(ydtr,ND)}))
    ds=ds.shuffle(4096,seed=seed).batch(BATCH).map(augment,num_parallel_calls=tf.data.AUTOTUNE).prefetch(2)
    m=build_b1(T); m.compile(optimizer=tf.keras.optimizers.Adam(LR),loss={'sound':focal(2.0),'diagnosis':focal(2.0)},
        loss_weights={'sound':1.0,'diagnosis':DIAG_W},metrics={'sound':'accuracy'})
    t0=time.time()
    m.fit(ds,epochs=EPOCHS,verbose=0,callbacks=[tf.keras.callbacks.LearningRateScheduler(
        lambda e: float(LR_MIN+0.5*(LR-LR_MIN)*(1+np.cos(np.pi*e/EPOCHS))))])
    pr=m.predict(Xw[tew].astype(np.float32)[...,None],batch_size=64,verbose=0)[0]
    aw=roc_auc_score(yw[tew],1-pr[:,0]); extra={}; msg=f'window AUC {aw:.3f}'
    if arm=='A':
        prc=m.predict(Xc[tec],batch_size=64,verbose=0)[0]; ac=roc_auc_score(ysc[tec]>0,1-prc[:,0])
        extra=dict(pc=prc,yc=ysc[tec]); msg+=f'   cycle AUC {ac:.3f}'
    print(f'  {arm} fold {k} seed {seed}: {msg}   {(time.time()-t0)/60:.1f}min')
    np.savez_compressed(path,p=pr,y=yw[tew],patient=pw[tew],**extra)
    del m; gc.collect()

for seed in SEEDS:
    for k in range(5):
        run('A',k,seed); run('B',k,seed)

# ---------- summary ----------
L=lambda arm,k,s: np.load(os.path.join(OOF,f'W_{arm}_f{k}_s{s}.npz'))
done=[s for s in [0,1,2] if all(os.path.exists(os.path.join(OOF,f'W_{a}_f{k}_s{s}.npz')) for a in 'AB' for k in range(5))]
print("\ncomplete seeds:",done)
if done:
    print("fold  A cycles  A windows  B windows   B minus A (windows)")
    dA=[];dB=[]
    for k in range(5):
        ac=np.mean([roc_auc_score(L('A',k,s)['yc']>0,1-L('A',k,s)['pc'][:,0]) for s in done])
        aw=np.mean([roc_auc_score(L('A',k,s)['y'],1-L('A',k,s)['p'][:,0]) for s in done])
        bw=np.mean([roc_auc_score(L('B',k,s)['y'],1-L('B',k,s)['p'][:,0]) for s in done])
        dA.append((ac,aw)); dB.append(bw); print(f"  {k}    {ac:.3f}     {aw:.3f}      {bw:.3f}      {bw-aw:+.3f}")
    ac=np.mean([a[0] for a in dA]); aw=np.mean([a[1] for a in dA]); bw=np.mean(dB)
    print(f"mean  {ac:.3f}     {aw:.3f}      {bw:.3f}      {bw-aw:+.3f}    (reference: B1 cycle CV mean fold AUC 0.732)")
    def se_at(arm):
        r=[]
        for s in done:
            y=np.concatenate([L(arm,k,s)['y'] for k in range(5)])>0; sc=np.concatenate([1-L(arm,k,s)['p'][:,0] for k in range(5)])
            fpr,tpr,_=roc_curve(y,sc); r.append(tpr[(1-fpr)>=0.70].max())
        return np.mean(r)
    print(f"pooled window sensitivity at specificity 0.70:  A {se_at('A'):.2f}   B {se_at('B'):.2f}")
    if done==[0,1,2]:
        wins=sum((dB[k]-dA[k][1])>0 for k in range(5))
        print(f"\nRULE: mean AUC gain {bw-aw:+.3f} > 0.02: {bw-aw>0.02}   better on >=4 of 5 folds: {wins>=4} ({wins}/5)")
        print("VERDICT:",'TRAIN ON WINDOWS' if (bw-aw>0.02 and wins>=4) else 'KEEP CYCLE TRAINING, report the gap')
    else: print("\nstage 1 only (seed 0): no verdict, the rule needs all 3 seeds")

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Mounted at /content/drive
amendment written
patients per fold: [19, 18, 16, 16, 13] | cycles per fold: [1056, 551, 577, 722, 784]
windows 6221 | per fold: [1855, 998, 879, 1089, 1400] | 4 class counts: [3350, 1628, 786, 457]
cycles 3690
  A fold 0 seed 0: window AUC 0.663   cycle AUC 0.708   2.8min
  B fold 0 seed 0: window AUC 0.705   4.3min
  A fold 1 seed 0: window AUC 0.599   cycle AUC 0.652   3.1min
  B fold 1 seed 0: window AUC 0.724   4.8min
  A fold 2 seed 0: window AUC 0.692   cycle AUC 0.728   3.0min
  B fold 2 seed 0: window AUC 0.692   5.4min
  A fold 3 seed 0: window AUC 0.720   cycle AUC 0.799   2.9min
  B fold 3 seed 0: window AUC 0.749   4.8min
  A fold 4 seed 0: window AUC 0.609   cycle AUC 0.727   2.9min
  B fold 4 seed 0: window AUC 0.710   4.7min

complete seeds: [0]
fold  A cycles  A windows  B windows   B minus A (windows)
  0    0.708     0.663      0.705      +0.042
  1    0.652     0.599  

In [ ]:
# ============================================================
# WINDOW EXPERIMENT: arm A (B1 trained on cycles) vs arm B (B1 trained on windows), both scored on windows
# GPU. self contained, resumable. No test data.
# ============================================================
import os, io, json, time, random, gc, zipfile, datetime
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.metrics import roc_auc_score, roc_curve

print("GPUs:", tf.config.list_physical_devices('GPU'))
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
PROJ='/content/drive/MyDrive/pulmoscan semester 7'; WORK='/content/work'
RES=os.path.join(PROJ,'results'); OOF=os.path.join(RES,'oof_probs'); os.makedirs(OOF,exist_ok=True)
PP=os.path.join(RES,'window_protocol.json')
N_MELS=128; T=192; EPOCHS=40; BATCH=32; LR=1e-3; LR_MIN=1e-6; DIAG_W=0.1; NS=4; ND=7
SEEDS=[0,1,2]        # stage 2: seed 0 is already done and will be skipped

d=json.load(open(PP))
if 'amendment_1' not in d:
    d['amendment_1']={'written_at':datetime.datetime.now().isoformat(),'made_before_any_run':True,
      'change':'arm B trains on four class labels derived from window coverage (Normal, or the largest of crackle only, wheeze only, both) with the same architecture, loss and diagnosis head as arm A, so only the training data differs; evaluation stays binary on P(abnormal)=1-p[Normal]',
      'staging':'stage 1 runs seed 0 for both arms (10 runs); stage 2 adds seeds 1 and 2; the adoption rule is applied only when all 3 seeds are complete',
      'extra_output':'arm A is also scored on the held out cycles, giving the cycle versus window gap for the same model'}
    json.dump(d,open(PP,'w'),indent=2); print('amendment written')

# ---------- folds (identical to every earlier CV) ----------
with zipfile.ZipFile(os.path.join(PROJ,'data','colab_training_package.zip')) as z:
    man=pd.read_csv(io.BytesIO(z.read('data/processed/manifest.csv')))
man['patient_id']=man.patient_id.astype(int)
devm=man[man.split.isin(['train','val'])].copy().reset_index(drop=True)
rng=np.random.RandomState(42)
pdf=devm.groupby('patient_id')['diagnosis_label'].agg(lambda x:x.mode()[0]).reset_index()
folds=[[] for _ in range(5)]
for _,g in pdf.groupby('diagnosis_label'):
    ps=g.patient_id.values.copy(); rng.shuffle(ps)
    for j,p in enumerate(ps): folds[j%5].append(int(p))
pc=devm.patient_id.values
cyc_per_fold=[int(np.isin(pc,f).sum()) for f in folds]
print("patients per fold:",[len(f) for f in folds],"| cycles per fold:",cyc_per_fold)
assert cyc_per_fold[:3]==[1056,551,577], "folds differ from the earlier CV, stop and paste this output"
diag_of=dict(zip(pdf.patient_id.astype(int),pdf.diagnosis_label.astype(int)))

# ---------- windows ----------
W=np.load(os.path.join(PROJ,'data','windows_dev.npz'))
Xw=W['X']; yw=W['y'].astype(int); pw=W['patient'].astype(int); cov=W['cov']
ym=np.where(yw==0,0,1+np.argmax(cov[:,1:4],axis=1)); ydw=np.array([diag_of[p] for p in pw])
print("windows",len(yw),"| per fold:",[int(np.isin(pw,f).sum()) for f in folds],"| 4 class counts:",np.bincount(ym).tolist())

# ---------- cycle features (arm A training and the cycle versus window gap) ----------
os.makedirs(WORK,exist_ok=True)
if not os.path.isdir(os.path.join(WORK,'feat','processed','train')):
    with zipfile.ZipFile(os.path.join(PROJ,'data','pulmoscan_data.zip')) as z: z.extractall(os.path.join(WORK,'feat'))
devm['fp']=devm.apply(lambda r: os.path.join(WORK,'feat','processed',r['split'],r['audio_file'].replace('.wav','.npy')),axis=1)
Xc=np.empty((len(devm),N_MELS,T,1),np.float32)
for i,fp in enumerate(devm.fp):
    a=np.load(fp); Xc[i,...,0]=np.pad(a,[(0,0),(0,T-a.shape[1])]) if a.shape[1]<T else a[:,:T]
ysc=devm.sound_label.values.astype(np.int32); ydc=devm.diagnosis_label.values.astype(np.int32)
print("cycles",len(ysc))

# ---------- loss, B1 model, augmentation (same as the B1 cross validation) ----------
def focal(g=2.0):
    def fn(yt,yp):
        yp=tf.clip_by_value(yp,1e-7,1.0); ce=-yt*tf.math.log(yp)
        p=tf.reduce_sum(yt*yp,axis=-1,keepdims=True)
        return tf.reduce_mean(tf.pow(1.0-p,g)*ce)
    fn.__name__='focal'; return fn
def build_b1(T):
    L=tf.keras.layers; inp=tf.keras.Input(shape=(N_MELS,T,1)); x=inp
    for f in [32,64,128]:
        x=L.Conv2D(f,3,padding='same')(x); x=L.BatchNormalization()(x); x=L.Activation('relu')(x); x=L.MaxPooling2D(2)(x)
    x=L.Conv2D(256,3,padding='same')(x); x=L.BatchNormalization()(x); x=L.Activation('relu')(x)
    x=L.GlobalAveragePooling2D()(x)
    sh=L.Dropout(0.5)(L.Dense(256,activation='relu')(x))
    s=L.Dropout(0.3)(L.Dense(128,activation='relu')(sh)); so=L.Dense(NS,activation='softmax',name='sound')(s)
    dd=L.Dropout(0.3)(L.Dense(128,activation='relu')(sh)); do=L.Dense(ND,activation='softmax',name='diagnosis')(dd)
    return tf.keras.Model(inp,[so,do])
def augment(x,y):
    B=tf.shape(x)[0]; F=tf.shape(x)[1]; Tt=tf.shape(x)[2]
    real=tf.cast(tf.not_equal(x,0.0),tf.float32)
    x=x+tf.random.normal([B,1,1,1],0.0,0.15)*real
    fw=tf.random.uniform([B,1,1,1],0,21,dtype=tf.int32)
    f0=tf.random.uniform([B,1,1,1],0,1000000,dtype=tf.int32)%tf.maximum(F-fw,1)
    fi=tf.reshape(tf.range(F),[1,-1,1,1]); fmask=tf.logical_and(fi>=f0,fi<f0+fw)
    tw=tf.random.uniform([B,1,1,1],0,31,dtype=tf.int32)
    t0=tf.random.uniform([B,1,1,1],0,1000000,dtype=tf.int32)%tf.maximum(Tt-tw,1)
    ti=tf.reshape(tf.range(Tt),[1,1,-1,1]); tmask=tf.logical_and(ti>=t0,ti<t0+tw)
    return tf.where(tf.logical_or(fmask,tmask),tf.zeros_like(x),x),y

# ---------- runs ----------
def run(arm,k,seed):
    path=os.path.join(OOF,f'W_{arm}_f{k}_s{seed}.npz')
    if os.path.exists(path): print('  SKIP',os.path.basename(path)); return
    tew=np.isin(pw,folds[k]); tec=np.isin(pc,folds[k])
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed); tf.keras.backend.clear_session()
    if arm=='A': Xtr,ytr,ydtr=Xc[~tec],ysc[~tec],ydc[~tec]
    else:        trw=~tew; Xtr,ytr,ydtr=Xw[trw].astype(np.float32)[...,None],ym[trw],ydw[trw]
    ds=tf.data.Dataset.from_tensor_slices((Xtr,{'sound':tf.keras.utils.to_categorical(ytr,NS),'diagnosis':tf.keras.utils.to_categorical(ydtr,ND)}))
    ds=ds.shuffle(4096,seed=seed).batch(BATCH).map(augment,num_parallel_calls=tf.data.AUTOTUNE).prefetch(2)
    m=build_b1(T); m.compile(optimizer=tf.keras.optimizers.Adam(LR),loss={'sound':focal(2.0),'diagnosis':focal(2.0)},
        loss_weights={'sound':1.0,'diagnosis':DIAG_W},metrics={'sound':'accuracy'})
    t0=time.time()
    m.fit(ds,epochs=EPOCHS,verbose=0,callbacks=[tf.keras.callbacks.LearningRateScheduler(
        lambda e: float(LR_MIN+0.5*(LR-LR_MIN)*(1+np.cos(np.pi*e/EPOCHS))))])
    pr=m.predict(Xw[tew].astype(np.float32)[...,None],batch_size=64,verbose=0)[0]
    aw=roc_auc_score(yw[tew],1-pr[:,0]); extra={}; msg=f'window AUC {aw:.3f}'
    if arm=='A':
        prc=m.predict(Xc[tec],batch_size=64,verbose=0)[0]; ac=roc_auc_score(ysc[tec]>0,1-prc[:,0])
        extra=dict(pc=prc,yc=ysc[tec]); msg+=f'   cycle AUC {ac:.3f}'
    print(f'  {arm} fold {k} seed {seed}: {msg}   {(time.time()-t0)/60:.1f}min')
    np.savez_compressed(path,p=pr,y=yw[tew],patient=pw[tew],**extra)
    del m; gc.collect()

for seed in SEEDS:
    for k in range(5):
        run('A',k,seed); run('B',k,seed)

# ---------- summary ----------
L=lambda arm,k,s: np.load(os.path.join(OOF,f'W_{arm}_f{k}_s{s}.npz'))
done=[s for s in [0,1,2] if all(os.path.exists(os.path.join(OOF,f'W_{a}_f{k}_s{s}.npz')) for a in 'AB' for k in range(5))]
print("\ncomplete seeds:",done)
if done:
    print("fold  A cycles  A windows  B windows   B minus A (windows)")
    dA=[];dB=[]
    for k in range(5):
        ac=np.mean([roc_auc_score(L('A',k,s)['yc']>0,1-L('A',k,s)['pc'][:,0]) for s in done])
        aw=np.mean([roc_auc_score(L('A',k,s)['y'],1-L('A',k,s)['p'][:,0]) for s in done])
        bw=np.mean([roc_auc_score(L('B',k,s)['y'],1-L('B',k,s)['p'][:,0]) for s in done])
        dA.append((ac,aw)); dB.append(bw); print(f"  {k}    {ac:.3f}     {aw:.3f}      {bw:.3f}      {bw-aw:+.3f}")
    ac=np.mean([a[0] for a in dA]); aw=np.mean([a[1] for a in dA]); bw=np.mean(dB)
    print(f"mean  {ac:.3f}     {aw:.3f}      {bw:.3f}      {bw-aw:+.3f}    (reference: B1 cycle CV mean fold AUC 0.732)")
    def se_at(arm):
        r=[]
        for s in done:
            y=np.concatenate([L(arm,k,s)['y'] for k in range(5)])>0; sc=np.concatenate([1-L(arm,k,s)['p'][:,0] for k in range(5)])
            fpr,tpr,_=roc_curve(y,sc); r.append(tpr[(1-fpr)>=0.70].max())
        return np.mean(r)
    print(f"pooled window sensitivity at specificity 0.70:  A {se_at('A'):.2f}   B {se_at('B'):.2f}")
    if done==[0,1,2]:
        wins=sum((dB[k]-dA[k][1])>0 for k in range(5))
        print(f"\nRULE: mean AUC gain {bw-aw:+.3f} > 0.02: {bw-aw>0.02}   better on >=4 of 5 folds: {wins>=4} ({wins}/5)")
        print("VERDICT:",'TRAIN ON WINDOWS' if (bw-aw>0.02 and wins>=4) else 'KEEP CYCLE TRAINING, report the gap')
    else: print("\nnot all 3 seeds complete: no verdict, the rule needs all 3 seeds")

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Mounted at /content/drive
patients per fold: [19, 18, 16, 16, 13] | cycles per fold: [1056, 551, 577, 722, 784]
windows 6221 | per fold: [1855, 998, 879, 1089, 1400] | 4 class counts: [3350, 1628, 786, 457]
cycles 3690
  SKIP W_A_f0_s0.npz
  SKIP W_B_f0_s0.npz
  SKIP W_A_f1_s0.npz
  SKIP W_B_f1_s0.npz
  SKIP W_A_f2_s0.npz
  SKIP W_B_f2_s0.npz
  SKIP W_A_f3_s0.npz
  SKIP W_B_f3_s0.npz
  SKIP W_A_f4_s0.npz
  SKIP W_B_f4_s0.npz
  A fold 0 seed 1: window AUC 0.676   cycle AUC 0.713   3.1min
  B fold 0 seed 1: window AUC 0.691   4.3min
  A fold 1 seed 1: window AUC 0.670   cycle AUC 0.711   3.1min
  B fold 1 seed 1: window AUC 0.769   4.9min
  A fold 2 seed 1: window AUC 0.688   cycle AUC 0.725   3.1min
  B fold 2 seed 1: window AUC 0.686   5.6min
  A fold 3 seed 1: window AUC 0.678   cycle AUC 0.784   3.0min
  B fold 3 seed 1: window AUC 0.759   4.9min
  A fold 4 seed 1: window AUC 0.619   cycle AUC 0.723   2.9min
  B

In [ ]:
import os, json, datetime
import numpy as np
from sklearn.metrics import roc_auc_score
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
PROJ='/content/drive/MyDrive/pulmoscan semester 7'; RES=os.path.join(PROJ,'results')
OOF=os.path.join(RES,'oof_probs'); PP=os.path.join(RES,'window_protocol.json'); SP=0.70

d=json.load(open(PP))
if 'amendment_2' not in d:
    d['amendment_2']={'written_at':datetime.datetime.now().isoformat(),
      'verdict':'TRAIN ON WINDOWS (3 seeds: mean fold AUC on windows A 0.667, B 0.720, gain +0.053; B better on 5 of 5 folds; fold gains 0.019, 0.093, 0.002, 0.057, 0.097)',
      'caveat':'pooled window sensitivity at specificity 0.70 was 0.56 for A and 0.55 for B, so the AUC gain is not visible at that operating point in pooled scores; checked per fold in this step',
      'deployed_model':'B1 architecture trained on windows of all 82 development patients, four class labels from window coverage, seed 0 deployed, seeds 1 and 2 for reference',
      'threshold_rule':'threshold on P(abnormal)=1-p[Normal] at pooled out of fold specificity 0.70 for arm B, mean over 3 seeds, same rule as the cycle models; not tuned on any test data',
      'test_set':'not used; building and scoring test windows would be a disclosed third use and is decided separately'}
    json.dump(d,open(PP,'w'),indent=2); print('amendment 2 written')

L=lambda arm,k,s: np.load(os.path.join(OOF,f'W_{arm}_f{k}_s{s}.npz'))
def fold_stat(arm,k,s):
    z=L(arm,k,s); y=z['y']>0; sc=1-z['p'][:,0]; th=np.quantile(sc[~y],SP)
    return (sc[y]>th).mean(), th, roc_auc_score(y,sc)
R={a:np.array([np.mean([fold_stat(a,k,s) for s in range(3)],axis=0) for k in range(5)]) for a in 'AB'}
print("\nwindows of held out patients, mean of 3 seeds")
print("fold | Se at Sp 0.70:  A     B    diff |  AUC:   A     B   | threshold at Sp 0.70:  A     B")
for k in range(5):
    print(f"  {k}  |            {R['A'][k,0]:.2f}  {R['B'][k,0]:.2f}  {R['B'][k,0]-R['A'][k,0]:+.2f} |      {R['A'][k,2]:.3f} {R['B'][k,2]:.3f} |            {R['A'][k,1]:.3f} {R['B'][k,1]:.3f}")
print(f"mean |            {R['A'][:,0].mean():.2f}  {R['B'][:,0].mean():.2f}  {R['B'][:,0].mean()-R['A'][:,0].mean():+.2f} |      {R['A'][:,2].mean():.3f} {R['B'][:,2].mean():.3f} |   SD across folds     {R['A'][:,1].std():.3f} {R['B'][:,1].std():.3f}")

def pooled(arm,s):
    y=np.concatenate([L(arm,k,s)['y'] for k in range(5)])>0
    sc=np.concatenate([1-L(arm,k,s)['p'][:,0] for k in range(5)]); return y,sc
ths={}
for arm in 'AB':
    r=[]
    for s in range(3):
        y,sc=pooled(arm,s); r.append(np.quantile(sc[~y],SP))
    ths[arm]=float(np.mean(r))
print(f"\npooled threshold at Sp 0.70:  A {ths['A']:.3f}   B {ths['B']:.3f}")
for arm in 'BA':
    print(f"\narm {arm}: ONE fixed threshold {ths[arm]:.3f} applied to each fold (mean of 3 seeds)")
    for k in range(5):
        se=[];sp=[]
        for s in range(3):
            z=L(arm,k,s); y=z['y']>0; sc=1-z['p'][:,0]
            se.append((sc[y]>ths[arm]).mean()); sp.append((sc[~y]<=ths[arm]).mean())
        print(f"  fold {k}: Se {np.mean(se):.2f}  Sp {np.mean(sp):.2f}")
d=json.load(open(PP)); d['deployment_threshold_B']={'value':ths['B'],'rule':'pooled out of fold, Sp 0.70, mean of 3 seeds','written_at':datetime.datetime.now().isoformat()}
json.dump(d,open(PP,'w'),indent=2); print(f"\nsaved deployment threshold for the window trained model: {ths['B']:.3f}")

Mounted at /content/drive
amendment 2 written

windows of held out patients, mean of 3 seeds
fold | Se at Sp 0.70:  A     B    diff |  AUC:   A     B   | threshold at Sp 0.70:  A     B
  0  |            0.55  0.54  -0.00 |      0.674 0.692 |            0.577 0.538
  1  |            0.50  0.63  +0.13 |      0.657 0.750 |            0.730 0.529
  2  |            0.60  0.56  -0.04 |      0.695 0.697 |            0.604 0.501
  3  |            0.55  0.68  +0.13 |      0.701 0.758 |            0.566 0.439
  4  |            0.41  0.60  +0.19 |      0.607 0.704 |            0.745 0.488
mean |            0.52  0.60  +0.08 |      0.667 0.720 |   SD across folds     0.077 0.035

pooled threshold at Sp 0.70:  A 0.626   B 0.508

arm B: ONE fixed threshold 0.508 applied to each fold (mean of 3 seeds)
  fold 0: Se 0.64  Sp 0.62
  fold 1: Se 0.70  Sp 0.64
  fold 2: Se 0.55  Sp 0.72
  fold 3: Se 0.40  Sp 0.90
  fold 4: Se 0.52  Sp 0.76

arm A: ONE fixed threshold 0.626 applied to each fold (mean of 3 s

In [ ]:
# ============================================================
# FINAL: B1 trained on windows of all 82 development patients (3 seeds) + int8 check on seed 0
# GPU. self contained, resumable. No test data.
# ============================================================
import os, io, json, time, random, gc, zipfile, datetime
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.metrics import roc_auc_score

print("GPUs:", tf.config.list_physical_devices('GPU'))
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
PROJ='/content/drive/MyDrive/pulmoscan semester 7'; CK=os.path.join(PROJ,'checkpoints'); TFD=os.path.join(PROJ,'tflite')
RES=os.path.join(PROJ,'results'); PP=os.path.join(RES,'window_protocol.json')
os.makedirs(CK,exist_ok=True); os.makedirs(TFD,exist_ok=True)
N_MELS=128; T=192; EPOCHS=40; BATCH=32; LR=1e-3; LR_MIN=1e-6; DIAG_W=0.1; NS=4; ND=7
d=json.load(open(PP)); THR=d['deployment_threshold_B']['value']; print("deployment threshold:",round(THR,3))
if 'amendment_3' not in d:
    d['amendment_3']={'written_at':datetime.datetime.now().isoformat(),
      'step':'final training of B1 on windows of all 82 dev patients, seeds 0 1 2, then int8 check of seed 0',
      'int8_bar':'agreement at the deployment threshold at least 0.95 and dev AUC drop (float minus int8) at most 0.02 on 800 random training windows; one calibration set of 1500 random windows (RandomState 1); no retries with other sets',
      'note':'the agreement check is in sample and measures quantisation drift only',
      'test_set':'not used'}
    json.dump(d,open(PP,'w'),indent=2); print('amendment 3 written')

with zipfile.ZipFile(os.path.join(PROJ,'data','colab_training_package.zip')) as z:
    man=pd.read_csv(io.BytesIO(z.read('data/processed/manifest.csv')))
devm=man[man.split.isin(['train','val'])]
diag_of=dict(zip(devm.groupby('patient_id')['diagnosis_label'].agg(lambda x:x.mode()[0]).index.astype(int),
                 devm.groupby('patient_id')['diagnosis_label'].agg(lambda x:x.mode()[0]).astype(int)))
W=np.load(os.path.join(PROJ,'data','windows_dev.npz'))
Xw=W['X']; yw=W['y'].astype(int); pw=W['patient'].astype(int); cov=W['cov']
ym=np.where(yw==0,0,1+np.argmax(cov[:,1:4],axis=1)); ydw=np.array([diag_of[p] for p in pw])
print("windows",len(yw),"patients",len(set(pw)),"4 class counts",np.bincount(ym).tolist())

def focal(g=2.0):
    def fn(yt,yp):
        yp=tf.clip_by_value(yp,1e-7,1.0); ce=-yt*tf.math.log(yp)
        p=tf.reduce_sum(yt*yp,axis=-1,keepdims=True)
        return tf.reduce_mean(tf.pow(1.0-p,g)*ce)
    fn.__name__='focal'; return fn
def build_b1(T):
    L=tf.keras.layers; inp=tf.keras.Input(shape=(N_MELS,T,1)); x=inp
    for f in [32,64,128]:
        x=L.Conv2D(f,3,padding='same')(x); x=L.BatchNormalization()(x); x=L.Activation('relu')(x); x=L.MaxPooling2D(2)(x)
    x=L.Conv2D(256,3,padding='same')(x); x=L.BatchNormalization()(x); x=L.Activation('relu')(x)
    x=L.GlobalAveragePooling2D()(x)
    sh=L.Dropout(0.5)(L.Dense(256,activation='relu')(x))
    s=L.Dropout(0.3)(L.Dense(128,activation='relu')(sh)); so=L.Dense(NS,activation='softmax',name='sound')(s)
    dd=L.Dropout(0.3)(L.Dense(128,activation='relu')(sh)); do=L.Dense(ND,activation='softmax',name='diagnosis')(dd)
    return tf.keras.Model(inp,[so,do])
def augment(x,y):
    B=tf.shape(x)[0]; F=tf.shape(x)[1]; Tt=tf.shape(x)[2]
    real=tf.cast(tf.not_equal(x,0.0),tf.float32)
    x=x+tf.random.normal([B,1,1,1],0.0,0.15)*real
    fw=tf.random.uniform([B,1,1,1],0,21,dtype=tf.int32)
    f0=tf.random.uniform([B,1,1,1],0,1000000,dtype=tf.int32)%tf.maximum(F-fw,1)
    fi=tf.reshape(tf.range(F),[1,-1,1,1]); fmask=tf.logical_and(fi>=f0,fi<f0+fw)
    tw=tf.random.uniform([B,1,1,1],0,31,dtype=tf.int32)
    t0=tf.random.uniform([B,1,1,1],0,1000000,dtype=tf.int32)%tf.maximum(Tt-tw,1)
    ti=tf.reshape(tf.range(Tt),[1,1,-1,1]); tmask=tf.logical_and(ti>=t0,ti<t0+tw)
    return tf.where(tf.logical_or(fmask,tmask),tf.zeros_like(x),x),y

Xtr=Xw.astype(np.float32)[...,None]; rs=np.random.RandomState(5); chk=rs.choice(len(Xtr),1000,replace=False)
m0=None
for seed in [0,1,2]:
    path=os.path.join(CK,f'final_W_B1_s{seed}.keras')
    if os.path.exists(path):
        m=tf.keras.models.load_model(path,compile=False); print(f"  loaded {os.path.basename(path)}")
    else:
        random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed); tf.keras.backend.clear_session()
        ds=tf.data.Dataset.from_tensor_slices((Xtr,{'sound':tf.keras.utils.to_categorical(ym,NS),'diagnosis':tf.keras.utils.to_categorical(ydw,ND)}))
        ds=ds.shuffle(4096,seed=seed).batch(BATCH).map(augment,num_parallel_calls=tf.data.AUTOTUNE).prefetch(2)
        m=build_b1(T); m.compile(optimizer=tf.keras.optimizers.Adam(LR),loss={'sound':focal(2.0),'diagnosis':focal(2.0)},
            loss_weights={'sound':1.0,'diagnosis':DIAG_W},metrics={'sound':'accuracy'})
        t0=time.time()
        m.fit(ds,epochs=EPOCHS,verbose=0,callbacks=[tf.keras.callbacks.LearningRateScheduler(
            lambda e: float(LR_MIN+0.5*(LR-LR_MIN)*(1+np.cos(np.pi*e/EPOCHS))))])
        try: m.save(path)
        except Exception as e:
            print("  full save failed:",str(e)[:100],"-> weights only"); m.save_weights(path.replace('.keras','.weights.h5'))
        print(f"  seed {seed}: trained {(time.time()-t0)/60:.1f} min, params {m.count_params()}")
    pr=m.predict(Xtr[chk],batch_size=64,verbose=0)[0]
    print(f"    sanity (in sample, 1000 windows): AUC {roc_auc_score(yw[chk],1-pr[:,0]):.3f}, mean P(abnormal) {np.mean(1-pr[:,0]):.3f}")
    if seed==0: m0=tf.keras.Model(m.input,m.outputs[0])
    else: del m
    gc.collect()

# ---------- int8 conversion of seed 0, one pre registered calibration set ----------
cal=np.random.RandomState(1).choice(len(Xtr),1500,replace=False)
def rep():
    for i in cal: yield [Xtr[i:i+1]]
conv=tf.lite.TFLiteConverter.from_keras_model(m0)
conv.optimizations=[tf.lite.Optimize.DEFAULT]; conv.representative_dataset=rep
conv.target_spec.supported_ops=[tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv.inference_input_type=tf.int8; conv.inference_output_type=tf.int8
tfl=conv.convert(); tp=os.path.join(TFD,'pulmoscan_W_B1_s0_int8.tflite'); open(tp,'wb').write(tfl)
ev=np.random.RandomState(0).choice(len(Xtr),800,replace=False)
pf=1-m0.predict(Xtr[ev],batch_size=64,verbose=0)[:,0]
try:
    itp=tf.lite.Interpreter(model_content=tfl,experimental_op_resolver_type=tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES)
except Exception: itp=tf.lite.Interpreter(model_content=tfl)
itp.allocate_tensors(); inp=itp.get_input_details()[0]; out=itp.get_output_details()[0]
si,zi=inp['quantization']; so,zo=out['quantization']; pq=np.empty(len(ev))
for j,i in enumerate(ev):
    q=np.clip(np.round(Xtr[i:i+1]/si+zi),-128,127).astype(np.int8)
    itp.set_tensor(inp['index'],q); itp.invoke()
    pq[j]=1-(itp.get_tensor(out['index']).astype(np.float32)-zo)[0,0]*so
agree=float(np.mean((pf>THR)==(pq>THR))); a_f=float(roc_auc_score(yw[ev],pf)); a_q=float(roc_auc_score(yw[ev],pq))
ok=(agree>=0.95 and a_f-a_q<=0.02); cut=(1-THR)/so+zo
print(f"\nW_B1 int8: size {len(tfl)/1e3:.0f} KB, mean abs diff {np.mean(np.abs(pf-pq)):.4f}, corr {np.corrcoef(pf,pq)[0,1]:.4f}, agreement {agree:.4f}, AUC float {a_f:.3f} int8 {a_q:.3f}")
print("RESULT:","PASS" if ok else "FAIL","(bar: agreement 0.95, AUC drop at most 0.02)")
print(f"flagged abnormal at {THR:.3f}: float {np.mean(pf>THR):.2f}  int8 {np.mean(pq>THR):.2f}")
print(f"constants: input scale {si:.6f} zp {int(zi)} | output scale {so:.6f} zp {int(zo)} | abnormal if output[0] < {cut:.2f}")
ops=sorted({x['op_name'] for x in itp._get_ops_details()}); print("operators:",ops)
json.dump({'input_scale':float(si),'input_zero_point':int(zi),'output_scale':float(so),'output_zero_point':int(zo),
           'threshold_p_abnormal':THR,'abnormal_if_output0_below':float(cut),'ops':ops,'size_bytes':len(tfl),'passed':bool(ok)},
          open(os.path.join(TFD,'firmware_constants_W.json'),'w'),indent=2)
with open(os.path.join(TFD,'pulmoscan_model_data_W.h'),'w') as f:
    f.write('// PulmoScan window trained B1 int8\n#include <stdint.h>\nalignas(16) const unsigned char pulmoscan_model[] = {\n')
    for k in range(0,len(tfl),24): f.write(', '.join(str(b) for b in tfl[k:k+24])+',\n')
    f.write(f'}};\nconst unsigned int pulmoscan_model_len = {len(tfl)};\n')
print("written: pulmoscan_W_B1_s0_int8.tflite, firmware_constants_W.json, pulmoscan_model_data_W.h in",TFD)

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
deployment threshold: 0.508
amendment 3 written
windows 6221 patients 82 4 class counts [3350, 1628, 786, 457]
  seed 0: trained 5.2 min, params 522763
    sanity (in sample, 1000 windows): AUC 0.851, mean P(abnormal) 0.541
  seed 1: trained 5.0 min, params 522763
    sanity (in sample, 1000 windows): AUC 0.842, mean P(abnormal) 0.537
  seed 2: trained 5.0 min, params 522763
    sanity (in sample, 1000 windows): AUC 0.846, mean P(abnormal) 0.530
Saved artifact at '/tmp/tmpe08oks6m'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 192, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  138972032159760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138972032159376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138972032160720: TensorSpec(shape=(), dtype=tf.reso

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



W_B1 int8: size 519 KB, mean abs diff 0.0058, corr 0.9994, agreement 0.9850, AUC float 0.863 int8 0.863
RESULT: PASS (bar: agreement 0.95, AUC drop at most 0.02)
flagged abnormal at 0.508: float 0.47  int8 0.48
constants: input scale 0.043229 zp -8 | output scale 0.003906 zp -128 | abnormal if output[0] < -2.09
operators: ['CONV_2D', 'FULLY_CONNECTED', 'MAX_POOL_2D', 'MEAN', 'SOFTMAX']
written: pulmoscan_W_B1_s0_int8.tflite, firmware_constants_W.json, pulmoscan_model_data_W.h in /content/drive/MyDrive/pulmoscan semester 7/tflite


In [ ]:
# ============================================================
# ONE TIME TEST EVALUATION on test windows (disclosed third use of the test set)
# Deployed model: window trained B1, int8, seed 0. CPU is fine. Refuses to run twice.
# ============================================================
import os, io, re, json, sys, time, zipfile, datetime
import numpy as np, pandas as pd, librosa, tensorflow as tf
from sklearn.metrics import roc_auc_score

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
PROJ='/content/drive/MyDrive/pulmoscan semester 7'; RES=os.path.join(PROJ,'results'); CK=os.path.join(PROJ,'checkpoints')
TFD=os.path.join(PROJ,'tflite'); WORK='/content/work'; TD='/content/icbhi_test'
RESF=os.path.join(RES,'test_result_windows_FINAL.json'); TPF=os.path.join(RES,'test_protocol_windows.json')
if os.path.exists(RESF):
    print("Already evaluated. The test set may be used once. Stored result:\n")
    print(json.dumps(json.load(open(RESF)),indent=1)[:5000]); raise SystemExit

THR=json.load(open(os.path.join(RES,'window_protocol.json')))['deployment_threshold_B']['value']
if not os.path.exists(TPF):
    json.dump({'written_at':datetime.datetime.now().isoformat(),
      'disclosure':'third use of the test set (1: 2026-09-06 cycle ensemble, official 40.2; 2: cycle trained A1, binary AUC 0.55 to 0.58); this evaluation is of a different model trained on windows',
      'models':'primary: final_W_B1_s0 int8 (the deployed file, reference kernels); reference: final_W_B1_s0, s1, s2 float',
      'threshold':f'{THR} on P(abnormal)=1-p[Normal], fixed from out of fold windows of 82 development patients; not tuned on test data',
      'test_windows':'same rule as development: 48896 samples at 16 kHz, start every 24000 samples, abnormal or normal by at least half coverage, mixed windows dropped; features from src/preprocessing.py per window',
      'metrics':'binary AUC, sensitivity and specificity at the fixed threshold, share flagged; patient level bootstrap 95 percent interval (1000 resamples); per device; sensitivity by abnormal kind',
      'cv_reference':'window CV of the same recipe: mean fold AUC 0.720; fixed threshold per fold Se 0.40 to 0.70 (mean 0.56), Sp 0.62 to 0.90 (mean 0.73)',
      'evaluations_permitted':1,
      'commitment':'result reported as is; no threshold change, model change, seed choice or rerun after seeing it'},open(TPF,'w'),indent=2)
    print('test protocol written')
print("threshold from protocol file:",round(THR,3))

# ---------- test windows ----------
SR=16000; NWIN=48896; STEP=24000; WSEC=NWIN/SR
os.makedirs(WORK+'/src',exist_ok=True); os.makedirs(TD,exist_ok=True)
with zipfile.ZipFile(os.path.join(PROJ,'data','colab_training_package.zip')) as z:
    open(WORK+'/src/preprocessing.py','wb').write(z.read('src/preprocessing.py'))
    man=pd.read_csv(io.BytesIO(z.read('data/processed/manifest.csv')))
sys.path.insert(0,WORK+'/src'); import preprocessing as P
test_pat=set(man[man.split=='test'].patient_id.astype(int)); print("test patients:",len(test_pat))
with zipfile.ZipFile(os.path.join(PROJ,'data','icbhi_kaggle.zip')) as z:
    for n in z.namelist():
        b=os.path.basename(n)
        if re.match(r'^\d+_',b) and b.endswith(('.wav','.txt')) and int(b.split('_')[0]) in test_pat and not os.path.exists(os.path.join(TD,b)):
            open(os.path.join(TD,b),'wb').write(z.read(n))
names=sorted({f[:-4] for f in os.listdir(TD) if f.endswith('.wav')}); print("test recordings:",len(names))

X=[];Y=[];PID=[];DEV=[];COV=[]; t0=time.time()
for name in names:
    pat=int(name.split('_')[0]); dev=name.split('_')[4]
    a=np.loadtxt(os.path.join(TD,name+'.txt'),ndmin=2); y,_=librosa.load(os.path.join(TD,name+'.wav'),sr=SR,mono=True)
    for s in range(0,len(y)-NWIN+1,STEP):
        ws=s/SR; ov=np.clip(np.minimum(a[:,1],ws+WSEC)-np.maximum(a[:,0],ws),0,None); cr,wh=a[:,2]>0,a[:,3]>0
        cov=np.array([ov[~cr&~wh].sum(),ov[cr&~wh].sum(),ov[~cr&wh].sum(),ov[cr&wh].sum()])
        if cov[0]>=WSEC/2: lab=0
        elif cov[1:].sum()>=WSEC/2: lab=1
        else: continue
        f=P.extract_mel_spectrogram(P.butterworth_filter(y[s:s+NWIN])); assert f.shape==(128,192)
        X.append(f); Y.append(lab); PID.append(pat); DEV.append(dev); COV.append(cov/WSEC)
X=np.stack(X)[...,None].astype(np.float32); Y=np.array(Y)==1; PID=np.array(PID); DEV=np.array(DEV); COV=np.array(COV)
print(f"windows {len(Y)} (normal {int((~Y).sum())}, abnormal {int(Y.sum())}) from {len(set(PID))} patients in {(time.time()-t0)/60:.1f} min")
print("feature check: mean %.3f std %.3f"%(X[::40].mean(),X[::40].std()),"| expected about 2767 windows from the earlier preview")

# ---------- scores ----------
tp=os.path.join(TFD,'pulmoscan_W_B1_s0_int8.tflite')
try: itp=tf.lite.Interpreter(model_path=tp,experimental_op_resolver_type=tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES)
except Exception: itp=tf.lite.Interpreter(model_path=tp)
itp.allocate_tensors(); inp=itp.get_input_details()[0]; out=itp.get_output_details()[0]
si,zi=inp['quantization']; so,zo=out['quantization']; s_int8=np.empty(len(X))
for i in range(len(X)):
    q=np.clip(np.round(X[i:i+1]/si+zi),-128,127).astype(np.int8)
    itp.set_tensor(inp['index'],q); itp.invoke()
    s_int8[i]=1-(itp.get_tensor(out['index']).astype(np.float32)[0,0]-zo)*so
scores={'int8_s0':s_int8}
for sd in [0,1,2]:
    m=tf.keras.models.load_model(os.path.join(CK,f'final_W_B1_s{sd}.keras'),compile=False)
    scores[f'float_s{sd}']=1-m.predict(X,batch_size=64,verbose=0)[0][:,0]; del m

def met(y,s,thr): p=s>thr; return roc_auc_score(y,s), p[y].mean(), (~p[~y]).mean(), p.mean()
res={k:dict(zip(['auc','se','sp','flagged'],map(float,met(Y,s,THR)))) for k,s in scores.items()}

# patient level bootstrap for the deployed model
rng=np.random.RandomState(0); up=np.unique(PID); idx={u:np.where(PID==u)[0] for u in up}; bs=[]
for _ in range(1000):
    ii=np.concatenate([idx[u] for u in rng.choice(up,len(up),replace=True)])
    if Y[ii].sum()==0 or (~Y[ii]).sum()==0: continue
    bs.append(met(Y[ii],s_int8[ii],THR)[:3])
ci=np.percentile(np.array(bs),[2.5,97.5],axis=0)

dev_rows={}
for d in sorted(set(DEV)):
    m=DEV==d; row={'windows':int(m.sum()),'patients':int(len(set(PID[m]))),'abnormal_share':float(Y[m].mean())}
    if Y[m].sum()>=30 and (~Y[m]).sum()>=30: row.update(dict(zip(['auc','se','sp'],map(float,met(Y[m],s_int8[m],THR)[:3]))))
    dev_rows[d]=row
kind={}
for j,nm in [(1,'crackle only'),(2,'wheeze only'),(3,'both')]:
    m=Y&(np.argmax(COV[:,1:4],axis=1)==j-1); kind[nm]={'windows':int(m.sum()),'se':float((s_int8[m]>THR).mean()) if m.sum() else None}

out_json={'threshold':THR,'n_windows':int(len(Y)),'n_patients':int(len(up)),'primary_int8_seed0':res['int8_s0'],
  'float_reference':{k:v for k,v in res.items() if k!='int8_s0'},
  'bootstrap_95ci_int8':{'auc':ci[:,0].tolist(),'se':ci[:,1].tolist(),'sp':ci[:,2].tolist()},
  'by_device':dev_rows,'by_abnormal_kind':kind,'written_at':datetime.datetime.now().isoformat()}
json.dump(out_json,open(RESF,'w'),indent=2)

r=res['int8_s0']
print("\n============ TEST RESULT, WINDOWS (final, do not rerun) ============")
print(f"deployed int8 seed 0 at threshold {THR:.3f}:  AUC {r['auc']:.3f}   Se {r['se']:.2f}   Sp {r['sp']:.2f}   flagged {r['flagged']:.2f}")
print(f"patient bootstrap 95% CI:  AUC {ci[0,0]:.3f} to {ci[1,0]:.3f}   Se {ci[0,1]:.2f} to {ci[1,1]:.2f}   Sp {ci[0,2]:.2f} to {ci[1,2]:.2f}")
for k in ['float_s0','float_s1','float_s2']: print(f"{k}:  AUC {res[k]['auc']:.3f}   Se {res[k]['se']:.2f}   Sp {res[k]['sp']:.2f}")
print("by device (deployed model):")
for d,v in dev_rows.items(): print(f"  {d:9s} patients {v['patients']:2d} windows {v['windows']:4d}  abnormal {v['abnormal_share']:.2f}"+(f"  AUC {v['auc']:.3f} Se {v['se']:.2f} Sp {v['sp']:.2f}" if 'auc' in v else "  (too few of one class for metrics)"))
print("sensitivity by abnormal kind:",{k:(v['windows'],None if v['se'] is None else round(v['se'],2)) for k,v in kind.items()})
print("\nreference: window CV fold AUC mean 0.720, Se 0.56 Sp 0.73 | earlier cycle model on test AUC 0.55 to 0.58 | all Normal: Se 0 Sp 1")

test protocol written
threshold from protocol file: 0.508
test patients: 19
test recordings: 222
windows 2767 (normal 1344, abnormal 1423) from 19 patients in 1.2 min
feature check: mean 0.000 std 1.000 | expected about 2767 windows from the earlier preview


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



============ TEST RESULT, WINDOWS (final, do not rerun) ============
deployed int8 seed 0 at threshold 0.508:  AUC 0.537   Se 0.47   Sp 0.56   flagged 0.46
patient bootstrap 95% CI:  AUC 0.459 to 0.660   Se 0.38 to 0.61   Sp 0.45 to 0.71
float_s0:  AUC 0.536   Se 0.46   Sp 0.57
float_s1:  AUC 0.554   Se 0.44   Sp 0.64
float_s2:  AUC 0.549   Se 0.42   Sp 0.65
by device (deployed model):
  AKGC417L  patients  7 windows 2221  abnormal 0.54  AUC 0.474 Se 0.47 Sp 0.45
  Litt3200  patients  1 windows   72  abnormal 0.26  (too few of one class for metrics)
  LittC2SE  patients  2 windows   72  abnormal 0.44  AUC 0.506 Se 0.28 Sp 0.90
  Meditron  patients  9 windows  402  abnormal 0.40  AUC 0.727 Se 0.51 Sp 0.89
sensitivity by abnormal kind: {'crackle only': (914, 0.34), 'wheeze only': (321, 0.72), 'both': (188, 0.69)}

reference: window CV fold AUC mean 0.720, Se 0.56 Sp 0.73 | earlier cycle model on test AUC 0.55 to 0.58 | all Normal: Se 0 Sp 1


In [ ]:
# ============================================================
# CELL 7: crackle front end features (n_fft 512, hop 128, 64 mels) for the same dev windows. CPU.
# ============================================================
import os, io, sys, json, time, zipfile, datetime, warnings
import numpy as np, librosa
warnings.filterwarnings('ignore')
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
PROJ='/content/drive/MyDrive/pulmoscan semester 7'; RES=os.path.join(PROJ,'results'); WORK='/content/work'; RD='/content/icbhi_dev'
SR=16000; NWIN=48896; NF=512; HOP=128; NM=64; FMIN=50; FMAX=2000; NFR=1+NWIN//HOP
os.makedirs(WORK+'/src',exist_ok=True); os.makedirs(RD,exist_ok=True)

fb=librosa.filters.mel(sr=SR,n_fft=NF,n_mels=NM,fmin=FMIN,fmax=FMAX)
empty=int((fb.max(axis=1)==0).sum()); used=int((fb.sum(axis=0)>0).sum())
print(f"mel filters {NM}, empty {empty}, FFT bins used {used}, frames per window {NFR}")
assert empty==0, "empty mel filters: stop and tell me, we lower the number of bands"

PPF=os.path.join(RES,'crackle_protocol.json')
if not os.path.exists(PPF):
    json.dump({'written_at':datetime.datetime.now().isoformat(),
      'question':'does a short window spectrogram (32 ms window, 8 ms hop) detect abnormal windows better than the current 128 ms window, especially crackles',
      'change':'n_fft 2048 to 512, hop 256 to 128, mel bands 128 to 64; input 128x192 becomes 64x383 (24448 versus 24576 values); nothing else changes',
      'held_fixed':'same dev windows and labels, same 5 folds, seeds 0 1 2, B1 architecture (522763 parameters), focal gamma 2, 40 epochs, same loss and diagnosis head; SpecAugment style masks scaled to 10 of 64 bands and 60 of 383 frames (same fraction as before)',
      'baseline':'arm B of the window experiment (out of fold probabilities W_B_*)',
      'primary_rule':'adopt only if mean fold AUC on windows (3 seed means) beats arm B by more than 0.02 and is better on at least 4 of 5 folds',
      'secondary_reported_not_decisive':'within fold sensitivity at specificity 0.70 overall and on crackle only windows',
      'staging':'stage 1 seed 0 (5 runs); stage 2 seeds 1 and 2; the rule is applied only when all 3 seeds are complete',
      'if_adopted':'final model trained on windows of all 82 patients, validated by CV only; test set not used again; firmware front end constants change (n_fft 512, hop 128, 64 mels); int8 check repeated',
      'test_set':'not used',
      'commitment':'result reported as is; rule not changed after results'},open(PPF,'w'),indent=2)
    print('protocol written')

with zipfile.ZipFile(os.path.join(PROJ,'data','colab_training_package.zip')) as z:
    open(WORK+'/src/preprocessing.py','wb').write(z.read('src/preprocessing.py'))
sys.path.insert(0,WORK+'/src'); import preprocessing as P
W=np.load(os.path.join(PROJ,'data','windows_dev.npz')); rec=W['rec']; start=W['start']; N=len(rec)
names=set(rec.tolist())
with zipfile.ZipFile(os.path.join(PROJ,'data','icbhi_kaggle.zip')) as z:
    for n in z.namelist():
        b=os.path.basename(n)
        if b.endswith('.wav') and b[:-4] in names and not os.path.exists(os.path.join(RD,b)):
            open(os.path.join(RD,b),'wb').write(z.read(n))
have=[f for f in os.listdir(RD) if f.endswith('.wav')]; print("recordings extracted:",len(have),"of",len(names)); assert len(have)==len(names)

def feat_c(seg):
    y=P.butterworth_filter(seg)
    mel=librosa.feature.melspectrogram(y=y,sr=SR,n_fft=NF,hop_length=HOP,n_mels=NM,fmin=FMIN,fmax=FMAX,power=2.0)
    db=librosa.power_to_db(mel,ref=np.max)
    return ((db-db.mean())/(db.std()+1e-8)).astype(np.float32)

order={}
for i,r in enumerate(rec): order.setdefault(r,[]).append(i)
X=np.empty((N,NM,NFR),np.float16); t0=time.time(); done=0
for r,idxs in order.items():
    y,_=librosa.load(os.path.join(RD,r+'.wav'),sr=SR,mono=True)
    for i in idxs:
        s=int(round(float(start[i])*SR)); seg=y[s:s+NWIN]; assert len(seg)==NWIN
        f=feat_c(seg); assert f.shape==(NM,NFR),f.shape
        X[i]=f
    done+=1
    if done%100==0: print(f"  {done} recordings, {(time.time()-t0)/60:.1f} min",flush=True)

rs=np.random.RandomState(0); diffs=[]
for i in rs.choice(N,6,replace=False):
    y,_=librosa.load(os.path.join(RD,rec[i]+'.wav'),sr=SR,mono=True); s=int(round(float(start[i])*SR))
    old=P.extract_mel_spectrogram(P.butterworth_filter(y[s:s+NWIN]))
    diffs.append(float(np.abs(old-W['X'][i].astype(np.float32)).max()))
print("alignment check (old features recomputed versus stored, float16 rounding expected):",np.round(diffs,4))
assert max(diffs)<0.05, "windows do not map to the same audio, stop and paste this output"
out=os.path.join(PROJ,'data','windows_dev_c.npz'); np.savez(out,X=X)
Xs=X[::40].astype(np.float32)
print(f"\nsaved {out} ({os.path.getsize(out)/1e6:.0f} MB)  shape {X.shape}  mean {Xs.mean():.3f} std {Xs.std():.3f}")

mel filters 64, empty 0, FFT bins used 62, frames per window 383
protocol written
recordings extracted: 499 of 499
  100 recordings, 0.5 min
  200 recordings, 0.7 min
  300 recordings, 1.0 min
  400 recordings, 1.2 min
alignment check (old features recomputed versus stored, float16 rounding expected): [0.001  0.001  0.001  0.001  0.0019 0.001 ]

saved /content/drive/MyDrive/pulmoscan semester 7/data/windows_dev_c.npz (305 MB)  shape (6221, 64, 383)  mean 0.000 std 1.000


In [ ]:
# ============================================================
# CELL 8: crackle front end CV. Arm C (B1 on 64x383 features of the same windows) vs arm B (saved). GPU. No test data.
# ============================================================
import os, io, json, time, random, gc, zipfile
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.metrics import roc_auc_score
print("GPUs:", tf.config.list_physical_devices('GPU'))
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
PROJ='/content/drive/MyDrive/pulmoscan semester 7'; RES=os.path.join(PROJ,'results'); OOF=os.path.join(RES,'oof_probs')
PPF=os.path.join(RES,'crackle_protocol.json'); assert os.path.exists(PPF), "run cell 7 first"
NM=64; T=383; EPOCHS=40; BATCH=32; LR=1e-3; LR_MIN=1e-6; DIAG_W=0.1; NS=4; ND=7
SEEDS=[0]          # stage 1; change to [0,1,2] for stage 2 (finished runs are skipped)

with zipfile.ZipFile(os.path.join(PROJ,'data','colab_training_package.zip')) as z:
    man=pd.read_csv(io.BytesIO(z.read('data/processed/manifest.csv')))
man['patient_id']=man.patient_id.astype(int); devm=man[man.split.isin(['train','val'])].copy().reset_index(drop=True)
rng=np.random.RandomState(42); pdf=devm.groupby('patient_id')['diagnosis_label'].agg(lambda x:x.mode()[0]).reset_index()
folds=[[] for _ in range(5)]
for _,g in pdf.groupby('diagnosis_label'):
    ps=g.patient_id.values.copy(); rng.shuffle(ps)
    for j,p in enumerate(ps): folds[j%5].append(int(p))
cyc=[int(np.isin(devm.patient_id.values,f).sum()) for f in folds]; print("cycles per fold:",cyc)
assert cyc[:3]==[1056,551,577], "folds differ from the earlier CV, stop"
diag_of=dict(zip(pdf.patient_id.astype(int),pdf.diagnosis_label.astype(int)))

W=np.load(os.path.join(PROJ,'data','windows_dev.npz')); yw=W['y'].astype(int); pw=W['patient'].astype(int); cov=W['cov']
ym=np.where(yw==0,0,1+np.argmax(cov[:,1:4],axis=1)); ydw=np.array([diag_of[p] for p in pw])
cr_only=(yw==1)&(np.argmax(cov[:,1:4],axis=1)==0)
Xc=np.load(os.path.join(PROJ,'data','windows_dev_c.npz'))['X']; assert len(Xc)==len(yw) and Xc.shape[1:]==(NM,T), Xc.shape
print("windows",len(yw),"features",Xc.shape,"| crackle only abnormal windows",int(cr_only.sum()))

def focal(g=2.0):
    def fn(yt,yp):
        yp=tf.clip_by_value(yp,1e-7,1.0); ce=-yt*tf.math.log(yp)
        p=tf.reduce_sum(yt*yp,axis=-1,keepdims=True)
        return tf.reduce_mean(tf.pow(1.0-p,g)*ce)
    fn.__name__='focal'; return fn
def build_b1():
    L=tf.keras.layers; inp=tf.keras.Input(shape=(NM,T,1)); x=inp
    for f in [32,64,128]:
        x=L.Conv2D(f,3,padding='same')(x); x=L.BatchNormalization()(x); x=L.Activation('relu')(x); x=L.MaxPooling2D(2)(x)
    x=L.Conv2D(256,3,padding='same')(x); x=L.BatchNormalization()(x); x=L.Activation('relu')(x)
    x=L.GlobalAveragePooling2D()(x)
    sh=L.Dropout(0.5)(L.Dense(256,activation='relu')(x))
    s=L.Dropout(0.3)(L.Dense(128,activation='relu')(sh)); so=L.Dense(NS,activation='softmax',name='sound')(s)
    dd=L.Dropout(0.3)(L.Dense(128,activation='relu')(sh)); do=L.Dense(ND,activation='softmax',name='diagnosis')(dd)
    return tf.keras.Model(inp,[so,do])
def augment(x,y):                      # masks scaled to 10 of 64 bands and 60 of 383 frames
    B=tf.shape(x)[0]; F=tf.shape(x)[1]; Tt=tf.shape(x)[2]
    real=tf.cast(tf.not_equal(x,0.0),tf.float32)
    x=x+tf.random.normal([B,1,1,1],0.0,0.15)*real
    fw=tf.random.uniform([B,1,1,1],0,11,dtype=tf.int32)
    f0=tf.random.uniform([B,1,1,1],0,1000000,dtype=tf.int32)%tf.maximum(F-fw,1)
    fi=tf.reshape(tf.range(F),[1,-1,1,1]); fmask=tf.logical_and(fi>=f0,fi<f0+fw)
    tw=tf.random.uniform([B,1,1,1],0,61,dtype=tf.int32)
    t0=tf.random.uniform([B,1,1,1],0,1000000,dtype=tf.int32)%tf.maximum(Tt-tw,1)
    ti=tf.reshape(tf.range(Tt),[1,1,-1,1]); tmask=tf.logical_and(ti>=t0,ti<t0+tw)
    return tf.where(tf.logical_or(fmask,tmask),tf.zeros_like(x),x),y

def run(k,seed):
    path=os.path.join(OOF,f'W_C_f{k}_s{seed}.npz')
    if os.path.exists(path): print('  SKIP',os.path.basename(path)); return
    tew=np.isin(pw,folds[k]); trw=~tew
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed); tf.keras.backend.clear_session()
    Xtr=Xc[trw].astype(np.float32)[...,None]
    ds=tf.data.Dataset.from_tensor_slices((Xtr,{'sound':tf.keras.utils.to_categorical(ym[trw],NS),'diagnosis':tf.keras.utils.to_categorical(ydw[trw],ND)}))
    ds=ds.shuffle(4096,seed=seed).batch(BATCH).map(augment,num_parallel_calls=tf.data.AUTOTUNE).prefetch(2)
    m=build_b1(); m.compile(optimizer=tf.keras.optimizers.Adam(LR),loss={'sound':focal(2.0),'diagnosis':focal(2.0)},
        loss_weights={'sound':1.0,'diagnosis':DIAG_W},metrics={'sound':'accuracy'})
    if k==0 and seed==SEEDS[0]: print("  parameters:",m.count_params(),"(arm B had 522763)")
    t0=time.time()
    m.fit(ds,epochs=EPOCHS,verbose=0,callbacks=[tf.keras.callbacks.LearningRateScheduler(
        lambda e: float(LR_MIN+0.5*(LR-LR_MIN)*(1+np.cos(np.pi*e/EPOCHS))))])
    pr=m.predict(Xc[tew].astype(np.float32)[...,None],batch_size=64,verbose=0)[0]
    print(f'  C fold {k} seed {seed}: window AUC {roc_auc_score(yw[tew],1-pr[:,0]):.3f}   {(time.time()-t0)/60:.1f}min')
    np.savez_compressed(path,p=pr,y=yw[tew],patient=pw[tew]); del m; gc.collect()
for seed in SEEDS:
    for k in range(5): run(k,seed)

# ---------- comparison with arm B on the same folds and seeds ----------
def fm(arm,k,s):
    z=np.load(os.path.join(OOF,f'W_{arm}_f{k}_s{s}.npz')); tew=np.isin(pw,folds[k]); y=z['y']>0
    assert (y==(yw[tew]==1)).all(), "windows are not aligned"
    sc=1-z['p'][:,0]; thr=np.quantile(sc[~y],0.70)
    return roc_auc_score(y,sc),(sc[y]>thr).mean(),(sc[cr_only[tew]]>thr).mean()
done=[s for s in [0,1,2] if all(os.path.exists(os.path.join(OOF,f'W_C_f{k}_s{s}.npz')) for k in range(5))]
print("\ncomplete seeds for arm C:",done)
if done:
    print("fold |  AUC:  B      C     diff | Se at Sp 0.70:  B     C | crackle only Se:  B     C")
    R={a:np.array([np.mean([fm(a,k,s) for s in done],axis=0) for k in range(5)]) for a in 'BC'}
    for k in range(5):
        print(f"  {k}  |      {R['B'][k,0]:.3f}  {R['C'][k,0]:.3f}  {R['C'][k,0]-R['B'][k,0]:+.3f} |            {R['B'][k,1]:.2f}  {R['C'][k,1]:.2f} |              {R['B'][k,2]:.2f}  {R['C'][k,2]:.2f}")
    g=R['C'][:,0]-R['B'][:,0]
    print(f"mean |      {R['B'][:,0].mean():.3f}  {R['C'][:,0].mean():.3f}  {g.mean():+.3f} |            {R['B'][:,1].mean():.2f}  {R['C'][:,1].mean():.2f} |              {R['B'][:,2].mean():.2f}  {R['C'][:,2].mean():.2f}")
    if done==[0,1,2]:
        wins=int((g>0).sum())
        print(f"\nRULE: mean AUC gain {g.mean():+.3f} > 0.02: {g.mean()>0.02}   better on >=4 of 5 folds: {wins>=4} ({wins}/5)")
        print("VERDICT:",'ADOPT CRACKLE FRONT END' if (g.mean()>0.02 and wins>=4) else 'KEEP CURRENT FRONT END, report as negative')
    else: print("\nstage 1 only: no verdict, the rule needs all 3 seeds")

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
cycles per fold: [1056, 551, 577, 722, 784]
windows 6221 features (6221, 64, 383) | crackle only abnormal windows 1628
  parameters: 522763 (arm B had 522763)
  C fold 0 seed 0: window AUC 0.644   4.4min
  C fold 1 seed 0: window AUC 0.769   4.6min
  C fold 2 seed 0: window AUC 0.654   4.8min
  C fold 3 seed 0: window AUC 0.706   4.5min
  C fold 4 seed 0: window AUC 0.693   4.4min

complete seeds for arm C: [0]
fold |  AUC:  B      C     diff | Se at Sp 0.70:  B     C | crackle only Se:  B     C
  0  |      0.705  0.644  -0.061 |            0.57  0.49 |              0.39  0.30
  1  |      0.724  0.769  +0.044 |            0.60  0.72 |              0.44  0.65
  2  |      0.692  0.654  -0.038 |            0.54  0.54 |              0.35  0.45
  3  |      0.749  0.706  -0.043 |            0.65  0.56 |              0.63  0.48
  4  |      0.710  0.693  -0.017 |            0.62  0.59 |              0.51  0.50
mean |     